# 🏭 Тестирование модуля построения модели (3-Statement Model)

**Упрощенная и структурированная версия** с учетом доработок канонической структуры данных.

## 📋 Структура ноутбука:

1. **Инициализация и настройка** - загрузка конфигурации, инициализация DataMart
2. **Загрузка и валидация данных** - загрузка истории, валидация канонической структуры
3. **Построение модели** - запуск модели, проверка сходимости
4. **Валидация результатов** - проверка BS Identity, CF Identity, Retained Earnings
5. **Анализ и отчетность** - сводка результатов, выявление проблем

---

**Примечание**: Этот ноутбук использует модули `engine.acceptance.checks` и `engine.model.core` для валидации и построения модели, что исключает дублирование кода.

### 🔍 Детальный анализ сопоставимости истории и прогноза

**Примечание**: Этот раздел использует функции загрузки данных из DataMart. 
Для построения модели используйте `engine.model.core.build_model()`, 
для валидации - `engine.acceptance.checks.run_acceptance()`.

Анализ каждой статьи с проверкой:
- Знаков (положительные/отрицательные)
- Порядка чисел (масштаб)
- Логики изменений
- Сопоставимости величин

In [ ]:
# Детальный анализ сопоставимости истории и прогноза
# ВАЖНО: Эти функции определены один раз и используются во всем ноутбуке
# Не дублируйте их определения в других ячейках!

import pandas as pd
import numpy as np
from IPython.display import display, Markdown, HTML
from pathlib import Path
from typing import Optional
from engine.database.data_mart import get_data_mart

# Определение вспомогательных функций (определяются один раз)
def _open_data_mart():
    try:
        return get_data_mart(ROOT, COMPANY)
    except Exception as exc:
        print(f"⚠️ Не удалось подключиться к витрине данных: {exc}")
        return None

def load_history_from_db(statement_type='IS', canonical=False):
    stmt = statement_type.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        try:
            if stmt == 'IS':
                df = mart.get_history_income_statement(canonical=canonical)
            elif stmt == 'BS':
                df = mart.get_history_balance_sheet(canonical=canonical)
            else:
                df = mart.get_history_cash_flow(canonical=canonical)
            if df is not None and not df.empty:
                print(f"✅ История {stmt} загружена из витрины")
        except Exception as exc:
            print(f"⚠️ Не удалось загрузить историю {stmt} из витрины: {exc}")
        finally:
            mart.close()
    
    file_map = {
        'IS': f"history/is_history_{COMPANY}.csv",
        'BS': f"history/bs_history_{COMPANY}.csv",
        'CF': f"history/cf_history_{COMPANY}.csv"
    }
    
    if df is None or df.empty:
        csv_path = croot / file_map.get(stmt, '')
        if csv_path.exists():
            print(f"📄 История {stmt} загружена из CSV")
            return pd.read_csv(csv_path)
        print(f"❌ История {stmt} не найдена ни в витрине, ни в CSV")
        return pd.DataFrame()
    
    if not canonical and 'metric' in df.columns:
        year_cols = [c for c in df.columns if str(c).isdigit()]
        if year_cols:
            df_long = df.set_index('metric')[year_cols].T.reset_index().rename(columns={'index': 'year'})
            df_long['year'] = df_long['year'].astype(int)
            return df_long
    return df

def load_model_forecast_from_db(statement_type='IS', canonical=True):
    stmt = statement_type.upper()
    csv_filename = f"outputs/model/3statement_{stmt}.csv"
    return load_model_table(stmt, canonical=canonical, csv_filename=csv_filename)

def load_model_table(table_name: str, canonical: bool = False, csv_filename: Optional[str] = None):
    table_key = table_name.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        version = _resolve_model_version(mart)
        if version:
            try:
                df = mart.get_model_results(version, table_key, canonical=canonical)
                if df is not None and not df.empty:
                    print(f"✅ Таблица {table_key} загружена из витрины (версия {version})")
                    mart.close()
                    return df
            except Exception as exc:
                print(f"⚠️ Не удалось загрузить таблицу {table_key} из витрины: {exc}")
            finally:
                mart.close()
        else:
            mart.close()
    
    if csv_filename:
        csv_path = croot / csv_filename
        if csv_path.exists():
            print(f"⚠️ Используем legacy CSV для {table_key}: {csv_path.relative_to(ROOT)}")
            return pd.read_csv(csv_path)
    
    print(f"⚠️ Таблица {table_key} не найдена ни в витрине, ни в CSV")
    return pd.DataFrame()

def _resolve_model_version(mart):
    global MODEL_VERSION
    if MODEL_VERSION:
        return MODEL_VERSION
    try:
        versions = mart.get_existing_versions()
        if versions:
            MODEL_VERSION = versions[0]['version']
            return MODEL_VERSION
    except Exception as exc:
        print(f"⚠️ Не удалось определить версию модели: {exc}")
    return None

# Проверка переменных окружения
if 'ROOT' not in globals():
    current_dir = Path.cwd()
    if (current_dir / 'engine').exists():
        ROOT = current_dir
    elif (current_dir.parent / 'engine').exists():
        ROOT = current_dir.parent
    elif (current_dir.parent.parent / 'engine').exists():
        ROOT = current_dir.parent.parent
    else:
        nb_path = Path(__file__) if '__file__' in globals() else Path.cwd()
        ROOT = nb_path.parent.parent.parent
    print(f"ROOT: {ROOT}")

if 'COMPANY' not in globals():
    COMPANY = 'us_steel'  # fallback
    current_dir = Path.cwd()
    if 'companies' in current_dir.parts:
        companies_idx = current_dir.parts.index('companies')
        if companies_idx + 1 < len(current_dir.parts):
            COMPANY = current_dir.parts[companies_idx + 1]
    print(f"COMPANY: {COMPANY}")

if 'croot' not in globals():
    croot = ROOT / 'companies' / COMPANY
    print(f"Company root: {croot}")

if 'MODEL_VERSION' not in globals():
    MODEL_VERSION = None

display(Markdown("### 🔍 Детальный анализ каждой статьи: История vs Прогноз"))

# Детальный анализ сопоставимости истории и прогноза


# Загружаем данные если еще не загружены
if 'hist_is' not in locals() or hist_is.empty:
    hist_is = load_history_from_db('IS', canonical=True)
if 'hist_bs' not in locals() or hist_bs.empty:
    hist_bs = load_history_from_db('BS', canonical=True)
if 'hist_cf' not in locals() or hist_cf.empty:
    hist_cf = load_history_from_db('CF', canonical=True)

if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)
if 'forecast_bs' not in locals() or forecast_bs.empty:
    forecast_bs = load_model_forecast_from_db('BS', canonical=True)
if 'forecast_cf' not in locals() or forecast_cf.empty:
    forecast_cf = load_model_forecast_from_db('CF', canonical=True)

# Преобразуем в wide format
def to_wide(df, metric_col='metric'):
    if df.empty:
        return pd.DataFrame()
    if 'year' in df.columns and metric_col in df.columns:
        return df.pivot(index=metric_col, columns='year', values='value')
    elif metric_col in df.columns:
        return df.set_index(metric_col)
    return df

hist_is_wide = to_wide(hist_is)
hist_bs_wide = to_wide(hist_bs)
hist_cf_wide = to_wide(hist_cf)
forecast_is_wide = to_wide(forecast_is)
forecast_bs_wide = to_wide(forecast_bs)
forecast_cf_wide = to_wide(forecast_cf)

# Получаем годы
hist_years = sorted([int(c) for c in hist_is_wide.columns if str(c).isdigit() and int(c) <= 2024])[-5:]
forecast_years = sorted([int(c) for c in forecast_is_wide.columns if str(c).isdigit() and int(c) > 2024])

def analyze_item(item_name, hist_df, forecast_df, statement_type='IS'):
    """Анализирует одну статью, сравнивая историю и прогноз"""
    issues = []
    warnings = []
    
    # Проверяем наличие в истории и прогнозе
    in_hist = item_name in hist_df.index if not hist_df.empty else False
    in_forecast = item_name in forecast_df.index if not forecast_df.empty else False
    
    if not in_hist and not in_forecast:
        return None
    
    # Получаем значения
    hist_values = {}
    forecast_values = {}
    
    if in_hist:
        for y in hist_years:
            val = pd.to_numeric(hist_df.loc[item_name, str(y)], errors='coerce') if str(y) in hist_df.columns else None
            if pd.notna(val):
                hist_values[y] = float(val)
    
    if in_forecast:
        for y in forecast_years:
            val = pd.to_numeric(forecast_df.loc[item_name, str(y)], errors='coerce') if str(y) in forecast_df.columns else None
            if pd.notna(val):
                forecast_values[y] = float(val)
    
    if not hist_values and not forecast_values:
        return None
    
    # Анализ
    result = {
        'Статья': item_name,
        'В истории': '✅' if in_hist else '❌',
        'В прогнозе': '✅' if in_forecast else '❌',
        'Проблемы': [],
        'Предупреждения': []
    }
    
    if hist_values and forecast_values:
        # 1. Проверка знаков
        hist_signs = [1 if v > 0 else (-1 if v < 0 else 0) for v in hist_values.values()]
        forecast_signs = [1 if v > 0 else (-1 if v < 0 else 0) for v in forecast_values.values()]
        
        hist_typical_sign = max(set(hist_signs), key=hist_signs.count) if hist_signs else 0
        forecast_typical_sign = max(set(forecast_signs), key=forecast_signs.count) if forecast_signs else 0
        
        if hist_typical_sign != 0 and forecast_typical_sign != 0 and hist_typical_sign != forecast_typical_sign:
            result['Проблемы'].append(f"❌ Смена знака: история {('+' if hist_typical_sign > 0 else '-')}, прогноз {('+' if forecast_typical_sign > 0 else '-')}")
        
        # 2. Проверка порядка чисел
        hist_abs_values = [abs(v) for v in hist_values.values() if v != 0]
        forecast_abs_values = [abs(v) for v in forecast_values.values() if v != 0]
        
        if hist_abs_values and forecast_abs_values:
            hist_avg_magnitude = np.mean(hist_abs_values)
            forecast_avg_magnitude = np.mean(forecast_abs_values)
            
            # Проверяем разницу в порядке
            if hist_avg_magnitude > 0:
                ratio = forecast_avg_magnitude / hist_avg_magnitude
                if ratio > 10 or ratio < 0.1:
                    result['Проблемы'].append(f"❌ Разница в масштабе: {ratio:.1f}x (история ~${hist_avg_magnitude:,.0f}, прогноз ~${forecast_avg_magnitude:,.0f})")
                elif ratio > 3 or ratio < 0.33:
                    result['Предупреждения'].append(f"⚠️ Значительное изменение масштаба: {ratio:.1f}x")
        
        # 3. Проверка тренда
        if len(hist_values) >= 2 and len(forecast_values) >= 1:
            hist_last = list(hist_values.values())[-1]
            hist_prev = list(hist_values.values())[-2]
            forecast_first = list(forecast_values.values())[0]
            
            hist_change = hist_last - hist_prev
            forecast_jump = forecast_first - hist_last
            
            # Проверяем резкий скачок
            if abs(hist_change) > 0:
                jump_ratio = abs(forecast_jump) / abs(hist_change) if hist_change != 0 else float('inf')
                if jump_ratio > 5:
                    result['Проблемы'].append(f"❌ Резкий скачок на границе: изменение {forecast_jump:,.0f} vs историческое {hist_change:,.0f} ({jump_ratio:.1f}x)")
    
    elif not in_hist and in_forecast:
        result['Предупреждения'].append(f"⚠️ Статья отсутствует в истории, но есть в прогнозе")
    elif in_hist and not in_forecast:
        result['Предупреждения'].append(f"⚠️ Статья есть в истории, но отсутствует в прогнозе")
    
    return result

# Анализируем все статьи
all_items = set()
if not hist_is_wide.empty:
    all_items.update(hist_is_wide.index)
if not forecast_is_wide.empty:
    all_items.update(forecast_is_wide.index)
if not hist_bs_wide.empty:
    all_items.update(hist_bs_wide.index)
if not forecast_bs_wide.empty:
    all_items.update(forecast_bs_wide.index)
if not hist_cf_wide.empty:
    all_items.update(hist_cf_wide.index)
if not forecast_cf_wide.empty:
    all_items.update(forecast_cf_wide.index)

# Анализ IS
is_analysis = []
is_items = ['revenue', 'cogs', 'sga', 'depreciation_owned', 'depreciation_rou', 'amortization', 
            'ebitda', 'ebit', 'interest_income', 'interest_expense', 'lease_interest', 'tax', 'net_income']
for item in is_items:
    result = analyze_item(item, hist_is_wide, forecast_is_wide, 'IS')
    if result:
        is_analysis.append(result)

# Анализ BS
bs_analysis = []
bs_items = ['cash', 'ar', 'inventory', 'ppe', 'intangibles', 'rou_asset', 'total_assets',
            'st_debt', 'ap', 'debt', 'lease_liability', 'equity', 'total_liabilities', 'total_equity']
for item in bs_items:
    result = analyze_item(item, hist_bs_wide, forecast_bs_wide, 'BS')
    if result:
        bs_analysis.append(result)

# Анализ CF
cf_analysis = []
cf_items = ['net_income', 'depreciation', 'amortization', 'wc_delta', 'cfo', 'capex', 'cfi', 
            'debt_borrowings', 'debt_repayments', 'dividends', 'cff', 'net_change']
for item in cf_items:
    result = analyze_item(item, hist_cf_wide, forecast_cf_wide, 'CF')
    if result:
        cf_analysis.append(result)

# Выводим результаты
def print_analysis(analysis_list, title):
    
    for item in analysis_list:
        display(Markdown(f"#### {item['Статья']}"))
        print(f"  В истории: {item['В истории']} | В прогнозе: {item['В прогнозе']}")
        
        if item['Проблемы']:
            print(f"  \n  🔴 ПРОБЛЕМЫ:")
            for prob in item['Проблемы']:
                print(f"    {prob}")
        
        if item['Предупреждения']:
            print(f"  \n  ⚠️ ПРЕДУПРЕЖДЕНИЯ:")
            for warn in item['Предупреждения']:
                print(f"    {warn}")
        
        if not item['Проблемы'] and not item['Предупреждения']:
            print(f"  ✅ Нет проблем")
        
        print()

print_analysis(is_analysis, '📊 Income Statement (IS)')
print_analysis(bs_analysis, '🏦 Balance Sheet (BS)')
print_analysis(cf_analysis, '💰 Cash Flow Statement (CF)')

# Сводная таблица проблем
all_issues = []
for analysis in [is_analysis, bs_analysis, cf_analysis]:
    for item in analysis:
        if item['Проблемы'] or item['Предупреждения']:
            all_issues.append({
                'Статья': item['Статья'],
                'Проблем': len(item['Проблемы']),
                'Предупреждений': len(item['Предупреждения']),
                'Статус': '❌' if item['Проблемы'] else '⚠️'
            })

if all_issues:
    display(Markdown("### 📋 Сводная таблица проблем"))
    issues_df = pd.DataFrame(all_issues)
    display(issues_df)
    print(f"\nВсего статей с проблемами: {len(all_issues)}")
    print(f"Критических проблем: {sum(1 for i in all_issues if i['Проблем'] > 0)}")
    print(f"Предупреждений: {sum(1 for i in all_issues if i['Предупреждений'] > 0)}")
else:
    print("✅ Все статьи проверены, критических проблем не обнаружено")

In [ ]:
# Детальный анализ конкретных проблем на основе данных
display(Markdown("### 🔴 Критические проблемы, обнаруженные в данных"))

# 1. DEPRECIATION OWNED - отрицательные значения в истории
display(Markdown("#### 1. DEPRECIATION OWNED - Аномальные знаки в истории"))
print("Проблема: В истории DEPRECIATION OWNED имеет отрицательные значения:")
print("  2020: -$5,880,930,233")
print("  2021: -$8,674,372,093")
print("  2022: -$12,644,000,000")
print("  2023: -$18,524,930,233")
print("  2024: -$26,758,232,558")
print("\nВ прогнозе: положительные значения (правильно):")
print("  2025: $1,739,100,000")
print("  2026: $1,647,764,079")
print("\n❌ КРИТИЧЕСКАЯ ПРОБЛЕМА: Амортизация должна быть положительной (добавление в CFO).")
print("   Вероятная причина: ошибка в исторических данных или их загрузке.")
print("\n" + "="*80)

# 2. NET INCOME - все прогнозы отрицательные
display(Markdown("#### 2. NET INCOME - Все прогнозы отрицательные"))
print("История:")
print("  2020: -$1,165,000,000")
print("  2021: $4,174,000,000")
print("  2022: $2,524,000,000")
print("  2023: $895,000,000")
print("  2024: $384,000,000")
print("\nПрогноз (все отрицательные):")
print("  2025: -$1,148,253,283")
print("  2026: -$1,130,329,028")
print("  2027: -$1,138,681,103")
print("  2028: -$1,131,591,510")
print("  2029: -$1,115,007,735")
print("\n⚠️ ПРОБЛЕМА: Все прогнозы показывают убытки, хотя последние 4 года истории прибыльны.")
print("   Причина: возможно, слишком агрессивное снижение Revenue или высокие расходы.")
print("\n" + "="*80)

# 3. EQUITY - резкое падение
display(Markdown("#### 3. EQUITY - Резкое падение в прогнозе"))
print("История (стабильная):")
print("  2020: $3,786,000,000")
print("  2021-2024: ~$20,235,000,000")
print("\nПрогноз (резкое падение):")
print("  2025: $11,737,834,858 (↓42%)")
print("  2026: $10,607,505,830 (↓48%)")
print("  2027: $9,468,824,727 (↓53%)")
print("  2028: $8,337,233,217 (↓59%)")
print("  2029: $7,222,225,483 (↓64%)")
print("\n⚠️ ПРОБЛЕМА: Резкое падение капитала из-за отрицательного Net Income.")
print("   Это логично при убытках, но требует проверки причин убытков.")
print("\n" + "="*80)

# 4. ST DEBT - отсутствует в истории, появляется в прогнозе
display(Markdown("#### 4. ST DEBT - Появление в прогнозе"))
print("История: отсутствует (0 или не отображается)")
print("\nПрогноз:")
print("  2025: $500,000,000")
print("  2026: $891,416,325")
print("  2027: $1,231,243,315")
print("  2028: $1,570,186,118")
print("  2029: $1,911,032,911")
print("\n✅ ЛОГИЧНО: Рост краткосрочного долга для покрытия убытков и поддержания ликвидности.")
print("\n" + "="*80)

# 5. REVENUE - резкое снижение
display(Markdown("#### 5. REVENUE - Резкое снижение в прогнозе"))
print("История:")
print("  2020: $8,765,000,000")
print("  2021: $20,275,000,000 (↑131%)")
print("  2022: $21,065,000,000")
print("  2023: $18,053,000,000 (↓14%)")
print("  2024: $15,640,000,000 (↓13%)")
print("\nПрогноз (продолжение снижения):")
print("  2025: $13,549,526,395 (↓13%)")
print("  2026: $11,738,469,662 (↓13%)")
print("  2027: $10,169,482,386 (↓13%)")
print("  2028: $8,810,209,080 (↓13%)")
print("  2029: $7,632,618,956 (↓13%)")
print("\n⚠️ ПРОБЛЕМА: Постоянное снижение на 13% в год может быть слишком агрессивным.")
print("   Требуется проверка макрофакторов и качества регрессии.")
print("\n" + "="*80)

# 6. DEPRECIATION в CF vs IS
display(Markdown("#### 6. DEPRECIATION - Несоответствие между IS и CF"))
print("В IS (история): отрицательные значения (неправильно)")
print("В CF (история): положительные значения (правильно)")
print("  CF 2020: $40,000,000")
print("  CF 2024: $182,000,000")
print("\nВ прогнозе: совпадают (правильно)")
print("  IS 2025: $1,840,700,000")
print("  CF 2025: $1,840,700,000")
print("\n❌ КРИТИЧЕСКАЯ ПРОБЛЕМА: Исторические данные IS содержат ошибку знака для Depreciation.")
print("\n" + "="*80)

# 7. EBIT - отсутствует в истории
display(Markdown("#### 7. EBIT - Отсутствует в истории"))
print("История: отсутствует (не рассчитывался или не сохранялся)")
print("Прогноз: рассчитывается (все отрицательные из-за убытков)")
print("\n⚠️ ПРЕДУПРЕЖДЕНИЕ: Нет исторических данных для сравнения тренда EBIT.")
print("\n" + "="*80)

# Итоговая сводка
display(Markdown("### 📋 Итоговая сводка проблем"))
print("\n🔴 КРИТИЧЕСКИЕ ПРОБЛЕМЫ:")
print("  1. DEPRECIATION OWNED в истории имеет неправильный знак (отрицательный)")
print("  2. DEPRECIATION в IS истории не соответствует CF (разные знаки)")
print("\n⚠️ СЕРЬЕЗНЫЕ ПРОБЛЕМЫ:")
print("  3. NET INCOME: все прогнозы отрицательные (убытки)")
print("  4. REVENUE: агрессивное снижение на 13% ежегодно")
print("  5. EQUITY: резкое падение из-за убытков")
print("\n✅ ЛОГИЧНЫЕ ИЗМЕНЕНИЯ:")
print("  6. ST DEBT: рост для покрытия убытков (логично)")
print("  7. CASH: снижение, но остается на приемлемом уровне")
print("\n📊 РЕКОМЕНДАЦИИ:")
print("  1. Проверить и исправить исторические данные для Depreciation")
print("  2. Проверить качество регрессии Revenue (возможно, слишком пессимистичный прогноз)")
print("  3. Проверить макрофакторы - возможно, они дают слишком негативный прогноз")
print("  4. Рассмотреть возможность использования fallback метода для Revenue")


## 🔧 Импорты и настройка <a id="section-setup"></a>

In [1]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import sys
from IPython.display import display, Markdown, HTML
import matplotlib.pyplot as plt

# Настройка для графиков в Jupyter
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')

# Определение корня проекта
current_dir = Path.cwd()
if (current_dir / 'engine').exists():
    ROOT = current_dir
elif (current_dir.parent / 'engine').exists():
    ROOT = current_dir.parent
elif (current_dir.parent.parent / 'engine').exists():
    ROOT = current_dir.parent.parent
else:
    nb_path = Path(__file__) if '__file__' in globals() else Path.cwd()
    ROOT = nb_path.parent.parent.parent

print(f"ROOT: {ROOT}")
print(f"Engine exists: {(ROOT / 'engine').exists()}")

sys.path.insert(0, str(ROOT))

# Определяем COMPANY автоматически из пути ноутбука
COMPANY = 'us_steel'  # fallback по умолчанию
if 'companies' in current_dir.parts:
    companies_idx = current_dir.parts.index('companies')
    if companies_idx + 1 < len(current_dir.parts):
        COMPANY = current_dir.parts[companies_idx + 1]

croot = ROOT / 'companies' / COMPANY
print(f"Company root: {croot}")

from typing import Optional
from engine.database.data_mart import get_data_mart

MODEL_VERSION: Optional[str] = None

def is_history_year(year):
    """Проверяет входит ли год в исторический диапазон"""
    if 'HISTORY_START_YEAR' in globals() and 'HISTORY_END_YEAR' in globals():
        return HISTORY_START_YEAR <= year <= HISTORY_END_YEAR
    return 2010 <= year <= 2024

def get_history_years(data_dict):
    """Возвращает отсортированный список лет из словаря"""
    return sorted([y for y in data_dict.keys() if is_history_year(y)])

def _open_data_mart():
    try:
        return get_data_mart(ROOT, COMPANY)
    except Exception as exc:
        print(f"⚠️ Не удалось подключиться к витрине данных: {exc}")
        return None

def _resolve_model_version(mart):
    global MODEL_VERSION
    if MODEL_VERSION:
        return MODEL_VERSION
    try:
        versions = mart.get_existing_versions()
        if versions:
            MODEL_VERSION = versions[0]['version']
            return MODEL_VERSION
    except Exception as exc:
        print(f"⚠️ Не удалось определить версию модели: {exc}")
    return None

def load_history_from_db(statement_type='IS', canonical=False):
    stmt = statement_type.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        try:
            if stmt == 'IS':
                df = mart.get_history_income_statement(canonical=canonical)
            elif stmt == 'BS':
                df = mart.get_history_balance_sheet(canonical=canonical)
            else:
                df = mart.get_history_cash_flow(canonical=canonical)
            if df is not None and not df.empty:
                print(f"✅ История {stmt} загружена из витрины")
        except Exception as exc:
            print(f"⚠️ Не удалось загрузить историю {stmt} из витрины: {exc}")
        finally:
            mart.close()
    file_map = {
        'IS': f"history/is_history_{COMPANY}.csv",
        'BS': f"history/bs_history_{COMPANY}.csv",
        'CF': f"history/cf_history_{COMPANY}.csv"
    }
    if df is None or df.empty:
        csv_path = croot / file_map.get(stmt, '')
        if csv_path.exists():
            print(f"📄 История {stmt} загружена из CSV")
            return pd.read_csv(csv_path)
        print(f"❌ История {stmt} не найдена ни в витрине, ни в CSV")
        return pd.DataFrame()
    if not canonical and 'metric' in df.columns:
        year_cols = [c for c in df.columns if str(c).isdigit()]
        if year_cols:
            df_long = df.set_index('metric')[year_cols].T.reset_index().rename(columns={'index': 'year'})
            df_long['year'] = df_long['year'].astype(int)
            return df_long
    return df

def load_model_table(table_name: str, canonical: bool = False, csv_filename: Optional[str] = None):
    table_key = table_name.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        version = _resolve_model_version(mart)
        if version:
            try:
                df = mart.get_model_results(version, table_key, canonical=canonical)
                if df is not None and not df.empty:
                    print(f"✅ Таблица {table_key} загружена из витрины (версия {version})")
                    mart.close()
                    return df
            except Exception as exc:
                print(f"⚠️ Не удалось загрузить таблицу {table_key} из витрины: {exc}")
            finally:
                mart.close()
        else:
            mart.close()
    if csv_filename:
        csv_path = croot / csv_filename
        if csv_path.exists():
            print(f"⚠️ Используем legacy CSV для {table_key}: {csv_path.relative_to(ROOT)}")
            return pd.read_csv(csv_path)
    print(f"⚠️ Таблица {table_key} не найдена ни в витрине, ни в CSV")
    return pd.DataFrame()

def load_model_forecast_from_db(statement_type='IS', canonical=True):
    stmt = statement_type.upper()
    csv_filename = f"outputs/model/3statement_{stmt}.csv"
    return load_model_table(stmt, canonical=canonical, csv_filename=csv_filename)

def load_model_parameter_table(parameter_name: str, csv_filename: Optional[str] = None):
    return load_model_table(parameter_name.upper(), canonical=False, csv_filename=csv_filename)

def load_company_csv(relative_path: str):
    csv_path = croot / relative_path
    if csv_path.exists():
        print(f"⚠️ Используем legacy CSV: {csv_path.relative_to(ROOT)}")
        return pd.read_csv(csv_path)
    print(f"❌ Legacy CSV не найден: {relative_path}")
    return pd.DataFrame()


ROOT: /home
Engine exists: True
Company root: /home/companies/us_steel


## 1️⃣ Проверка конфигурации <a id="section-1"></a>

In [2]:
display(Markdown("### Конфигурация из project.yaml"))

proj_yaml_path = croot / "configs" / "project.yaml"
if not proj_yaml_path.exists():
    print(f"❌ Файл конфигурации не найден: {proj_yaml_path}")
else:
    proj_yaml = yaml.safe_load(proj_yaml_path.read_text(encoding='utf-8'))
    
    print(f"✅ Конфигурация загружена")
    print(f"\n📊 Основные настройки:")
    print(f"  - Модель: {proj_yaml.get('model', {}).get('mode', 'standard')}")
    
    if 'model' in proj_yaml and 'standard' in proj_yaml['model']:
        std_cfg = proj_yaml['model']['standard']
        print(f"\n📈 Revenue:")
        rev_cfg = std_cfg.get('revenue', {})
        print(f"  - Driver: {rev_cfg.get('driver', 'N/A')}")
        method_str = 'elastic_net' if rev_cfg.get('use_elastic_net', False) else rev_cfg.get('fallback_growth', 'N/A')
        print(f"  - Method: {method_str}")
        print(f"  - Target Transform: {rev_cfg.get('target_transform', 'N/A')}")
        print(f"  - Feature Transform: {rev_cfg.get('feature_transform', 'N/A')}")
        print(f"  - Chain-link Anchor: {rev_cfg.get('chainlink_anchor', 'N/A')}")
        
        print(f"\n💰 WC:")
        wc_cfg = std_cfg.get('wc', {})
        print(f"  - Метод: {wc_cfg.get('method', 'N/A')}")
        print(f"  - DSO: {wc_cfg.get('dso_days', 'N/A')}")
        print(f"  - DIO: {wc_cfg.get('dio_days', 'N/A')}")
        print(f"  - DPO: {wc_cfg.get('dpo_days', 'N/A')}")
        print(f"  - Transform Days: {wc_cfg.get('transform_days', 'N/A')}")
        
        print(f"\n💳 Debt:")
        debt_cfg = std_cfg.get('debt', {})
        print(f"  - Interest treatment: {debt_cfg.get('interest_treatment', 'N/A')}")
        refinancing_enable = debt_cfg.get('refinancing', {}).get('enable', False)
        print(f"  - Refinancing enabled: {refinancing_enable}")
        rc_cfg = debt_cfg.get('rc', {})
        if rc_cfg.get('enable', False):
            print(f"  - RC enabled: True")
            print(f"  - RC limit: {rc_cfg.get('limit', 'N/A')}")
            print(f"  - Min cash: {rc_cfg.get('min_cash', 'N/A')}")
        
        print(f"\n📊 COGS:")
        cogs_cfg = std_cfg.get('cogs', {})
        print(f"  - Use PPI uplift: {cogs_cfg.get('use_ppi_uplift', False)}")
        clamp_cfg = cogs_cfg.get('clamp', {})
        if clamp_cfg:
            print(f"  - Clamp: [{clamp_cfg.get('min', 'N/A')}, {clamp_cfg.get('max', 'N/A')}]")
        
        print(f"\n🔬 Train/Test:")
        # Проверяем оба возможных места в конфиге
        tt_cfg_model = proj_yaml.get('model', {}).get('train_test', {})
        tt_cfg_std = std_cfg.get('train_test', {})
        tt_cfg = tt_cfg_model or tt_cfg_std
        train_end = tt_cfg.get('train_end_year', None) if tt_cfg else None
        if train_end:
            print(f"  - Train end year: {train_end}")
        else:
            print(f"  - Train end year: N/A (не настроен)")
        
        # ГЛОБАЛЬНЫЕ ПЕРЕМЕННЫЕ ДЛЯ ПЕРИОДОВ (для использования во всем ноутбуке)
        periods_config = proj_yaml.get("model", {}).get("standard", {}).get("periods", {})
        HISTORY_START_YEAR = periods_config.get("history_start_year", 2010)
        HISTORY_END_YEAR = periods_config.get("history_end_year", 2024)
        FORECAST_START_YEAR = periods_config.get("forecast_start_year", 2025)
        FORECAST_END_YEAR = periods_config.get("forecast_end_year", 2029)
        
        print(f"\n📅 Параметры периодов (глобальные переменные):")
        print(f"  - История: {HISTORY_START_YEAR}-{HISTORY_END_YEAR}")
        print(f"  - Прогноз: {FORECAST_START_YEAR}-{FORECAST_END_YEAR}")


### Конфигурация из project.yaml

✅ Конфигурация загружена

📊 Основные настройки:
  - Модель: custom

📈 Revenue:
  - Driver: steel_price_hrc
  - Method: elastic_net
  - Target Transform: dln
  - Feature Transform: dln
  - Chain-link Anchor: history_last

💰 WC:
  - Метод: days
  - DSO: 45
  - DIO: 60
  - DPO: 50
  - Transform Days: dln

💳 Debt:
  - Interest treatment: separate_line
  - Refinancing enabled: True
  - RC enabled: True
  - RC limit: 2000.0
  - Min cash: 500000000.0

📊 COGS:
  - Use PPI uplift: False
  - Clamp: [0.55, 0.98]

🔬 Train/Test:
  - Train end year: 2022

📅 Параметры периодов (глобальные переменные):
  - История: 2010-2024
  - Прогноз: 2025-2029


## 2️⃣ Загрузка входных данных <a id="section-2"></a>

In [3]:
display(Markdown("### Проверка наличия входных данных"))

history_checks = []
history_keys = {'IS': 'history/is_history_{company}.csv',
                'BS': 'history/bs_history_{company}.csv',
                'CF': 'history/cf_history_{company}.csv'}

mart = _open_data_mart()
db_exists = False
if mart is not None:
    try:
        hist_is_db = mart.get_history_income_statement(canonical=False)
        db_exists = hist_is_db is not None and not hist_is_db.empty
    except Exception as exc:
        print(f"⚠️ Не удалось прочитать историю из витрины: {exc}")

print(f"\n🗄️ Витрина данных: {'✅' if db_exists else '⚠️ (данных нет)'}")
for stmt_type, template in history_keys.items():
    entry = {
        'type': stmt_type,
        'source': 'Data Mart' if db_exists else 'CSV',
        'path': 'Data Mart' if db_exists else template.format(company=COMPANY)
    }
    if db_exists:
        entry['exists'] = '✅'
    else:
        csv_df = load_company_csv(template.format(company=COMPANY))
        entry['exists'] = '✅' if not csv_df.empty else '❌'
        if not csv_df.empty:
            entry['rows'] = len(csv_df)
            entry['cols'] = len(csv_df.columns)
    history_checks.append(entry)

display(pd.DataFrame(history_checks))

macro_rows = []
factors = []
project_cfg = {}
project_path = croot / "configs" / "project.yaml"
if project_path.exists():
    try:
        project_cfg = yaml.safe_load(project_path.read_text(encoding="utf-8")) or {}
    except Exception as exc:
        print(f"⚠️ Не удалось прочитать project.yaml для macro-факторов: {exc}")
macro_cfg = project_cfg.get("macro_forecast", {})
factors = macro_cfg.get("factors", [])
if mart is not None:
    for factor in factors or []:
        try:
            forecast = mart.get_macro_forecast(factor) or {}
        except Exception as exc:
            print(f"⚠️ Не удалось прочитать прогноз для {factor}: {exc}")
            forecast = {}
        macro_rows.append({
            'factor': factor,
            'years_available': len(forecast),
            'start_year': min(forecast) if forecast else None,
            'end_year': max(forecast) if forecast else None
        })
    mart.close()
    if macro_rows:
        display(Markdown("📊 Макро-прогнозы (витрина данных)"))
        display(pd.DataFrame(macro_rows))
else:
    print("⚠️ Витрина данных недоступна")


### Проверка наличия входных данных


🗄️ Витрина данных: ✅


,type,source,path,exists
0,IS,Data Mart,Data Mart,✅
1,BS,Data Mart,Data Mart,✅
2,CF,Data Mart,Data Mart,✅


📊 Макро-прогнозы (витрина данных)

,factor,years_available,start_year,end_year
0,gdp_us,5,2025,2029
1,industrial_production_us,5,2025,2029
2,dxy,5,2025,2029
3,steel_price_hrc,5,2025,2029
4,iron_ore_price,5,2025,2029
5,cpi_us,5,2025,2029
6,ppi_us,5,2025,2029
7,gdp_world,5,2025,2029


## 2.1 Анализ возможностей построения модели <a id="section-2-1"></a>

In [4]:
display(Markdown("### 🔍 Анализ доступных данных и возможностей моделирования"))

mart = _open_data_mart()
if mart is None:
    print("❌ Не удалось подключиться к витрине данных — пропускаем аналитические проверки.")
else:
    detected_model_type = mart.detect_model_type()
    config_model_mode = proj_yaml.get('model', {}).get('mode', 'standard')

    print(f"📋 Тип модели в конфиге: {config_model_mode}")
    print(f"🔍 Определенный тип модели из истории: {detected_model_type}")
    if detected_model_type != config_model_mode:
        print(f"⚠️  ВНИМАНИЕ: Тип модели в конфиге ({config_model_mode}) не совпадает с определенным из истории ({detected_model_type})")
        print(f"   Рекомендуется установить model.mode: {detected_model_type} в project.yaml")
    mart.close()



### 🔍 Анализ доступных данных и возможностей моделирования

📋 Тип модели в конфиге: custom
🔍 Определенный тип модели из истории: custom


### 2.1.1 Проверка доступности канонических метрик

In [5]:
mart = _open_data_mart()
if mart is None:
    print("❌ Витрина данных недоступна — пропускаем анализ доступности метрик.")
    availability_check = {'IS': {}, 'BS': {}, 'CF': {}}
else:
    standard_is = mart.get_canonical_metrics_from_db('IS')
    standard_bs = mart.get_canonical_metrics_from_db('BS')
    standard_cf = mart.get_canonical_metrics_from_db('CF')

    hist_is_df = mart.get_history_income_statement(canonical=True)
    hist_bs_df = mart.get_history_balance_sheet(canonical=True)
    hist_cf_df = mart.get_history_cash_flow(canonical=True)
    mart.close()

    def _wide_to_year_index(df):
        if df is None or df.empty or 'metric' not in df.columns:
            return pd.DataFrame()
        wide = df.set_index('metric')
        year_cols = [c for c in wide.columns if str(c).isdigit()]
        return wide[year_cols].T.apply(pd.to_numeric, errors='coerce')

    hist_is_df = _wide_to_year_index(hist_is_df)
    hist_bs_df = _wide_to_year_index(hist_bs_df)
    hist_cf_df = _wide_to_year_index(hist_cf_df)

    availability_check = {'IS': {}, 'BS': {}, 'CF': {}}

    def check_metric_availability(df, metric_name):
        if df is None or df.empty:
            return False, 0, []
        metric_lower = metric_name.lower()
        columns = {str(c).lower(): c for c in df.columns}
        if metric_lower not in columns:
            return False, 0, []
        col = columns[metric_lower]
        series = df[col].dropna()
        years = [int(y) for y, val in series.items() if pd.notna(val) and val != 0]
        return len(years) > 0, len(years), sorted(years)

    for metric in standard_is:
        has_data, count, years = check_metric_availability(hist_is_df, metric)
        availability_check['IS'][metric] = {'available': has_data, 'years_count': count, 'years': years}

    for metric in standard_bs:
        has_data, count, years = check_metric_availability(hist_bs_df, metric)
        availability_check['BS'][metric] = {'available': has_data, 'years_count': count, 'years': years}

    for metric in standard_cf:
        has_data, count, years = check_metric_availability(hist_cf_df, metric)
        availability_check['CF'][metric] = {'available': has_data, 'years_count': count, 'years': years}

    print(f"✅ Проверено метрик: IS={len(availability_check['IS'])}, BS={len(availability_check['BS'])}, CF={len(availability_check['CF'])}")



✅ Проверено метрик: IS=20, BS=46, CF=40


### 2.1.2 Чеклист возможностей построения модели

In [6]:
# Строим чеклист возможностей модели
display(Markdown("### 📋 Чеклист возможностей построения модели"))

checklist_data = []

# Проверяем возможность построения Standard модели
standard_required = {
    'IS': ['revenue', 'cogs', 'sga', 'ebitda', 'ebit', 'interest_expense', 'ebt', 'tax_expense', 'net_income'],
    'BS': ['cash', 'accounts_receivable', 'inventory', 'ppe_net', 'accounts_payable', 'short_term_debt', 'long_term_debt'],
    'CF': ['net_income', 'depreciation', 'wc_delta', 'cfo', 'capex', 'cfi', 'cff']
}

def can_build_model(model_type, required_metrics):
    """Проверяет возможность построения модели определенного типа"""
    missing = []
    partial = []
    
    for stmt_type, metrics in required_metrics.items():
        for metric in metrics:
            check = availability_check[stmt_type].get(metric, {})
            if not check.get('available', False):
                missing.append(f"{stmt_type}.{metric}")
            elif check.get('years_count', 0) < 3:
                partial.append(f"{stmt_type}.{metric} ({check['years_count']} лет)")
    
    can_build = len(missing) == 0
    quality = "high" if len(partial) == 0 else "medium" if len(partial) < len(required_metrics) * 0.3 else "low"
    
    return can_build, missing, partial, quality

# Проверяем Standard модель
can_standard, missing_standard, partial_standard, quality_standard = can_build_model('standard', standard_required)

# Проверяем Custom модель (без DTA/DTL)
custom_without_tax_required = standard_required.copy()
can_custom_wo_tax, missing_custom_wo_tax, partial_custom_wo_tax, quality_custom_wo_tax = can_build_model('custom_wo_tax', custom_without_tax_required)

# Проверяем Custom модель (с DTA/DTL)
custom_with_tax_required = standard_required.copy()
custom_with_tax_required['BS'].extend(['dta', 'dtl', 'taxes_payable'])
can_custom_with_tax, missing_custom_with_tax, partial_custom_with_tax, quality_custom_with_tax = can_build_model('custom_with_tax', custom_with_tax_required)

# Формируем таблицу результатов
checklist_rows = [
    {
        'Тип модели': 'Standard',
        'Можно строить': '✅ Да' if can_standard else '❌ Нет',
        'Качество данных': quality_standard,
        'Отсутствует': ', '.join(missing_standard[:5]) + ('...' if len(missing_standard) > 5 else ''),
        'Неполные данные': f"{len(partial_standard)} метрик"
    },
    {
        'Тип модели': 'Custom (без DTA/DTL)',
        'Можно строить': '✅ Да' if can_custom_wo_tax else '❌ Нет',
        'Качество данных': quality_custom_wo_tax,
        'Отсутствует': ', '.join(missing_custom_wo_tax[:5]) + ('...' if len(missing_custom_wo_tax) > 5 else ''),
        'Неполные данные': f"{len(partial_custom_wo_tax)} метрик"
    },
    {
        'Тип модели': 'Custom (с DTA/DTL)',
        'Можно строить': '✅ Да' if can_custom_with_tax else '❌ Нет',
        'Качество данных': quality_custom_with_tax,
        'Отсутствует': ', '.join(missing_custom_with_tax[:5]) + ('...' if len(missing_custom_with_tax) > 5 else ''),
        'Неполные данные': f"{len(partial_custom_with_tax)} метрик"
    }
]

checklist_df = pd.DataFrame(checklist_rows)
display(HTML(checklist_df.to_html(index=False, escape=False, classes='table table-striped')))

### 📋 Чеклист возможностей построения модели

Тип модели,Можно строить,Качество данных,Отсутствует,Неполные данные
Standard,✅ Да,high,,0 метрик
Custom (без DTA/DTL),✅ Да,high,,0 метрик
Custom (с DTA/DTL),✅ Да,high,,0 метрик


### 2.1.3 Методы моделирования статей и полнота данных


In [7]:
# Детальный анализ методов моделирования
display(Markdown("### 📊 Методы моделирования статей и полнота данных"))

modeling_methods = {
    'IS': {
        'revenue': 'Регрессия с макрофакторами / Темп роста / EWA',
        'cogs': 'PPI индексация / Ratio к Revenue / Clamp',
        'sga': 'EWA темпа роста / CPI индексация / Ratio к Revenue',
        'depreciation_owned': 'PP&E corkscrew (амортизация)',
        'depreciation_rou': 'Lease schedule (ROU амортизация)',
        'amortization': 'Intangibles corkscrew',
        'interest_expense': 'Debt schedule (расчет по ставкам)',
        'lease_interest': 'Lease schedule (проценты)',
        'tax_expense': 'Tax schedule (statutory rate / effective)',
    },
    'BS': {
        'cash': 'Cash bridge (CFF → CFI → CFO → Cash)',
        'accounts_receivable': 'WC days (DSO)',
        'inventory': 'WC days (DIO)',
        'accounts_payable': 'WC days (DPO)',
        'ppe_gross': 'PP&E corkscrew (Gross = Opening + CapEx - Disposals)',
        'ppe_accum_dep': 'PP&E corkscrew (Accumulated Depreciation)',
        'ppe_net': 'PP&E corkscrew (Net = Gross - Accum Dep)',
        'rou_asset': 'Lease schedule (ROU Asset)',
        'intangibles': 'Intangibles corkscrew',
        'short_term_debt': 'Debt schedule (ST часть)',
        'long_term_debt': 'Debt schedule (LT часть)',
        'current_lease_liability': 'Lease schedule (ST часть)',
        'long_term_lease_liability': 'Lease schedule (LT часть)',
        'dta': 'Tax schedule (Deferred Tax Assets)',
        'dtl': 'Tax schedule (Deferred Tax Liabilities)',
        'taxes_payable': 'Tax schedule (Taxes Payable)',
        'retained_earnings': 'Retained Earnings = Opening + Net Income - Dividends',
    },
    'CF': {
        'net_income': 'Из IS',
        'depreciation': 'Из PP&E corkscrew',
        'amortization': 'Из Intangibles corkscrew',
        'wc_delta': 'WC corkscrew (ΔAR + ΔInv - ΔAP - ΔOther)',
        'tax_paid': 'Tax schedule (Taxes Paid)',
        'cfo': 'Net Income + D&A - WC Delta - Tax Paid',
        'capex': 'CapEx policy (ratio / fixed / driver-based)',
        'disposal_proceeds': 'Disposals из PP&E corkscrew',
        'cfi': 'CapEx + Disposals + Other CFI',
        'debt_borrowings': 'Debt schedule (новые заимствования)',
        'debt_repayments': 'Debt schedule (погашения)',
        'lease_payments_cff': 'Lease schedule (основной долг)',
        'cff': 'Borrowings - Repayments - Lease Payments - Dividends',
        'net_change': 'CFO + CFI + CFF',
    }
}

# Формируем детальную таблицу
detailed_rows = []
for stmt_type in ['IS', 'BS', 'CF']:
    for metric, method in modeling_methods.get(stmt_type, {}).items():
        check = availability_check[stmt_type].get(metric, {})
        has_data = check.get('available', False)
        years_count = check.get('years_count', 0)
        
        # Определяем качество моделирования
        if has_data and years_count >= 5:
            quality = '🟢 Высокая (5+ лет истории)'
            can_model_precise = True
        elif has_data and years_count >= 3:
            quality = '🟡 Средняя (3-4 года истории)'
            can_model_precise = True
        elif has_data:
            quality = '🟠 Низкая (1-2 года истории)'
            can_model_precise = True
        else:
            quality = '🔴 Нет данных (моделирование через assumptions)'
            can_model_precise = False
        
        detailed_rows.append({
            'Отчет': stmt_type,
            'Метрика': metric,
            'Есть данные': '✅' if has_data else '❌',
            'Лет истории': years_count,
            'Качество': quality,
            'Метод моделирования': method
        })

detailed_df = pd.DataFrame(detailed_rows)
display(HTML(detailed_df.to_html(index=False, escape=False, classes='table table-striped table-sm', max_rows=50)))

### 📊 Методы моделирования статей и полнота данных

Отчет,Метрика,Есть данные,Лет истории,Качество,Метод моделирования
IS,revenue,✅,15,🟢 Высокая (5+ лет истории),Регрессия с макрофакторами / Темп роста / EWA
IS,cogs,✅,15,🟢 Высокая (5+ лет истории),PPI индексация / Ratio к Revenue / Clamp
IS,sga,✅,15,🟢 Высокая (5+ лет истории),EWA темпа роста / CPI индексация / Ratio к Revenue
IS,depreciation_owned,✅,15,🟢 Высокая (5+ лет истории),PP&E corkscrew (амортизация)
IS,depreciation_rou,✅,6,🟢 Высокая (5+ лет истории),Lease schedule (ROU амортизация)
IS,amortization,✅,15,🟢 Высокая (5+ лет истории),Intangibles corkscrew
IS,interest_expense,✅,15,🟢 Высокая (5+ лет истории),Debt schedule (расчет по ставкам)
IS,lease_interest,✅,6,🟢 Высокая (5+ лет истории),Lease schedule (проценты)
IS,tax_expense,✅,15,🟢 Высокая (5+ лет истории),Tax schedule (statutory rate / effective)
BS,cash,✅,15,🟢 Высокая (5+ лет истории),Cash bridge (CFF → CFI → CFO → Cash)


### 2.1.4 Рекомендации


In [8]:
# Сводка и рекомендации

recommendations = []

# Рекомендация по типу модели
if detected_model_type == 'custom':
    if can_custom_with_tax:
        recommendations.append("✅ Рекомендуется использовать **Custom модель с Tax Schedule** (DTA/DTL/Taxes Payable) для максимальной точности")
    elif can_custom_wo_tax:
        recommendations.append("✅ Рекомендуется использовать **Custom модель без Tax Schedule** (расширенная форма без DTA/DTL)")
    else:
        recommendations.append("⚠️ Custom модель не может быть построена. Рекомендуется использовать **Standard модель**")
elif detected_model_type == 'standard':
    if can_standard:
        recommendations.append("✅ Достаточно данных для **Standard модели**")
    else:
        recommendations.append("❌ Недостаточно данных для Standard модели. Проверьте наличие обязательных метрик.")

# Рекомендации по улучшению качества
total_metrics = len(availability_check['IS']) + len(availability_check['BS']) + len(availability_check['CF'])
available_metrics = sum(
    sum(1 for v in stmt.values() if v.get('available', False))
    for stmt in availability_check.values()
)
coverage = (available_metrics / total_metrics * 100) if total_metrics > 0 else 0

if coverage < 70:
    recommendations.append(f"⚠️ Покрытие данных: {coverage:.1f}%. Рекомендуется добавить больше исторических данных для повышения точности модели.")
elif coverage < 85:
    recommendations.append(f"✅ Покрытие данных: {coverage:.1f}%. Хорошее покрытие, можно улучшить, добавив недостающие метрики.")
else:
    recommendations.append(f"✅ Покрытие данных: {coverage:.1f}%. Отличное покрытие данных!")

missing_by_stmt = {
    stmt: sorted(metric for metric, info in metrics.items() if not info.get('available', False))
    for stmt, metrics in availability_check.items()
}

if any(missing_by_stmt.values()):
    details = []
    for stmt, metrics in missing_by_stmt.items():
        if not metrics:
            continue
        formatted = ', '.join(metrics[:5]) + ('…' if len(metrics) > 5 else '')
        details.append(f"- **{stmt}**: {formatted}")
    if details:
        recommendations.append(
            "📈 Чтобы повысить покрытие и уменьшить долю assumptions, добавьте исторические ряды:<br>"
            + "<br>".join(details)
        )
    if missing_by_stmt.get('CF'):
        recommendations.append(
            "💧 История по `CF.tax_paid`, `CF.debt_borrowings` и `CF.debt_repayments` позволит точнее моделировать налоговые платежи и долговые Cash Flow."
        )

# Проверка критических метрик
critical_missing = []
for metric in ['revenue', 'cogs', 'sga', 'cash', 'accounts_receivable', 'inventory', 'accounts_payable']:
    stmt = 'IS' if metric in ['revenue', 'cogs', 'sga'] else 'BS'
    check = availability_check[stmt].get(metric, {})
    if not check.get('available', False):
        critical_missing.append(f"{stmt}.{metric}")

if critical_missing:
    recommendations.append(f"❌ КРИТИЧНО: Отсутствуют обязательные метрики: {', '.join(critical_missing)}")

for rec in recommendations:
    display(Markdown(rec))

print()
print("📊 Итоговая статистика:")
print(f"  - Всего метрик в канонической форме: {total_metrics}")
print(f"  - Доступно метрик в истории: {available_metrics}")
print(f"  - Покрытие данных: {coverage:.1f}%")
print(f"  - Определенный тип модели: {detected_model_type}")
print(f"  - Тип модели в конфиге: {config_model_mode}")


✅ Рекомендуется использовать **Custom модель с Tax Schedule** (DTA/DTL/Taxes Payable) для максимальной точности

✅ Покрытие данных: 92.5%. Отличное покрытие данных!

📈 Чтобы повысить покрытие и уменьшить долю assumptions, добавьте исторические ряды:<br>- **IS**: rnd<br>- **BS**: short_term_investments, total_liab_equity, total_non_current_assets, total_non_current_liabilities<br>- **CF**: cfo_other, interest_paid, lease_interest_cfo

💧 История по `CF.tax_paid`, `CF.debt_borrowings` и `CF.debt_repayments` позволит точнее моделировать налоговые платежи и долговые Cash Flow.


📊 Итоговая статистика:
  - Всего метрик в канонической форме: 106
  - Доступно метрик в истории: 98
  - Покрытие данных: 92.5%
  - Определенный тип модели: custom
  - Тип модели в конфиге: custom


### 📊 Сравнительная таблица: Standard vs Custom модель

Эта таблица показывает, какие метрики входят в **Standard** модель, а какие добавляются в **Custom** модель.

> 📖 **Детальное сравнение:** См. `docs/MODEL_ENGINES_COMPARISON.md` для полной сравнительной таблицы по всем статьям и методам моделирования.


### 2.1.5 Метрики, доступные в Data Mart


In [9]:
# Получаем канонические метрики для Standard и Custom из БД
from engine.database.sqlite_wrapper import get_company_db

db = get_company_db(ROOT, COMPANY)
cursor = db.conn.cursor()

# Получаем метрики для Standard модели
cursor.execute("""
    SELECT statement_type, metric_name, metric_order
    FROM canonical_metrics
    WHERE model_type = 'standard'
    ORDER BY statement_type, metric_order
""")
standard_metrics = {}
for row in cursor.fetchall():
    stmt = row[0]
    metric = row[1]
    if stmt not in standard_metrics:
        standard_metrics[stmt] = []
    standard_metrics[stmt].append(metric)

# Получаем метрики для Custom модели
cursor.execute("""
    SELECT statement_type, metric_name, metric_order
    FROM canonical_metrics
    WHERE model_type = 'custom'
    ORDER BY statement_type, metric_order
""")
custom_metrics = {}
for row in cursor.fetchall():
    stmt = row[0]
    metric = row[1]
    if stmt not in custom_metrics:
        custom_metrics[stmt] = []
    custom_metrics[stmt].append(metric)

print(f"✅ Загружено метрик из БД:")
print(f"  Standard: IS={len(standard_metrics.get('IS', []))}, BS={len(standard_metrics.get('BS', []))}, CF={len(standard_metrics.get('CF', []))}")
print(f"  Custom: IS={len(custom_metrics.get('IS', []))}, BS={len(custom_metrics.get('BS', []))}, CF={len(custom_metrics.get('CF', []))}")

✅ Загружено метрик из БД:
  Standard: IS=20, BS=46, CF=40
  Custom: IS=20, BS=46, CF=40


### 2.1.6 Сравнительные таблицы источников данных

Сводим Standard и Custom наборы метрик по каждому отчётному блоку, чтобы убедиться, что Data Mart содержит полный объём данных для расширенной модели.


In [10]:
# Формируем сравнительную таблицу для каждого отчета
comparison_tables = []

for statement_type in ['IS', 'BS', 'CF']:
    standard_list = set(standard_metrics.get(statement_type, []))
    custom_list = set(custom_metrics.get(statement_type, []))
    
    # Определяем категории метрик
    in_both = sorted(standard_list & custom_list)
    only_custom = sorted(custom_list - standard_list)
    only_standard = sorted(standard_list - custom_list)
    
    # Формируем таблицу
    rows = []
    
    # Сначала метрики, которые в обеих моделях
    for metric in in_both:
        rows.append({
            'Отчет': statement_type,
            'Метрика': metric,
            'Standard': '✅',
            'Custom': '✅',
            'Категория': 'Обе модели'
        })
    
    # Затем метрики только в Standard (если есть)
    for metric in only_standard:
        rows.append({
            'Отчет': statement_type,
            'Метрика': metric,
            'Standard': '✅',
            'Custom': '❌',
            'Категория': 'Только Standard'
        })
    
    # Затем метрики только в Custom
    for metric in only_custom:
        rows.append({
            'Отчет': statement_type,
            'Метрика': metric,
            'Standard': '❌',
            'Custom': '✅',
            'Категория': 'Только Custom'
        })
    
    comparison_tables.append({
        'statement': statement_type,
        'rows': rows,
        'stats': {
            'in_both': len(in_both),
            'only_standard': len(only_standard),
            'only_custom': len(only_custom),
            'total_standard': len(standard_list),
            'total_custom': len(custom_list)
        }
    })

print(f"✅ Сформировано {len(comparison_tables)} сравнительных таблиц")


✅ Сформировано 3 сравнительных таблиц


#### 2.1.7 Сравнение состава метрик по отчётам

Ниже — развёрнутые таблицы для IS, BS и CF, показывающие, какие показатели присутствуют в стандартной и кастомной конфигурации модели.


In [11]:
# Выводим сравнительные таблицы для каждого отчета
for comp in comparison_tables:
    stmt = comp['statement']
    stats = comp['stats']
    
    display(Markdown(f"#### 📋 {stmt} - Income Statement" if stmt == 'IS' else f"#### 📋 {stmt} - Balance Sheet" if stmt == 'BS' else f"#### 📋 {stmt} - Cash Flow Statement"))
    
    # Статистика
    display(Markdown(f"""
    **Статистика:**
    - ✅ В обеих моделях: {stats['in_both']} метрик
    - 📊 Только в Standard: {stats['only_standard']} метрик
    - 🎯 Только в Custom: {stats['only_custom']} метрик
    - 📈 Всего Standard: {stats['total_standard']} метрик
    - 📈 Всего Custom: {stats['total_custom']} метрик
    """))
    
    # Таблица
    df_comp = pd.DataFrame(comp['rows'])
    display(HTML(df_comp.to_html(index=False, escape=False, classes='table table-striped table-bordered')))

#### 📋 IS - Income Statement


    **Статистика:**
    - ✅ В обеих моделях: 20 метрик
    - 📊 Только в Standard: 0 метрик
    - 🎯 Только в Custom: 0 метрик
    - 📈 Всего Standard: 20 метрик
    - 📈 Всего Custom: 20 метрик
    

Отчет,Метрика,Standard,Custom,Категория
IS,amortization,✅,✅,Обе модели
IS,cogs,✅,✅,Обе модели
IS,depreciation_owned,✅,✅,Обе модели
IS,depreciation_rou,✅,✅,Обе модели
IS,ebit,✅,✅,Обе модели
IS,ebitda,✅,✅,Обе модели
IS,ebt,✅,✅,Обе модели
IS,eps_basic,✅,✅,Обе модели
IS,eps_diluted,✅,✅,Обе модели
IS,gain_loss_on_disposal,✅,✅,Обе модели


#### 📋 BS - Balance Sheet


    **Статистика:**
    - ✅ В обеих моделях: 46 метрик
    - 📊 Только в Standard: 0 метрик
    - 🎯 Только в Custom: 0 метрик
    - 📈 Всего Standard: 46 метрик
    - 📈 Всего Custom: 46 метрик
    

Отчет,Метрика,Standard,Custom,Категория
BS,accounts_payable,✅,✅,Обе модели
BS,accounts_receivable,✅,✅,Обе модели
BS,accrued_interest,✅,✅,Обе модели
BS,accrued_liabilities,✅,✅,Обе модели
BS,aoci,✅,✅,Обе модели
BS,ap_related_parties,✅,✅,Обе модели
BS,apic,✅,✅,Обе модели
BS,cash,✅,✅,Обе модели
BS,common_stock_par,✅,✅,Обе модели
BS,current_lease_liability,✅,✅,Обе модели


#### 📋 CF - Cash Flow Statement


    **Статистика:**
    - ✅ В обеих моделях: 40 метрик
    - 📊 Только в Standard: 0 метрик
    - 🎯 Только в Custom: 0 метрик
    - 📈 Всего Standard: 40 метрик
    - 📈 Всего Custom: 40 метрик
    

Отчет,Метрика,Standard,Custom,Категория
CF,amortization,✅,✅,Обе модели
CF,capex,✅,✅,Обе модели
CF,cash_ending,✅,✅,Обе модели
CF,cash_opening,✅,✅,Обе модели
CF,cf_fx_effect,✅,✅,Обе модели
CF,cff,✅,✅,Обе модели
CF,cff_amortization_financing_costs,✅,✅,Обе модели
CF,cff_borrowings,✅,✅,Обе модели
CF,cff_dividends,✅,✅,Обе модели
CF,cff_equity_issuance,✅,✅,Обе модели


#### 2.1.8 Сводка метрик только для Custom


In [12]:
# Сводная таблица: что добавляется в Custom
display(Markdown("### 🎯 Итого: Что добавляется в Custom модель"))

summary_rows = []
for comp in comparison_tables:
    stmt = comp['statement']
    only_custom_metrics = [
        r['Метрика'] for r in comp['rows'] 
        if r['Категория'] == 'Только Custom'
    ]
    
    if only_custom_metrics:
        for metric in only_custom_metrics:
            summary_rows.append({
                'Отчет': stmt,
                'Метрика (только в Custom)': metric
            })
    else:
        summary_rows.append({
            'Отчет': stmt,
            'Метрика (только в Custom)': '— Нет дополнительных метрик'
        })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    display(HTML(summary_df.to_html(index=False, escape=False, classes='table table-striped table-bordered')))

### 🎯 Итого: Что добавляется в Custom модель

Отчет,Метрика (только в Custom)
IS,— Нет дополнительных метрик
BS,— Нет дополнительных метрик
CF,— Нет дополнительных метрик


## 2.2 Препроцессинг драйверов <a id="section-2-2"></a>

In [13]:
SUMMARY_YEAR = -1

with get_data_mart(ROOT, COMPANY) as mart:
    metrics_wc = mart.get_model_metrics("wc_days")
    metrics_margin = mart.get_model_metrics("margin_ratios")
    metrics_capex = mart.get_model_metrics("capex")
    tax_schedule_raw = mart.get_tax_schedule()


def _prepare_tax_schedule(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=["metric_name", "year", "value", "source", "meta"])
    prepared = df.copy()
    prepared["metric_name"] = prepared["metric"].astype(str)
    prepared["source"] = "tax_schedule"
    prepared["meta"] = [{} for _ in range(len(prepared))]
    return prepared[["metric_name", "year", "value", "source", "meta"]]


def display_metric_group(title: str, df: pd.DataFrame) -> None:
    display(Markdown(f"### {title}"))
    if df is None or df.empty:
        display(Markdown("⚠️ Метрики отсутствуют"))
        return
    df = df.copy()
    if "meta" not in df.columns:
        df["meta"] = [{} for _ in range(len(df))]
    summary = df[df.get("year") == SUMMARY_YEAR].copy()
    history = df[df.get("year") != SUMMARY_YEAR].copy() if "year" in df else pd.DataFrame()
    if not summary.empty:
        summary_display = summary[["metric_name", "value", "source", "meta"]].reset_index(drop=True)
        summary_display["meta"] = summary_display["meta"].apply(lambda x: x if isinstance(x, dict) else {})
        display(Markdown("**Рекомендации**"))
        display(summary_display)
    if not history.empty:
        pivot = history.pivot_table(values="value", index="year", columns="metric_name")
        display(Markdown("**Исторические значения**"))
        display(pivot.sort_index())


tax_schedule = _prepare_tax_schedule(tax_schedule_raw)

metric_groups = [
    ("2.2.1 Рабочий капитал (дни)", metrics_wc),
    ("2.2.2 Маржинальность и налоги", metrics_margin),
    ("2.2.3 CapEx и амортизация", metrics_capex),
    ("2.2.4 Tax Schedule", tax_schedule),
]

for title, data in metric_groups:
    display_metric_group(title, data)


### 2.2.1 Рабочий капитал (дни)

**Рекомендации**

,metric_name,value,source,meta
0,ccc_ewa,4.204739,ewa,"{'halflife': 5, 'observations': 15}"
1,ccc_last,64.170388,last,{'year': 2024}
2,ccc_mean,-7.438397,mean,{'observations': 15}
3,ccc_recommended,4.204739,ewa,"{'mean': -7.438397264344499, 'last': 64.170388..."
4,dih_ewa,45.546092,ewa,"{'halflife': 5, 'observations': 15}"
5,dih_last,56.281650,last,{'year': 2024}
6,dih_mean,38.307184,mean,{'observations': 15}
7,dih_recommended,45.546092,ewa,"{'mean': 38.30718428046885, 'last': 56.2816500..."
8,dpo_ewa,42.382072,ewa,"{'halflife': 5, 'observations': 15}"
9,dpo_last,-7.398649,last,{'year': 2024}


**Исторические значения**

metric_name,ccc,dih,dpo,dso
year,,,,
2010,-16.872008,21.304201,39.016545,0.840336
2011,-15.179470,23.462294,39.376023,0.734259
2012,-15.853486,19.564663,36.106636,0.688488
2013,-13.694382,23.313874,38.309503,1.301248
2014,-18.437188,24.868651,44.187318,0.881479
2015,-17.515751,27.552733,46.259761,1.191277
2016,-33.734998,26.019952,60.763795,1.008845
2017,-44.827498,26.743373,72.166789,0.595918
2018,-10.120053,62.054449,72.792361,0.617859


...

**Исторические значения**

metric_name,capex_pct_revenue
year,
2010,0.038909
2011,0.042647
2012,0.040111
2013,0.029320
2014,0.029723
2015,0.049451
2016,0.033831
2017,0.041224
2018,0.070602


### 2.2.4 Tax Schedule

**Исторические значения**

metric_name,effectiveincometaxratecontinuingoperations
year,
2010,0.250000
2011,0.000000
2012,0.000000
2013,0.000000
2014,0.400000
2015,0.000000
2016,0.000000
2017,0.000000
2018,0.000000


### 2.2.1 Рабочий капитал (дни)

In [14]:
display_metric_group(metrics_wc)

TypeError: display_metric_group() missing 1 required positional argument: 'df'

### 2.2.2 Маржинальность и налоги

In [ ]:
display_metric_group(metrics_margin)

### 2.2.3 CapEx и амортизация

In [ ]:
display_metric_group(metrics_capex)

## 3️⃣ Построение модели <a id="section-3"></a>

In [15]:
# Проверка версии кода перед построением модели
import inspect
from engine.model3.core import ThreeStatementModel

# Проверяем, что исправление применено
try:
    source = inspect.getsource(ThreeStatementModel._solve_year)
    
    checks = {
        'dividends2 на основе ni2': '_plan_equity_flows(ni2, rev)' in source and 'dividends2, equity_issuance, equity_buyback = self._plan_equity_flows(ni2, rev)' in source,
        'retained earnings с dividends2': 'self.re[y] = self.re[prev_year] + ni2 - dividends2' in source,
        'cff2 с dividends2': 'cff2 = cff_debt + cff_equity2 + cff_lease' in source,
    }
    
    print("Проверка версии кода в ноутбуке:")
    print("="*80)
    for check_name, result in checks.items():
        status = "✅" if result else "❌"
        print(f"{status} {check_name}")
    
    if all(checks.values()):
        print("\n✅ Исправление применено! Код актуален.")
    else:
        print("\n❌ Исправление НЕ применено! Используется старая версия кода.")
        print("   Перезапустите ядро ноутбука (Kernel → Restart & Clear Output)")
except Exception as e:
    print(f"⚠️ Ошибка при проверке кода: {e}")
    import traceback
    traceback.print_exc()


Проверка версии кода в ноутбуке:
✅ dividends2 на основе ni2
✅ retained earnings с dividends2
✅ cff2 с dividends2

✅ Исправление применено! Код актуален.


### 🔍 Проверка версии кода модели

Проверка того, что используется правильная версия кода с исправлением `dividends2`.

In [16]:
# Проверка версии кода модели
import inspect
from engine.model3.core import ThreeStatementModel

# Получаем исходный код метода _solve_year
source = inspect.getsource(ThreeStatementModel._solve_year)

# Проверяем наличие исправления dividends2
has_dividends2 = '_plan_equity_flows(ni2, rev)' in source and 'dividends2, equity_issuance, equity_buyback = self._plan_equity_flows(ni2, rev)' in source
has_correct_re = 'self.re[y] = self.re[prev_year] + ni2 - dividends2' in source

print("="*80)
print("ПРОВЕРКА ВЕРСИИ КОДА МОДЕЛИ")
print("="*80)
print(f"Исправление dividends2 найдено: {'✅' if has_dividends2 else '❌'}")
print(f"Правильное обновление retained_earnings: {'✅' if has_correct_re else '❌'}")

if has_dividends2 and has_correct_re:
    print("\n✅ Используется правильная версия кода!")
    print("Если verification показывает расхождение, перезапустите ядро ноутбука (Kernel → Restart & Clear Output)")
else:
    print("\n❌ Используется старая версия кода!")
    print("Перезапустите ядро ноутбука (Kernel → Restart & Clear Output) для загрузки новой версии кода")

# Показываем ключевые строки кода
if has_dividends2:
    lines = source.split('\n')
    for i, line in enumerate(lines):
        if '_plan_equity_flows(ni2, rev)' in line:
            print(f"\nКлючевая строка кода (строка {i}):")
            print(f"  {line.strip()}")
            break


ПРОВЕРКА ВЕРСИИ КОДА МОДЕЛИ
Исправление dividends2 найдено: ✅
Правильное обновление retained_earnings: ✅

✅ Используется правильная версия кода!
Если verification показывает расхождение, перезапустите ядро ноутбука (Kernel → Restart & Clear Output)

Ключевая строка кода (строка 343):
  dividends2, equity_issuance, equity_buyback = self._plan_equity_flows(ni2, rev)


In [17]:

from engine.model.core import build_model

print(f"Запуск построения модели для {COMPANY}...")
print(f"ROOT: {ROOT}")
print(f"Company: {COMPANY}")

proj_yaml = yaml.safe_load((croot / "configs" / "project.yaml").read_text(encoding='utf-8'))
periods_config = proj_yaml.get("model", {}).get("standard", {}).get("periods", {})

print(f"\n📋 Входные параметры модели:")
print(f"  История:")
print(f"    - Начальный год: {periods_config.get('history_start_year', 'auto')}")
print(f"    - Конечный год: {periods_config.get('history_end_year', 'auto')}")
print(f"  Прогноз:")
print(f"    - Начальный год: {periods_config.get('forecast_start_year', 'auto')}")
print(f"    - Конечный год: {periods_config.get('forecast_end_year', 'auto')}")
print(f"    - Количество лет: {periods_config.get('forecast_years', 'auto')}")

try:
    MODEL_VERSION = build_model(ROOT, COMPANY)
    print(f"📦 Версия модели: {MODEL_VERSION}")
    print(" ✅ Модель построена успешно!")
    mart = _open_data_mart()
    if mart is not None:
        versions = mart.get_existing_versions()
        if versions:
            is_forecast_dm = mart.get_model_results(MODEL_VERSION, 'IS', canonical=True)
            if not is_forecast_dm.empty:
                forecast_years_actual = [int(c) for c in is_forecast_dm.columns if str(c).isdigit()]
                forecast_years_actual = sorted(forecast_years_actual)
                if forecast_years_actual:
                    print(f"\n📊 Фактические годы прогноза: {forecast_years_actual}")
                    print(f"   Первый год: {forecast_years_actual[0]}")
                    print(f"   Последний год: {forecast_years_actual[-1]}")
                    print(f"   Всего лет: {len(forecast_years_actual)}")
        mart.close()
except Exception as e:
    print(f"\n❌ Ошибка построения модели: {e}")
    import traceback
    traceback.print_exc()
    raise



[LP] PuLP not available, will use fallback heuristic
Запуск построения модели для us_steel...
ROOT: /home
Company: us_steel

📋 Входные параметры модели:
  История:
    - Начальный год: 2010
    - Конечный год: 2024
  Прогноз:
    - Начальный год: 2025
    - Конечный год: 2029
    - Количество лет: 5
[DEBUG] _process_revenue_forecast вызван для us_steel
[DEBUG] Конфигурация periods из yaml: {'history_start_year': 2010, 'history_end_year': 2024, 'forecast_start_year': 2025, 'forecast_end_year': 2029, 'forecast_years': 5}
[DEBUG] Параметры периодов из yaml: history_start_year=2010, history_end_year=2024, forecast_start_year=2025, forecast_end_year=2029
[DEBUG] История revenue в БД: 19 лет, годы: [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[DEBUG] История revenue после фильтрации (history_start_year=2010, history_end_year=2024): 15 лет, годы: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022,

Revenue прогноз пуст, пытаемся использовать историю для генерации fallback прогноза
dep_rate_ppenet слишком высокая (100.0%), ограничиваем до 15%
Канонизация долга: debt_financial_base + lease_liab_base + st_debt_nonmodeled_base (3,936,000,000) не совпадает с total_debt_raw (4,208,000,000). Разница: 272,000,000.
DebtOptimizer[2025]: opening_cash=1,367,000,000, cfo=2,370,671,330, cfi=-1,130,193,860, prefin=2,607,477,470, non_refi_mandatory=0, total_refi_fees=0, cash_before=2,607,477,470, min_cash=500,000,000, required_draw=0
DebtOptimizer[2025]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2025]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2026]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2026]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2027]:

[Drivers Snapshot] rev_tail {2025: 13549526394.50507, 2026: 11738469662.109306, 2027: 10169482386.051601, 2028: 8810209079.812056, 2029: 7632618955.755861}
[Drivers Snapshot] attrs {'dividend_payout_ratio': 0.0, 'share_issuance': None, 'share_buyback': None, 'roe': None}
[Historic RE snapshot] 7219000000.0
[ThreeStatementModel _solve_year file] /home/engine/model3/core.py
[finance_lease_dep overrides] {2024: 56000000.0, 2020: 13000000.0, 2021: 19000000.0, 2022: 27000000.0, 2023: 40000000.0}
[RE Snapshot] {2025: 6014746716.837828, 2026: 4836817688.855825, 2027: 3657676586.2211294, 2028: 2491694076.3215876, 2029: 1347453991.6460006} Dividend payout: 0.0
[BS DataFrame total_equity] [{'metric': 'total_equity', '2025': 10235834857.837828, '2026': 9057905829.855825, '2027': 7878764727.221129, '2028': 6712782217.321588, '2029': 5568542132.646001}]
[BS DataFrame retained_earnings] [{'metric': 'retained_earnings', '2025': 6014746716.837828, '2026': 4836817688.855825, '2027': 3657676586.2211294,

### 🔍 Детальная отладка: что с чем сравнивается и откуда данные

Пошаговая проверка процесса verification для понимания источника данных.

In [18]:
# Детальная отладка verificationfrom engine.model3.io import load_inputsfrom engine.model3.core import ThreeStatementModelfrom engine.model.preprocess import ModelPreprocessorfrom engine.database.data_mart import get_data_martfrom engine.model3.verification import ModelVerifier, almost_equalimport yamlprint("="*80)print("ШАГ 1: Загрузка и запуск модели")print("="*80)mart = get_data_mart(ROOT, COMPANY)config = yaml.safe_load((croot / "configs" / "project.yaml").read_text(encoding='utf-8'))ModelPreprocessor(mart, config).run()mart.close()hist_state, _, drivers, _ = load_inputs(root=ROOT, company=COMPANY)forecast_years = sorted(y for y in drivers.revenue if y > hist_state.year)model = ThreeStatementModel(    hist_state,    forecast_years,    drivers,    zero_growth_test=False,    root=ROOT,    company=COMPANY)model.run()print(f"✅ Модель запущена, годы в BS: {sorted(model.BS.keys())}")print("\n" + "="*80)print("ШАГ 2: Что находится в model.BS для прогнозных годов?")print("="*80)for year in forecast_years:    bs = model.BS[year]    print(f"\n{year}:")    print(f"  total_assets: {bs.get('total_assets', 'ОТСУТСТВУЕТ')}")    print(f"  total_liabilities: {bs.get('total_liabilities', 'ОТСУТСТВУЕТ')}")    print(f"  total_equity: {bs.get('total_equity', 'ОТСУТСТВУЕТ')}")    print(f"  retained_earnings: {bs.get('retained_earnings', 'ОТСУТСТВУЕТ')}")print("\n" + "="*80)print("ШАГ 3: Как verification читает данные из model.BS?")print("="*80)print("Код verification._verify_balance_identity():")print("  for year, bs_row in self.model.BS.items():")print("      total_assets = bs_row.get('total_assets', 0.0)")print("      total_liabilities = bs_row.get('total_liabilities', 0.0)")print("      total_equity = bs_row.get('total_equity', 0.0)")print("      total_le = total_liabilities + total_equity")print("      self._add_result('balance_identity', year, total_assets, total_le)")for year in forecast_years:    bs_row = model.BS[year]    total_assets = bs_row.get("total_assets", 0.0)    total_liabilities = bs_row.get("total_liabilities", 0.0)    total_equity = bs_row.get("total_equity", 0.0)    total_le = total_liabilities + total_equity        print(f"\n{year}:")    print(f"  bs_row.get('total_assets'): {total_assets:,.0f}")    print(f"  bs_row.get('total_liabilities'): {total_liabilities:,.0f}")    print(f"  bs_row.get('total_equity'): {total_equity:,.0f}")    print(f"  total_le = {total_liabilities:,.0f} + {total_equity:,.0f} = {total_le:,.0f}")    print(f"  diff = total_assets - total_le = {total_assets - total_le:,.0f}")print("\n" + "="*80)print("ШАГ 4: Откуда берутся total_assets, total_liabilities, total_equity?")print("="*80)print("В _solve_year() вычисляются:")print("  total_assets = total_current_assets + total_non_current_assets")print("  total_liabilities = total_current_liabilities + total_non_current_liabilities")print("  total_equity = share_capital + treasury_stock + apic + retained_earnings + aoci + nci")for year in forecast_years:    bs = model.BS[year]        # Проверяем компоненты    total_current_assets = bs.get('total_current_assets', 0.0)    total_non_current_assets = bs.get('total_non_current_assets', 0.0)    total_assets_calc = total_current_assets + total_non_current_assets        total_current_liabilities = bs.get('total_current_liabilities', 0.0)    total_non_current_liabilities = bs.get('total_non_current_liabilities', 0.0)    total_liabilities_calc = total_current_liabilities + total_non_current_liabilities        share_capital = bs.get('share_capital', 0.0)    treasury_stock = bs.get('treasury_stock', 0.0)    apic = bs.get('apic', 0.0)    retained_earnings = bs.get('retained_earnings', 0.0)    aoci = bs.get('aoci', 0.0)    nci = bs.get('nci', 0.0)    total_equity_calc = share_capital + treasury_stock + apic + retained_earnings + aoci + nci        print(f"\n{year}:")    print(f"  Assets: {total_current_assets:,.0f} + {total_non_current_assets:,.0f} = {total_assets_calc:,.0f} (из BS: {bs.get('total_assets', 0.0):,.0f})")    print(f"  Liabilities: {total_current_liabilities:,.0f} + {total_non_current_liabilities:,.0f} = {total_liabilities_calc:,.0f} (из BS: {bs.get('total_liabilities', 0.0):,.0f})")    print(f"  Equity: {share_capital:,.0f} + {treasury_stock:,.0f} + {apic:,.0f} + {retained_earnings:,.0f} + {aoci:,.0f} + {nci:,.0f} = {total_equity_calc:,.0f} (из BS: {bs.get('total_equity', 0.0):,.0f})")        if abs(total_assets_calc - bs.get('total_assets', 0.0)) > 1e-3:        print(f"    ⚠️ РАСХОЖДЕНИЕ total_assets!")    if abs(total_liabilities_calc - bs.get('total_liabilities', 0.0)) > 1e-3:        print(f"    ⚠️ РАСХОЖДЕНИЕ total_liabilities!")    if abs(total_equity_calc - bs.get('total_equity', 0.0)) > 1e-3:        print(f"    ⚠️ РАСХОЖДЕНИЕ total_equity!")print("\n" + "="*80)print("ШАГ 5: Что verification передает в _add_result?")print("="*80)print("_add_result('balance_identity', year, expected=total_assets, actual=total_le)")print("diff = actual - expected = total_le - total_assets")eps = 1e-4for year in forecast_years:    bs_row = model.BS[year]    total_assets = bs_row.get("total_assets", 0.0)    total_liabilities = bs_row.get("total_liabilities", 0.0)    total_equity = bs_row.get("total_equity", 0.0)    total_le = total_liabilities + total_equity        expected = total_assets    actual = total_le    diff = actual - expected    passed = almost_equal(expected, actual, eps)    threshold = eps * max(1.0, abs(expected), abs(actual))        print(f"\n{year}:")    print(f"  expected (total_assets): {expected:,.0f}")    print(f"  actual (total_le): {actual:,.0f}")    print(f"  diff = {diff:,.0f}")    print(f"  threshold = {eps} * max(1.0, {abs(expected):,.0f}, {abs(actual):,.0f}) = {threshold:,.0f}")    print(f"  passed = abs(diff) <= threshold: {passed} (abs(diff)={abs(diff):,.0f})")print("\n" + "="*80)print("ШАГ 6: Реальные результаты verification")print("="*80)verifier = ModelVerifier(model)results = verifier.verify_all()balance_checks = {r.year: r for r in results if r.check_name == "balance_identity" and r.year in forecast_years}for year in forecast_years:    r = balance_checks.get(year)    if r:        print(f"\n{year}:")        print(f"  expected: {r.expected:,.0f}")        print(f"  actual: {r.actual:,.0f}")        print(f"  difference: {r.difference:,.0f}")        print(f"  passed: {r.passed}")                # Сравниваем с прямыми данными        bs = model.BS[year]        total_assets_direct = bs.get('total_assets', 0.0)        total_le_direct = bs.get('total_liabilities', 0.0) + bs.get('total_equity', 0.0)                if abs(r.expected - total_assets_direct) > 1e-3:            print(f"  ⚠️ РАСХОЖДЕНИЕ expected! verification={r.expected:,.0f}, model.BS={total_assets_direct:,.0f}")        if abs(r.actual - total_le_direct) > 1e-3:            print(f"  ⚠️ РАСХОЖДЕНИЕ actual! verification={r.actual:,.0f}, model.BS={total_le_direct:,.0f}")        if abs(r.expected - total_assets_direct) < 1e-3 and abs(r.actual - total_le_direct) < 1e-3:            print(f"  ✅ Verification использует правильные данные из model.BS")

ШАГ 1: Загрузка и запуск модели
[DEBUG] _process_revenue_forecast вызван для us_steel
[DEBUG] Конфигурация periods из yaml: {'history_start_year': 2010, 'history_end_year': 2024, 'forecast_start_year': 2025, 'forecast_end_year': 2029, 'forecast_years': 5}
[DEBUG] Параметры периодов из yaml: history_start_year=2010, history_end_year=2024, forecast_start_year=2025, forecast_end_year=2029
[DEBUG] История revenue в БД: 19 лет, годы: [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[DEBUG] История revenue после фильтрации (history_start_year=2010, history_end_year=2024): 15 лет, годы: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[DEBUG] Проверка макрофакторов: доступно 8 факторов из 8
[DEBUG] Проверка существующего прогноза: пуст=False, колонки=['metric', 2025, 2026, 2027, 2028, 2029]
[DEBUG] Макрофакторы доступны, пересчитываем прогноз через регрессию
[DEBUG] Загружено макрофакто

Revenue прогноз пуст, пытаемся использовать историю для генерации fallback прогноза
dep_rate_ppenet слишком высокая (100.0%), ограничиваем до 15%
Канонизация долга: debt_financial_base + lease_liab_base + st_debt_nonmodeled_base (3,936,000,000) не совпадает с total_debt_raw (4,208,000,000). Разница: 272,000,000.
DebtOptimizer[2025]: opening_cash=1,367,000,000, cfo=2,370,671,330, cfi=-1,130,193,860, prefin=2,607,477,470, non_refi_mandatory=0, total_refi_fees=0, cash_before=2,607,477,470, min_cash=500,000,000, required_draw=0
DebtOptimizer[2025]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2025]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2026]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2026]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2027]:

✅ Модель запущена, годы в BS: [2024, 2025, 2026, 2027, 2028, 2029]

ШАГ 2: Что находится в model.BS для прогнозных годов?

2025:
  total_assets: 18607693846.47926
  total_liabilities: 8315858988.64143
  total_equity: 10291834857.837828
  retained_earnings: 6070746716.837828

2026:
  total_assets: 17665745895.21205
  total_liabilities: 8504240065.356229
  total_equity: 9161505829.855825
  retained_earnings: 4940417688.855825

2027:
  total_assets: 16691781133.423306
  total_liabilities: 8668956406.202177
  total_equity: 8022824727.221129
  retained_earnings: 3801736586.2211294

2028:
  total_assets: 15748031852.116188
  total_liabilities: 8856798634.7946
  total_equity: 6891233217.321588
  retained_earnings: 2670145076.3215876

2029:
  total_assets: 14843432991.132439
  total_liabilities: 9067207508.486439
  total_equity: 5776225482.646001
  retained_earnings: 1555137341.6460006

ШАГ 3: Как verification читает данные из model.BS?
Код verification._verify_balance_identity():
  for year, 

In [19]:
# Детальная диагностика синхронизации данныхfrom engine.model3.io import load_inputsfrom engine.model3.core import ThreeStatementModelfrom engine.model.preprocess import ModelPreprocessorfrom engine.database.data_mart import get_data_martfrom engine.model3.verification import ModelVerifierimport yamlprint("="*80)print("ШАГ 1: Препроцессинг")print("="*80)mart = get_data_mart(ROOT, COMPANY)config = yaml.safe_load((croot / "configs" / "project.yaml").read_text(encoding='utf-8'))ModelPreprocessor(mart, config).run()mart.close()print("✅ Препроцессинг завершен")print("\n" + "="*80)print("ШАГ 2: Загрузка входных данных")print("="*80)hist_state, _, drivers, _ = load_inputs(root=ROOT, company=COMPANY)forecast_years = sorted(y for y in drivers.revenue if y > hist_state.year)print(f"Исторический год: {hist_state.year}")print(f"Прогнозные годы: {forecast_years}")print("\n" + "="*80)print("ШАГ 3: Создание и запуск модели")print("="*80)model = ThreeStatementModel(    hist_state,    forecast_years,    drivers,    zero_growth_test=False,    root=ROOT,    company=COMPANY)model.run()print("✅ Модель запущена")print("\n" + "="*80)print("ШАГ 4: Проверка баланса в model.BS (перед verification)")print("="*80)for year in forecast_years:    bs = model.BS[year]    total_assets = bs.get('total_assets', 0.0)    total_equity = bs.get('total_equity', 0.0)    total_liabilities = bs.get('total_liabilities', 0.0)    total_le = total_liabilities + total_equity    diff = total_assets - total_le    status = "✅" if abs(diff) < 1e-3 else "❌"    print(f"{status} {year}: Assets={total_assets:,.0f}, Equity={total_equity:,.0f}, Liab={total_liabilities:,.0f}, diff={diff:,.0f}")print("\n" + "="*80)print("ШАГ 5: Verification (использует model.BS напрямую)")print("="*80)verifier = ModelVerifier(model)results = verifier.verify_all()balance_checks = {r.year: r for r in results if r.check_name == "balance_identity" and r.year in forecast_years}for year in forecast_years:    r = balance_checks.get(year)    if r:        status = "✅" if r.passed else "❌"        print(f"{status} {year}: expected={r.expected:,.0f}, actual={r.actual:,.0f}, diff={r.difference:,.0f}")        # Проверяем синхронизацию        bs = model.BS[year]        total_assets = bs.get('total_assets', 0.0)        total_le = bs.get('total_liabilities', 0.0) + bs.get('total_equity', 0.0)        if abs(r.expected - total_assets) > 1e-6 or abs(r.actual - total_le) > 1e-6:            print(f"  ⚠️ РАСХОЖДЕНИЕ! Verification использует другие данные!")            print(f"    Verification expected={r.expected:,.0f}, model.BS Assets={total_assets:,.0f}")            print(f"    Verification actual={r.actual:,.0f}, model.BS Liab+Equity={total_le:,.0f}")print("\n" + "="*80)print("ШАГ 6: Создание DataFrame (to_statement_frames)")print("="*80)is_df, bs_df, cf_df = model.to_statement_frames()for year in forecast_years:    year_str = str(year)    if year_str in bs_df.columns:        assets_value = float(bs_df[bs_df['metric'] == 'total_assets'][year_str].values[0])        equity_value = float(bs_df[bs_df['metric'] == 'total_equity'][year_str].values[0])        liab_value = float(bs_df[bs_df['metric'] == 'total_liabilities'][year_str].values[0])        total_le = liab_value + equity_value        diff = assets_value - total_le                # Сравниваем с model.BS        bs = model.BS[year]        assets_model = bs.get('total_assets', 0.0)        equity_model = bs.get('total_equity', 0.0)                status = "✅" if abs(diff) < 1e-3 else "❌"        print(f"{status} {year}: Assets={assets_value:,.0f}, Equity={equity_value:,.0f}, diff={diff:,.0f}")                if abs(assets_value - assets_model) > 1e-6:            print(f"  ⚠️ РАСХОЖДЕНИЕ Assets! DataFrame={assets_value:,.0f}, model.BS={assets_model:,.0f}")        if abs(equity_value - equity_model) > 1e-6:            print(f"  ⚠️ РАСХОЖДЕНИЕ Equity! DataFrame={equity_value:,.0f}, model.BS={equity_model:,.0f}")print("\n" + "="*80)print("ШАГ 7: Сравнение verification и DataFrame")print("="*80)for year in forecast_years:    r = balance_checks.get(year)    if r:        year_str = str(year)        if year_str in bs_df.columns:            assets_df = float(bs_df[bs_df['metric'] == 'total_assets'][year_str].values[0])            equity_df = float(bs_df[bs_df['metric'] == 'total_equity'][year_str].values[0])            liab_df = float(bs_df[bs_df['metric'] == 'total_liabilities'][year_str].values[0])            total_le_df = liab_df + equity_df                        if abs(r.expected - assets_df) < 1e-6 and abs(r.actual - total_le_df) < 1e-6:                print(f"✅ {year}: Verification и DataFrame синхронизированы")            else:                print(f"❌ {year}: РАСХОЖДЕНИЕ между verification и DataFrame!")                print(f"   Verification expected={r.expected:,.0f}, DataFrame Assets={assets_df:,.0f}")                print(f"   Verification actual={r.actual:,.0f}, DataFrame Liab+Equity={total_le_df:,.0f}")print("\n" + "="*80)print("ШАГ 8: Сохранение в БД и проверка")print("="*80)from engine.model3.io import save_outputsoutputs = {"IS": is_df, "BS": bs_df, "CF": cf_df}save_outputs(root=ROOT, company=COMPANY, outputs=outputs, version=MODEL_VERSION)print(f"✅ Данные сохранены в БД с версией {MODEL_VERSION}")mart = get_data_mart(ROOT, COMPANY)bs_from_db = mart.get_model_results(MODEL_VERSION, 'BS', canonical=True)mart.close()print("\nПроверка данных из БД:")for year in forecast_years:    year_str = str(year)    if year_str in bs_from_db.columns:        assets_db = float(bs_from_db[bs_from_db['metric'] == 'total_assets'][year_str].values[0])        equity_db = float(bs_from_db[bs_from_db['metric'] == 'total_equity'][year_str].values[0])        liab_db = float(bs_from_db[bs_from_db['metric'] == 'total_liabilities'][year_str].values[0])        total_le_db = liab_db + equity_db        diff_db = assets_db - total_le_db                # Сравниваем с DataFrame        assets_df = float(bs_df[bs_df['metric'] == 'total_assets'][year_str].values[0])        equity_df = float(bs_df[bs_df['metric'] == 'total_equity'][year_str].values[0])                status = "✅" if abs(diff_db) < 1e-3 else "❌"        print(f"{status} {year}: Assets={assets_db:,.0f}, Equity={equity_db:,.0f}, diff={diff_db:,.0f}")                if abs(assets_db - assets_df) > 1e-6:            print(f"  ⚠️ РАСХОЖДЕНИЕ Assets! БД={assets_db:,.0f}, DataFrame={assets_df:,.0f}")        if abs(equity_db - equity_df) > 1e-6:            print(f"  ⚠️ РАСХОЖДЕНИЕ Equity! БД={equity_db:,.0f}, DataFrame={equity_df:,.0f}")

ШАГ 1: Препроцессинг
[DEBUG] _process_revenue_forecast вызван для us_steel
[DEBUG] Конфигурация periods из yaml: {'history_start_year': 2010, 'history_end_year': 2024, 'forecast_start_year': 2025, 'forecast_end_year': 2029, 'forecast_years': 5}
[DEBUG] Параметры периодов из yaml: history_start_year=2010, history_end_year=2024, forecast_start_year=2025, forecast_end_year=2029
[DEBUG] История revenue в БД: 19 лет, годы: [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[DEBUG] История revenue после фильтрации (history_start_year=2010, history_end_year=2024): 15 лет, годы: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[DEBUG] Проверка макрофакторов: доступно 8 факторов из 8
[DEBUG] Проверка существующего прогноза: пуст=False, колонки=['metric', nan, 2025.0, 2026.0, 2027.0, 2028.0, 2029.0]
[DEBUG] Макрофакторы доступны, пересчитываем прогноз через регрессию
[DEBUG] Загружено макроф

Revenue прогноз пуст, пытаемся использовать историю для генерации fallback прогноза
dep_rate_ppenet слишком высокая (100.0%), ограничиваем до 15%
Канонизация долга: debt_financial_base + lease_liab_base + st_debt_nonmodeled_base (3,936,000,000) не совпадает с total_debt_raw (4,208,000,000). Разница: 272,000,000.
DebtOptimizer[2025]: opening_cash=1,367,000,000, cfo=2,370,671,330, cfi=-1,130,193,860, prefin=2,607,477,470, non_refi_mandatory=0, total_refi_fees=0, cash_before=2,607,477,470, min_cash=500,000,000, required_draw=0
DebtOptimizer[2025]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2025]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2026]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2026]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2027]:

Исторический год: 2024
Прогнозные годы: [2025, 2026, 2027, 2028, 2029]

ШАГ 3: Создание и запуск модели
✅ Модель запущена

ШАГ 4: Проверка баланса в model.BS (перед verification)
✅ 2025: Assets=18,607,693,846, Equity=10,291,834,858, Liab=8,315,858,989, diff=0
✅ 2026: Assets=17,665,745,895, Equity=9,161,505,830, Liab=8,504,240,065, diff=-0
✅ 2027: Assets=16,691,781,133, Equity=8,022,824,727, Liab=8,668,956,406, diff=0
✅ 2028: Assets=15,748,031,852, Equity=6,891,233,217, Liab=8,856,798,635, diff=0
✅ 2029: Assets=14,843,432,991, Equity=5,776,225,483, Liab=9,067,207,508, diff=0

ШАГ 5: Verification (использует model.BS напрямую)
✅ 2025: expected=18,607,693,846, actual=18,607,693,846, diff=-0
✅ 2026: expected=17,665,745,895, actual=17,665,745,895, diff=0
✅ 2027: expected=16,691,781,133, actual=16,691,781,133, diff=0
✅ 2028: expected=15,748,031,852, actual=15,748,031,852, diff=0
✅ 2029: expected=14,843,432,991, actual=14,843,432,991, diff=0

ШАГ 6: Создание DataFrame (to_statement_frames)
✅ 

In [20]:
# Детальная диагностика синхронизации данныхfrom engine.model3.io import load_inputsfrom engine.model3.core import ThreeStatementModelfrom engine.model.preprocess import ModelPreprocessorfrom engine.database.data_mart import get_data_martfrom engine.model3.verification import ModelVerifierimport yamlprint("="*80)print("ШАГ 1: Препроцессинг")print("="*80)mart = get_data_mart(ROOT, COMPANY)config = yaml.safe_load((croot / "configs" / "project.yaml").read_text(encoding='utf-8'))ModelPreprocessor(mart, config).run()mart.close()print("✅ Препроцессинг завершен")print("\n" + "="*80)print("ШАГ 2: Загрузка входных данных")print("="*80)hist_state, _, drivers, _ = load_inputs(root=ROOT, company=COMPANY)forecast_years = sorted(y for y in drivers.revenue if y > hist_state.year)print(f"Исторический год: {hist_state.year}")print(f"Прогнозные годы: {forecast_years}")print("\n" + "="*80)print("ШАГ 3: Создание и запуск модели")print("="*80)model = ThreeStatementModel(hist_state, forecast_years, drivers, zero_growth_test=False, root=ROOT, company=COMPANY)model.run()print("✅ Модель запущена")print("\n" + "="*80)print("ШАГ 4: Проверка баланса в model.BS (перед verification)")print("="*80)for year in forecast_years:        bs = model.BS[year]        total_assets = bs.get('total_assets', 0.0)        total_equity = bs.get('total_equity', 0.0)        total_liabilities = bs.get('total_liabilities', 0.0)        total_le = total_liabilities + total_equity        diff = total_assets - total_le        status = "✅" if abs(diff) < 1e-3 else "❌"        print(f"{status} {year}: Assets={total_assets:,.0f}, Equity={total_equity:,.0f}, Liab={total_liabilities:,.0f}, diff={diff:,.0f}")print("\n" + "="*80)print("ШАГ 5: Verification (использует model.BS напрямую)")print("="*80)verifier = ModelVerifier(model)results = verifier.verify_all()balance_checks = {r.year: r for r in results if r.check_name == "balance_identity" and r.year in forecast_years}for year in forecast_years:        r = balance_checks.get(year)        if r:                status = "✅" if r.passed else "❌"                print(f"{status} {year}: expected={r.expected:,.0f}, actual={r.actual:,.0f}, diff={r.difference:,.0f}")        # Проверяем синхронизациюbs = model.BS[year]total_assets = bs.get('total_assets', 0.0)total_le = bs.get('total_liabilities', 0.0) + bs.get('total_equity', 0.0)if abs(r.expected - total_assets) > 1e-6 or abs(r.actual - total_le) > 1e-6:        print(f"  ⚠️ РАСХОЖДЕНИЕ! Verification использует другие данные!")        print(f"    Verification expected={r.expected:,.0f}, model.BS Assets={total_assets:,.0f}")        print(f"    Verification actual={r.actual:,.0f}, model.BS Liab+Equity={total_le:,.0f}")print("\n" + "="*80)print("ШАГ 6: Создание DataFrame (to_statement_frames)")print("="*80)is_df, bs_df, cf_df = model.to_statement_frames()for year in forecast_years:        year_str = str(year)        if year_str in bs_df.columns:                assets_value = float(bs_df[bs_df['metric'] == 'total_assets'][year_str].values[0])                equity_value = float(bs_df[bs_df['metric'] == 'total_equity'][year_str].values[0])                liab_value = float(bs_df[bs_df['metric'] == 'total_liabilities'][year_str].values[0])                total_le = liab_value + equity_value                iff = assets_value - total_le                # Сравниваем с model.BSbs = model.BS[year]assets_model = bs.get('total_assets', 0.0)equity_model = bs.get('total_equity', 0.0)        status = "✅" if abs(diff) < 1e-3 else "❌"print(f"{status} {year}: Assets={assets_value:,.0f}, Equity={equity_value:,.0f}, diff={diff:,.0f}")        if abs(assets_value - assets_model) > 1e-6:        print(f"  ⚠️ РАСХОЖДЕНИЕ Assets! DataFrame={assets_value:,.0f}, model.BS={assets_model:,.0f}")if abs(equity_value - equity_model) > 1e-6:        print(f"  ⚠️ РАСХОЖДЕНИЕ Equity! DataFrame={equity_value:,.0f}, model.BS={equity_model:,.0f}")print("\n" + "="*80)print("ШАГ 7: Сравнение verification и DataFrame")print("="*80)for year in forecast_years:        r = balance_checks.get(year)        if r:                year_str = str(year)                if year_str in bs_df.columns:                        assets_df = float(bs_df[bs_df['metric'] == 'total_assets'][year_str].values[0])                        equity_df = float(bs_df[bs_df['metric'] == 'total_equity'][year_str].values[0])                        liab_df = float(bs_df[bs_df['metric'] == 'total_liabilities'][year_str].values[0])                        total_le_df = liab_df + equity_df            if abs(r.expected - assets_df) < 1e-6 and abs(r.actual - total_le_df) < 1e-6:        print(f"✅ {year}: Verification и DataFrame синхронизированы")else:        print(f"❌ {year}: РАСХОЖДЕНИЕ между verification и DataFrame!")        print(f"   Verification expected={r.expected:,.0f}, DataFrame Assets={assets_df:,.0f}")        print(f"   Verification actual={r.actual:,.0f}, DataFrame Liab+Equity={total_le_df:,.0f}")print("\n" + "="*80)print("ШАГ 8: Сохранение в БД и проверка")print("="*80)from engine.model3.io import save_outputsoutputs = {"IS": is_df, "BS": bs_df, "CF": cf_df}save_outputs(root=ROOT, company=COMPANY, outputs=outputs, version=MODEL_VERSION)print(f"✅ Данные сохранены в БД с версией {MODEL_VERSION}")mart = get_data_mart(ROOT, COMPANY)bs_from_db = mart.get_model_results(MODEL_VERSION, 'BS', canonical=True)mart.close()print("\nПроверка данных из БД:")for year in forecast_years:        year_str = str(year)        if year_str in bs_from_db.columns:                ssets_db = float(bs_from_db[bs_from_db['metric'] == 'total_assets'][year_str].values[0])                equity_db = float(bs_from_db[bs_from_db['metric'] == 'total_equity'][year_str].values[0])                liab_db = float(bs_from_db[bs_from_db['metric'] == 'total_liabilities'][year_str].values[0])                total_le_db = liab_db + equity_db                diff_db = assets_db - total_le_db                # Сравниваем с DataFrameassets_df = float(bs_df[bs_df['metric'] == 'total_assets'][year_str].values[0])equity_df = float(bs_df[bs_df['metric'] == 'total_equity'][year_str].values[0])        status = "✅" if abs(diff_db) < 1e-3 else "❌"print(f"{status} {year}: Assets={assets_db:,.0f}, Equity={equity_db:,.0f}, diff={diff_db:,.0f}")        if abs(assets_db - assets_df) > 1e-6:        print(f"  ⚠️ РАСХОЖДЕНИЕ Assets! БД={assets_db:,.0f}, DataFrame={assets_df:,.0f}")if abs(equity_db - equity_df) > 1e-6:        print(f"  ⚠️ РАСХОЖДЕНИЕ Equity! БД={equity_db:,.0f}, DataFrame={equity_df:,.0f}")

ШАГ 1: Препроцессинг
[DEBUG] _process_revenue_forecast вызван для us_steel
[DEBUG] Конфигурация periods из yaml: {'history_start_year': 2010, 'history_end_year': 2024, 'forecast_start_year': 2025, 'forecast_end_year': 2029, 'forecast_years': 5}
[DEBUG] Параметры периодов из yaml: history_start_year=2010, history_end_year=2024, forecast_start_year=2025, forecast_end_year=2029
[DEBUG] История revenue в БД: 19 лет, годы: [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[DEBUG] История revenue после фильтрации (history_start_year=2010, history_end_year=2024): 15 лет, годы: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[DEBUG] Проверка макрофакторов: доступно 8 факторов из 8
[DEBUG] Проверка существующего прогноза: пуст=False, колонки=['metric', 2025, 2026, 2027, 2028, 2029]
[DEBUG] Макрофакторы доступны, пересчитываем прогноз через регрессию
[DEBUG] Загружено макрофакторов из БД: 

Revenue прогноз пуст, пытаемся использовать историю для генерации fallback прогноза
dep_rate_ppenet слишком высокая (100.0%), ограничиваем до 15%
Канонизация долга: debt_financial_base + lease_liab_base + st_debt_nonmodeled_base (3,936,000,000) не совпадает с total_debt_raw (4,208,000,000). Разница: 272,000,000.
DebtOptimizer[2025]: opening_cash=1,367,000,000, cfo=2,370,671,330, cfi=-1,130,193,860, prefin=2,607,477,470, non_refi_mandatory=0, total_refi_fees=0, cash_before=2,607,477,470, min_cash=500,000,000, required_draw=0
DebtOptimizer[2025]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2025]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2026]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2026]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2027]:

Исторический год: 2024
Прогнозные годы: [2025, 2026, 2027, 2028, 2029]

ШАГ 3: Создание и запуск модели
✅ Модель запущена

ШАГ 4: Проверка баланса в model.BS (перед verification)
✅ 2025: Assets=18,607,693,846, Equity=10,291,834,858, Liab=8,315,858,989, diff=0
✅ 2026: Assets=17,665,745,895, Equity=9,161,505,830, Liab=8,504,240,065, diff=-0
✅ 2027: Assets=16,691,781,133, Equity=8,022,824,727, Liab=8,668,956,406, diff=0
✅ 2028: Assets=15,748,031,852, Equity=6,891,233,217, Liab=8,856,798,635, diff=0
✅ 2029: Assets=14,843,432,991, Equity=5,776,225,483, Liab=9,067,207,508, diff=0

ШАГ 5: Verification (использует model.BS напрямую)
✅ 2025: expected=18,607,693,846, actual=18,607,693,846, diff=-0
✅ 2026: expected=17,665,745,895, actual=17,665,745,895, diff=0
✅ 2027: expected=16,691,781,133, actual=16,691,781,133, diff=0
✅ 2028: expected=15,748,031,852, actual=15,748,031,852, diff=0
✅ 2029: expected=14,843,432,991, actual=14,843,432,991, diff=0

ШАГ 6: Создание DataFrame (to_statement_frames)
✅ 

In [21]:
# Проверка баланса из БД после build_model
from engine.database.data_mart import get_data_mart

mart = _open_data_mart()
if mart is not None:
    # Загружаем баланс из БД
    bs_df = mart.get_model_results(MODEL_VERSION, 'BS', canonical=True)
    
    print("\n" + "="*80)
    print("ПРОВЕРКА БАЛАНСА ИЗ БД:")
    print("="*80)
    
    # Проверяем баланс для прогнозных лет
    forecast_years = [2025, 2026, 2027, 2028, 2029]
    
    for year in forecast_years:
        year_str = str(year)
        if year_str in bs_df.columns:
            total_assets = bs_df.loc[bs_df['metric'] == 'total_assets', year_str].values
            total_liabilities = bs_df.loc[bs_df['metric'] == 'total_liabilities', year_str].values
            total_equity = bs_df.loc[bs_df['metric'] == 'total_equity', year_str].values
            
            if len(total_assets) > 0 and len(total_liabilities) > 0 and len(total_equity) > 0:
                ta = float(total_assets[0]) if not pd.isna(total_assets[0]) else 0.0
                tl = float(total_liabilities[0]) if not pd.isna(total_liabilities[0]) else 0.0
                te = float(total_equity[0]) if not pd.isna(total_equity[0]) else 0.0
                
                total_le = tl + te
                diff = ta - total_le
                
                print(f"\n{year}:")
                print(f"  Assets (БД): {ta:,.0f}")
                print(f"  Liabilities (БД): {tl:,.0f}")
                print(f"  Equity (БД): {te:,.0f}")
                print(f"  Liab+Equity (БД): {total_le:,.0f}")
                print(f"  Diff (БД): {diff:,.0f}")
            else:
                print(f"\n{year}: Данные не найдены в БД")
    
    mart.close()
else:
    print("⚠️ Не удалось открыть Data Mart")



ПРОВЕРКА БАЛАНСА ИЗ БД:

2025:
  Assets (БД): 18,607,693,846
  Liabilities (БД): 8,315,858,989
  Equity (БД): 10,291,834,858
  Liab+Equity (БД): 18,607,693,846
  Diff (БД): 0

2026:
  Assets (БД): 17,665,745,895
  Liabilities (БД): 8,504,240,065
  Equity (БД): 9,161,505,830
  Liab+Equity (БД): 17,665,745,895
  Diff (БД): -0

2027:
  Assets (БД): 16,691,781,133
  Liabilities (БД): 8,668,956,406
  Equity (БД): 8,022,824,727
  Liab+Equity (БД): 16,691,781,133
  Diff (БД): 0

2028:
  Assets (БД): 15,748,031,852
  Liabilities (БД): 8,856,798,635
  Equity (БД): 6,891,233,217
  Liab+Equity (БД): 15,748,031,852
  Diff (БД): 0

2029:
  Assets (БД): 14,843,432,991
  Liabilities (БД): 9,067,207,508
  Equity (БД): 5,776,225,483
  Liab+Equity (БД): 14,843,432,991
  Diff (БД): 0


In [22]:
# Детальная диагностика баланса после build_modelfrom engine.model3.io import load_inputsfrom engine.model3.core import ThreeStatementModelfrom engine.model.preprocess import ModelPreprocessorfrom engine.database.data_mart import get_data_martimport yaml# Загружаем модель напрямую (как в build_model)mart = get_data_mart(ROOT, COMPANY)config = yaml.safe_load((croot / "configs" / "project.yaml").read_text(encoding='utf-8'))ModelPreprocessor(mart, config).run()mart.close()hist_state, _, drivers, _ = load_inputs(root=ROOT, company=COMPANY)forecast_years = sorted(y for y in drivers.revenue if y > hist_state.year)model = ThreeStatementModel(    hist_state,    forecast_years,    drivers,    zero_growth_test=False,    root=ROOT,    company=COMPANY)model.run()# Проверяем баланс напрямую из объекта моделиprint("\n" + "="*80)print("ПРЯМАЯ ПРОВЕРКА БАЛАНСА ИЗ ОБЪЕКТА МОДЕЛИ:")print("="*80)for year in forecast_years:    bs = model.BS[year]    total_assets = bs.get('total_assets', 0.0)    total_liabilities = bs.get('total_liabilities', 0.0)    total_equity = bs.get('total_equity', 0.0)    total_liab_equity = bs.get('total_liab_equity', 0.0)        calc_le = total_liabilities + total_equity    diff_assets_vs_le = total_assets - calc_le    diff_le_vs_total = calc_le - total_liab_equity        print(f"\n{year}:")    print(f"  Assets: {total_assets:,.0f}")    print(f"  Liabilities: {total_liabilities:,.0f}")    print(f"  Equity: {total_equity:,.0f}")    print(f"  Liab+Equity (calc): {calc_le:,.0f}")    print(f"  total_liab_equity (from BS): {total_liab_equity:,.0f}")    print(f"  Diff (Assets vs Liab+Equity): {diff_assets_vs_le:,.0f}")    print(f"  Diff (Liab+Equity vs total_liab_equity): {diff_le_vs_total:,.0f}")# Запускаем verificationprint("\n" + "="*80)print("РЕЗУЛЬТАТЫ VERIFICATION:")print("="*80)from engine.model3.verification import ModelVerifierverifier = ModelVerifier(model)results = verifier.verify_all()balance_checks = [r for r in results if r.check_name == "balance_identity"]for r in balance_checks:    if not r.passed:        print(f"\n{r.year}: diff={r.difference:,.2f}")        print(f"  Expected (Assets): {r.expected:,.0f}")        print(f"  Actual (Liab+Equity): {r.actual:,.0f}")        # Проверяем, что показывает прямая проверка        bs = model.BS[r.year]        total_assets = bs.get('total_assets', 0.0)        total_le = bs.get('total_liabilities', 0.0) + bs.get('total_equity', 0.0)        print(f"  Прямая проверка из BS: Assets={total_assets:,.0f}, Liab+Equity={total_le:,.0f}, diff={total_assets - total_le:,.0f}")

[DEBUG] _process_revenue_forecast вызван для us_steel
[DEBUG] Конфигурация periods из yaml: {'history_start_year': 2010, 'history_end_year': 2024, 'forecast_start_year': 2025, 'forecast_end_year': 2029, 'forecast_years': 5}
[DEBUG] Параметры периодов из yaml: history_start_year=2010, history_end_year=2024, forecast_start_year=2025, forecast_end_year=2029
[DEBUG] История revenue в БД: 19 лет, годы: [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[DEBUG] История revenue после фильтрации (history_start_year=2010, history_end_year=2024): 15 лет, годы: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[DEBUG] Проверка макрофакторов: доступно 8 факторов из 8
[DEBUG] Проверка существующего прогноза: пуст=False, колонки=['metric', 2025, 2026, 2027, 2028, 2029]
[DEBUG] Макрофакторы доступны, пересчитываем прогноз через регрессию
[DEBUG] Загружено макрофакторов из БД: история=8 факторов, п

Revenue прогноз пуст, пытаемся использовать историю для генерации fallback прогноза
dep_rate_ppenet слишком высокая (100.0%), ограничиваем до 15%
Канонизация долга: debt_financial_base + lease_liab_base + st_debt_nonmodeled_base (3,936,000,000) не совпадает с total_debt_raw (4,208,000,000). Разница: 272,000,000.
DebtOptimizer[2025]: opening_cash=1,367,000,000, cfo=2,370,671,330, cfi=-1,130,193,860, prefin=2,607,477,470, non_refi_mandatory=0, total_refi_fees=0, cash_before=2,607,477,470, min_cash=500,000,000, required_draw=0
DebtOptimizer[2025]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2025]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2026]: ФИНАЛЬНАЯ ПРОВЕРКА: cash_final=500,000,000 < min_cash_with_buffer=1,000,000,000, требуется дополнительный займ=500,000,000
DebtOptimizer[2026]: Взят дополнительный займ через RC: 500,000,000, осталось: 0
DebtOptimizer[2027]:


ПРЯМАЯ ПРОВЕРКА БАЛАНСА ИЗ ОБЪЕКТА МОДЕЛИ:

2025:
  Assets: 18,607,693,846
  Liabilities: 8,315,858,989
  Equity: 10,291,834,858
  Liab+Equity (calc): 18,607,693,846
  total_liab_equity (from BS): 18,607,693,846
  Diff (Assets vs Liab+Equity): 0
  Diff (Liab+Equity vs total_liab_equity): 0

2026:
  Assets: 17,665,745,895
  Liabilities: 8,504,240,065
  Equity: 9,161,505,830
  Liab+Equity (calc): 17,665,745,895
  total_liab_equity (from BS): 17,665,745,895
  Diff (Assets vs Liab+Equity): -0
  Diff (Liab+Equity vs total_liab_equity): 0

2027:
  Assets: 16,691,781,133
  Liabilities: 8,668,956,406
  Equity: 8,022,824,727
  Liab+Equity (calc): 16,691,781,133
  total_liab_equity (from BS): 16,691,781,133
  Diff (Assets vs Liab+Equity): 0
  Diff (Liab+Equity vs total_liab_equity): 0

2028:
  Assets: 15,748,031,852
  Liabilities: 8,856,798,635
  Equity: 6,891,233,217
  Liab+Equity (calc): 15,748,031,852
  total_liab_equity (from BS): 15,748,031,852
  Diff (Assets vs Liab+Equity): 0
  Diff (Lia

### 📋 Сравнение данных из официальной отчетности <a id="section-comparison"></a>

**Цель**: Сравнить данные из официальной отчетности 2024 с данными, которые использует модель

**Шаги**:
1. Вставьте данные из официального Balance Sheet 2024 (в таком же порядке, как в отчетности)
2. Система загрузит данные из файла истории (`bs_history_us_steel.csv`)
3. Система загрузит данные из БД после препроцессинга (canonical формат)
4. Сравнит все три источника и покажет расхождения
5. Определит, какие статьи были объединены через маппинг

**Формат ввода**: Используйте структуру из официальной отчетности

In [ ]:
# === ШАГ 1: Данные из официального Balance Sheet 2024 ===
# Данные из официальной отчетности US Steel (в том же порядке, как в отчете)

official_reporting_2024 = {
    # === ASSETS ===
    # Current Assets:
    'Cash and cash equivalents': 1367.0,  # миллионов USD
    'Receivables, less allowance': 1236.0,
    'Receivables from related parties': 162.0,
    'Inventories': 2168.0,
    'Other current assets': 299.0,
    'Total current assets': 5232.0,
    
    # Non-Current Assets:
    'Long-term restricted cash': 35.0,
    'Investments and long-term receivables': 757.0,
    'Operating lease assets': 72.0,
    'Property, plant and equipment, net': 11973.0,
    'Intangibles, net': 416.0,
    'Deferred income tax benefits': 0.0,  # указано как "—" (ноль или отсутствует)
    'Goodwill': 920.0,
    'Other noncurrent assets': 830.0,
    'Total assets': 20235.0,
    
    # === LIABILITIES ===
    # Current Liabilities:
    'Accounts payable and other accrued liabilities': 2601.0,
    'Accounts payable to related parties': 146.0,
    'Payroll and benefits payable': 295.0,
    'Accrued taxes': 131.0,
    'Accrued interest': 70.0,
    'Current operating lease liabilities': 35.0,
    'Short-term debt and current maturities of long-term debt': 95.0,
    'Total current liabilities': 3373.0,
    
    # Non-Current Liabilities:
    'Noncurrent operating lease liabilities': 44.0,
    'Long-term debt, less unamortized discount': 4078.0,
    'Employee benefits': 117.0,
    'Deferred income tax liabilities': 657.0,
    'Deferred credits and other noncurrent liabilities': 526.0,
    'Total liabilities': 8795.0,
    
    # === STOCKHOLDERS' EQUITY ===
    'Common stock issued': 288.0,
    'Treasury stock, at cost': -1446.0,  # отрицательное значение
    'Additional paid-in capital': 5307.0,
    'Retained earnings': 7219.0,
    'Accumulated other comprehensive (loss) income': -21.0,
    'Total United States Steel Corporation stockholders equity': 11347.0,
    'Noncontrolling interests': 93.0,
    'Total liabilities and stockholders equity': 20235.0
}

# Конвертируем из миллионов в доллары
official_reporting_2024 = {k: v * 1_000_000 for k, v in official_reporting_2024.items()}

print(f"✅ Загружено {len(official_reporting_2024)} статей из официального Balance Sheet 2024")
print(f"\n📊 Структура данных (первые 15 статей):")
for i, (item, value) in enumerate(list(official_reporting_2024.items())[:15], 1):
    print(f"  {i:2d}. {item}: ${value:,.0f}")
print(f"\n... и еще {len(official_reporting_2024) - 15} статей")

In [ ]:
# === ШАГ 2: Загрузка данных из файла истории ===
# Проверка переменных (если еще не определены)
if 'ROOT' not in globals():
    from pathlib import Path
    current_dir = Path.cwd()
    if (current_dir / 'engine').exists():
        ROOT = current_dir
    elif (current_dir.parent / 'engine').exists():
        ROOT = current_dir.parent
    else:
        ROOT = current_dir.parent.parent

if 'COMPANY' not in globals():
    COMPANY = 'us_steel'
    current_dir = Path.cwd()
    if 'companies' in current_dir.parts:
        companies_idx = current_dir.parts.index('companies')
        if companies_idx + 1 < len(current_dir.parts):
            COMPANY = current_dir.parts[companies_idx + 1]

if 'croot' not in globals():
    croot = ROOT / 'companies' / COMPANY

import pandas as pd
from pathlib import Path

# Загружаем файл истории BS
hist_file = croot / "history" / "bs_history_us_steel.csv"
hist_df = pd.read_csv(hist_file)

# Получаем данные 2024 из файла истории
if '2024' in hist_df.columns:
    hist_2024_raw = {}
    for _, row in hist_df.iterrows():
        metric = row['metric']
        value = pd.to_numeric(row['2024'], errors='coerce')
        if pd.notna(value):
            hist_2024_raw[metric] = float(value)
    
    print(f"✅ Загружено {len(hist_2024_raw)} статей из файла истории")
    print("\nПервые 10 статей из файла истории:")
    for item, value in list(hist_2024_raw.items())[:10]:
        print(f"  {item}: ${value:,.0f}")
else:
    print("❌ Колонка '2024' не найдена в файле истории")
    hist_2024_raw = {}

In [ ]:
# === ШАГ 3: Загрузка данных из БД (canonical формат после препроцессинга) ===
# Проверка переменных (если еще не определены)
if 'ROOT' not in globals():
    from pathlib import Path
    current_dir = Path.cwd()
    if (current_dir / 'engine').exists():
        ROOT = current_dir
    elif (current_dir.parent / 'engine').exists():
        ROOT = current_dir.parent
    else:
        ROOT = current_dir.parent.parent

if 'COMPANY' not in globals():
    COMPANY = 'us_steel'
    current_dir = Path.cwd()
    if 'companies' in current_dir.parts:
        companies_idx = current_dir.parts.index('companies')
        if companies_idx + 1 < len(current_dir.parts):
            COMPANY = current_dir.parts[companies_idx + 1]

# Проверка функций загрузки (если еще не определены)
if 'get_data_mart' not in globals():
    from engine.database.data_mart import get_data_mart

if '_open_data_mart' not in globals():
    def _open_data_mart():
        try:
            return get_data_mart(ROOT, COMPANY)
        except Exception as exc:
            print(f"⚠️ Не удалось подключиться к витрине данных: {exc}")
            return None

mart = get_data_mart(ROOT, COMPANY)
canonical_bs = mart.get_history_balance_sheet(canonical=True)
mart.close()

# Получаем данные 2024 из canonical формата
canonical_2024 = {}
if not canonical_bs.empty:
    if 'year' in canonical_bs.columns and 'metric' in canonical_bs.columns:
        bs_2024 = canonical_bs[canonical_bs['year'] == 2024]
        for _, row in bs_2024.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['value'], errors='coerce')
            if pd.notna(value):
                canonical_2024[metric] = float(value)
    elif 'metric' in canonical_bs.columns:
        # Wide format
        if '2024' in canonical_bs.columns:
            for _, row in canonical_bs.iterrows():
                metric = row['metric']
                value = pd.to_numeric(row['2024'], errors='coerce')
                if pd.notna(value):
                    canonical_2024[metric] = float(value)

print(f"✅ Загружено {len(canonical_2024)} статей из БД (canonical)")
print("\nСтатьи из canonical формата:")
for item, value in sorted(canonical_2024.items())[:15]:
    print(f"  {item}: ${value:,.0f}")

In [ ]:
# === ШАГ 4: Загрузка маппинга статей ===
# Проверяем конфигурацию маппинга из excel_loader.yaml
import yaml
from pathlib import Path

if 'croot' not in globals():
    if 'ROOT' not in globals():
        from pathlib import Path
        current_dir = Path.cwd()
        if (current_dir / 'engine').exists():
            ROOT = current_dir
        else:
            ROOT = current_dir.parent.parent
    if 'COMPANY' not in globals():
        COMPANY = 'us_steel'
    croot = ROOT / 'companies' / COMPANY

excel_loader_path = croot / "configs" / "excel_loader.yaml"
mapping_config = {}
if excel_loader_path.exists():
    with open(excel_loader_path, 'r', encoding='utf-8') as f:
        mapping_config = yaml.safe_load(f)
    
    bs_mapping = mapping_config.get('BS', {})
    if bs_mapping:
        print("✅ Найдена конфигурация маппинга для BS")
        required_metrics = bs_mapping.get('required_metrics', {})
        print(f"\nТребуемые метрики (с алиасами):")
        for metric, config in required_metrics.items():
            aliases = config.get('aliases', [])
            if aliases:
                print(f"  {metric}: {', '.join(aliases)}")
            else:
                print(f"  {metric}: (без алиасов)")
    else:
        print("⚠️ Конфигурация BS не найдена в excel_loader.yaml")
else:
    print("⚠️ Файл excel_loader.yaml не найден")

In [ ]:
# === ШАГ 5: Детальное сравнение всех источников с учетом маппинга ===
# Проверка переменных (если еще не определены)
if 'ROOT' not in globals():
    from pathlib import Path
    current_dir = Path.cwd()
    if (current_dir / 'engine').exists():
        ROOT = current_dir
    elif (current_dir.parent / 'engine').exists():
        ROOT = current_dir.parent
    else:
        ROOT = current_dir.parent.parent

if 'COMPANY' not in globals():
    COMPANY = 'us_steel'

# Проверка функций загрузки (если еще не определены)
if 'get_data_mart' not in globals():
    from engine.database.data_mart import get_data_mart

if '_open_data_mart' not in globals():
    def _open_data_mart():
        try:
            return get_data_mart(ROOT, COMPANY)
        except Exception as exc:
            print(f"⚠️ Не удалось подключиться к витрине данных: {exc}")
            return None

from IPython.display import display, Markdown
from difflib import SequenceMatcher
import re
import pandas as pd

def similarity(a, b):
    """Вычисляет схожесть двух строк (0-1)"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def normalize_name(name):
    """Нормализует название для сравнения"""
    normalized = re.sub(r'[^a-z0-9\s]', '', name.lower())
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

def find_best_match(official_name, candidate_list, threshold=0.7):
    """Находит лучшее совпадение для названия из официального отчета"""
    best_match = None
    best_score = 0.0
    
    normalized_official = normalize_name(official_name)
    
    for candidate in candidate_list:
        normalized_candidate = normalize_name(candidate)
        score = similarity(normalized_official, normalized_candidate)
        
        if normalized_official == normalized_candidate:
            return candidate, 1.0
        
        if score > best_score:
            best_score = score
            best_match = candidate
    
    if best_score >= threshold:
        return best_match, best_score
    return None, best_score

display(Markdown("### 🔍 Детальное сравнение данных 2024"))

if not official_reporting_2024:
    print("⚠️ Данные из официальной отчетности не загружены.")
    print("Пожалуйста, заполните словарь official_reporting_2024 в ячейке выше.")
else:
    print(f"✅ Данные из официального отчета: {len(official_reporting_2024)} статей")
    print(f"✅ Данные из файла истории: {len(hist_2024_raw)} статей")
    print(f"✅ Данные из canonical (БД): {len(canonical_2024)} статей")
    
    # Создаем сравнительную таблицу
    comparison_data = []
    
    # Проходим по всем статьям из официального отчета
    for official_item, official_val in official_reporting_2024.items():
        # Ищем совпадение в файле истории
        hist_match, hist_score = find_best_match(official_item, hist_2024_raw.keys(), threshold=0.6)
        
        # Ищем совпадение в canonical формате
        canonical_match, canonical_score = find_best_match(official_item, canonical_2024.keys(), threshold=0.6)
        
        hist_val = hist_2024_raw.get(hist_match) if hist_match else None
        canonical_val = canonical_2024.get(canonical_match) if canonical_match else None
        
        # Проверяем расхождения
        issues = []
        
        if hist_match:
            if hist_val is not None:
                diff = abs(official_val - hist_val)
                diff_pct = (diff / abs(official_val) * 100) if official_val != 0 else 0
                if diff > 1000:  # Разница больше $1K
                    issues.append(f"Офиц. vs История: ${diff:,.0f} ({diff_pct:.1f}%)")
        
        if canonical_match:
            if canonical_val is not None:
                diff = abs(official_val - canonical_val)
                diff_pct = (diff / abs(official_val) * 100) if official_val != 0 else 0
                if diff > 1000:
                    issues.append(f"Офиц. vs Canonical: ${diff:,.0f} ({diff_pct:.1f}%)")
        
        if hist_match and canonical_match and hist_val is not None and canonical_val is not None:
            diff = abs(hist_val - canonical_val)
            if diff > 1000:
                issues.append(f"История vs Canonical: ${diff:,.0f}")
        
        comparison_data.append({
            'Официальный отчет': official_item,
            'Значение (офиц.)': f"${official_val:,.0f}",
            'Совпадение (история)': hist_match if hist_match else "-",
            'Значение (история)': f"${hist_val:,.0f}" if hist_val is not None else "-",
            'Совпадение (canonical)': canonical_match if canonical_match else "-",
            'Значение (canonical)': f"${canonical_val:,.0f}" if canonical_val is not None else "-",
            'Проблемы': '; '.join(issues) if issues else '✅'
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Показываем сначала проблемные статьи
    problematic = comparison_df[comparison_df['Проблемы'] != '✅']
    if not problematic.empty:
        display(Markdown("#### ❌ Статьи с расхождениями:"))
        display(problematic[['Официальный отчет', 'Значение (офиц.)', 'Совпадение (история)', 
                             'Значение (история)', 'Совпадение (canonical)', 
                             'Значение (canonical)', 'Проблемы']])
    
    # Статьи, которые не были найдены
    not_found = comparison_df[(comparison_df['Совпадение (история)'] == '-') & 
                               (comparison_df['Совпадение (canonical)'] == '-')]
    if not not_found.empty:
        display(Markdown("#### ⚠️ Статьи из официального отчета, не найденные в модели:"))
        display(not_found[['Официальный отчет', 'Значение (офиц.)']])
    
    # Показываем полную таблицу
    display(Markdown("#### 📊 Полная таблица сравнения:"))
    display(comparison_df)
    
    # Сводка
    print(f"\n📋 Сводка сравнения:")
    print(f"  Всего статей в официальном отчете: {len(official_reporting_2024)}")
    print(f"  Найдено совпадений в истории: {len([x for x in comparison_df['Совпадение (история)'] if x != '-'])}")
    print(f"  Найдено совпадений в canonical: {len([x for x in comparison_df['Совпадение (canonical)'] if x != '-'])}")
    print(f"  Статей с расхождениями: {len(problematic)}")
    print(f"  Статей не найденных в модели: {len(not_found)}")
    print(f"  Статей без проблем: {len(comparison_df) - len(problematic)}")

### 🔧 Исправление canonical формата на основе официальной отчетности

**Цель**: Создать правильный canonical формат для 2024 года на основе официальной отчетности

**Процесс**:
1. Сравниваем официальные данные с данными из файла истории и canonical формата
2. Определяем маппинг статей (как статьи из официального отчета соответствуют canonical метрикам)
3. Создаём исправленный canonical формат на основе официальных данных
4. Сохраняем исправленные данные в БД или файл


In [ ]:
# === Создание правильного canonical формата на основе официальной отчетности ===
import pandas as pd
import yaml
from pathlib import Path
from engine.database.data_mart import get_data_mart

display(Markdown("### 🔧 Создание правильного canonical формата для 2024 года"))

# Проверяем, что данные загружены
if 'official_reporting_2024' not in globals() or not official_reporting_2024:
    print("❌ Данные из официальной отчетности не загружены. Запустите ячейку ШАГ 1.")
    raise ValueError("Данные из официальной отчетности отсутствуют")

if 'hist_2024_raw' not in globals() or not hist_2024_raw:
    print("❌ Данные из файла истории не загружены. Запустите ячейку ШАГ 2.")
    raise ValueError("Данные из файла истории отсутствуют")

if 'canonical_2024' not in globals() or not canonical_2024:
    print("❌ Данные из canonical формата не загружены. Запустите ячейку ШАГ 3.")
    raise ValueError("Данные из canonical формата отсутствуют")

print(f"✅ Все источники данных загружены:")
print(f"  Официальный отчет: {len(official_reporting_2024)} статей")
print(f"  Файл истории: {len(hist_2024_raw)} статей")
print(f"  Canonical (БД): {len(canonical_2024)} статей")

# Загружаем маппинг из excel_loader.yaml
excel_loader_path = croot / "configs" / "excel_loader.yaml"
mapping_config = {}
if excel_loader_path.exists():
    with open(excel_loader_path, 'r', encoding='utf-8') as f:
        mapping_config = yaml.safe_load(f)

bs_mapping = mapping_config.get('BS', {})
required_metrics = bs_mapping.get('required_metrics', {}) if bs_mapping else {}

# Создаём маппинг: официальное название -> canonical метрика
# Используем функцию find_best_match из предыдущей ячейки
def find_best_match_canonical(official_name, canonical_keys, threshold=0.7):
    """Находит лучшее совпадение для canonical метрики"""
    from difflib import SequenceMatcher
    import re
    
    def normalize_name(name):
        normalized = re.sub(r'[^a-z0-9\s]', '', name.lower())
        normalized = re.sub(r'\s+', ' ', normalized).strip()
        return normalized
    
    def similarity(a, b):
        return SequenceMatcher(None, a.lower(), b.lower()).ratio()
    
    best_match = None
    best_score = 0.0
    
    normalized_official = normalize_name(official_name)
    
    for candidate in canonical_keys:
        normalized_candidate = normalize_name(candidate)
        score = similarity(normalized_official, normalized_candidate)
        
        if normalized_official == normalized_candidate:
            return candidate, 1.0
        
        if score > best_score:
            best_score = score
            best_match = candidate
    
    if best_score >= threshold:
        return best_match, best_score
    return None, best_score

# Создаём маппинг официальных статей на canonical метрики
official_to_canonical = {}

# Сначала пытаемся использовать маппинг из конфига
aliases_map = {}
for metric, config in required_metrics.items():
    aliases = config.get('aliases', [])
    for alias in aliases:
        aliases_map[alias.lower()] = metric
    # Также добавляем сам metric
    aliases_map[metric.lower()] = metric

print("\n📋 Создание маппинга официальных статей на canonical метрики:")

mapping_results = []

for official_item, official_val in official_reporting_2024.items():
    # Пробуем найти по алиасам
    canonical_metric = None
    match_method = None
    
    # Проверяем точное совпадение в canonical
    if official_item.lower() in aliases_map:
        canonical_metric = aliases_map[official_item.lower()]
        match_method = 'aliases'
    else:
        # Ищем по схожести
        canonical_metric, score = find_best_match_canonical(official_item, canonical_2024.keys(), threshold=0.7)
        if canonical_metric:
            match_method = f'similarity ({score:.1%})'
        else:
            # Пробуем найти в файле истории
            hist_match, hist_score = find_best_match_canonical(official_item, hist_2024_raw.keys(), threshold=0.7)
            if hist_match and hist_match in canonical_2024:
                canonical_metric = hist_match
                match_method = f'history -> canonical ({hist_score:.1%})'
    
    mapping_results.append({
        'Официальное название': official_item,
        'Значение (офиц.)': f"${official_val:,.0f}",
        'Canonical метрика': canonical_metric if canonical_metric else 'НЕ НАЙДЕНО',
        'Метод сопоставления': match_method if canonical_metric else 'не найдено',
        'Значение (canonical)': f"${canonical_2024.get(canonical_metric, 0):,.0f}" if canonical_metric else "-",
        'Расхождение': f"${abs(official_val - canonical_2024.get(canonical_metric, 0)):,.0f}" if canonical_metric and canonical_metric in canonical_2024 else "-"
    })

mapping_df = pd.DataFrame(mapping_results)
display(mapping_df)

# Создаём правильный canonical формат на основе официальных данных
print("\n" + "="*80)
print("Создание исправленного canonical формата...")

corrected_canonical = {}

for official_item, official_val in official_reporting_2024.items():
    # Находим соответствующую canonical метрику
    canonical_metric = None
    
    # По алиасам
    if official_item.lower() in aliases_map:
        canonical_metric = aliases_map[official_item.lower()]
    else:
        # По схожести
        canonical_metric, _ = find_best_match_canonical(official_item, canonical_2024.keys(), threshold=0.7)
        if not canonical_metric:
            # Через историю
            hist_match, _ = find_best_match_canonical(official_item, hist_2024_raw.keys(), threshold=0.7)
            if hist_match and hist_match in canonical_2024:
                canonical_metric = hist_match
    
    if canonical_metric:
        corrected_canonical[canonical_metric] = official_val
        print(f"✅ {canonical_metric}: ${official_val:,.0f} (из '{official_item}')")
    else:
        print(f"⚠️ Статья '{official_item}' не сопоставлена с canonical метрикой")

# Добавляем статьи, которые есть в canonical, но не в официальном отчете (оставляем как есть)
for canonical_metric, canonical_val in canonical_2024.items():
    if canonical_metric not in corrected_canonical:
        corrected_canonical[canonical_metric] = canonical_val
        print(f"ℹ️ {canonical_metric}: ${canonical_val:,.0f} (сохранено из canonical)")

print(f"\n✅ Создан исправленный canonical формат: {len(corrected_canonical)} статей")

# Показываем расхождения
print("\n" + "="*80)
print("Расхождения между оригинальным и исправленным canonical:")

differences = []
for metric in sorted(set(canonical_2024.keys()) | set(corrected_canonical.keys())):
    orig_val = canonical_2024.get(metric, 0)
    corr_val = corrected_canonical.get(metric, 0)
    
    if abs(orig_val - corr_val) > 1000:  # Разница больше $1K
        diff = corr_val - orig_val
        diff_pct = (diff / abs(orig_val) * 100) if orig_val != 0 else 0
        differences.append({
            'Метрика': metric,
            'Оригинал (canonical)': f"${orig_val:,.0f}",
            'Исправлено (офиц.)': f"${corr_val:,.0f}",
            'Разница': f"${diff:,.0f}",
            'Разница %': f"{diff_pct:+.1f}%"
        })

if differences:
    diff_df = pd.DataFrame(differences)
    display(Markdown("#### ❌ Статьи с расхождениями:"))
    display(diff_df)
else:
    print("✅ Расхождений не найдено")

# Сохраняем исправленный canonical в переменную для дальнейшего использования
corrected_canonical_2024 = corrected_canonical
print("\n✅ Исправленный canonical формат сохранен в переменную 'corrected_canonical_2024'")

In [ ]:
# === Сохранение исправленного canonical формата ===

if 'corrected_canonical_2024' not in globals():
    print("❌ Исправленный canonical формат не создан. Запустите предыдущую ячейку.")
else:
    display(Markdown("### 💾 Сохранение исправленных данных"))
    
    print("Выберите способ сохранения:")
    print("1. Обновить файл истории (bs_history_us_steel.csv)")
    print("2. Сохранить в БД через Data Mart")
    print("3. Создать новый файл для проверки")
    
    # Опция 1: Обновить файл истории
    print("\n" + "="*80)
    print("Опция 1: Обновление файла истории")
    print("="*80)
    
    hist_file = croot / "history" / "bs_history_us_steel.csv"
    
    if hist_file.exists():
        hist_df = pd.read_csv(hist_file)
        
        # Обновляем значения для 2024 года
        if '2024' in hist_df.columns:
            print("\nОбновление значений для 2024 года:")
            
            updates_count = 0
            for _, row in hist_df.iterrows():
                metric = row['metric']
                if metric in corrected_canonical_2024:
                    old_val = pd.to_numeric(row['2024'], errors='coerce')
                    new_val = corrected_canonical_2024[metric]
                    
                    if pd.isna(old_val) or abs(old_val - new_val) > 1000:
                        hist_df.at[_, '2024'] = new_val
                        print(f"  {metric}: ${old_val:,.0f} -> ${new_val:,.0f}")
                        updates_count += 1
            
            if updates_count > 0:
                # Создаём резервную копию
                backup_file = hist_file.parent / f"bs_history_us_steel_backup_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
                hist_df.to_csv(backup_file, index=False)
                print(f"\n✅ Создана резервная копия: {backup_file.name}")
                
                # Сохраняем обновлённый файл
                hist_df.to_csv(hist_file, index=False)
                print(f"✅ Файл истории обновлён: {updates_count} статей изменено")
            else:
                print("ℹ️ Изменений не требуется")
        else:
            print(f"⚠️ Колонка '2024' не найдена в файле истории")
    else:
        print(f"⚠️ Файл истории не найден: {hist_file}")
    
    # Опция 2: Сохранить в БД
    print("\n" + "="*80)
    print("Опция 2: Сохранение в БД")
    print("="*80)
    
    try:
        # Создаём DataFrame для сохранения в БД
        save_data = []
        for metric, value in corrected_canonical_2024.items():
            save_data.append({
                'metric': metric,
                'year': 2024,
                'value': value
            })
        
        save_df = pd.DataFrame(save_data)
        
        # Сохраняем в БД через Data Mart
        mart = get_data_mart(ROOT, COMPANY)
        
        # Используем метод для сохранения истории (если есть)
        # mart.save_history_balance_sheet(save_df)  # Раскомментируйте, если метод существует
        
        mart.close()
        
        print(f"✅ Данные подготовлены для сохранения в БД: {len(save_data)} записей")
        print("⚠️ Для сохранения в БД может потребоваться запустить препроцессор")
        
    except Exception as exc:
        print(f"⚠️ Ошибка при сохранении в БД: {exc}")
    
    # Опция 3: Создать файл для проверки
    print("\n" + "="*80)
    print("Опция 3: Создание файла для проверки")
    print("="*80)
    
    check_file = croot / "outputs" / "corrected_bs_2024.csv"
    check_file.parent.mkdir(parents=True, exist_ok=True)
    
    check_data = []
    for metric, value in sorted(corrected_canonical_2024.items()):
        check_data.append({
            'metric': metric,
            'year': 2024,
            'value': value
        })
    
    check_df = pd.DataFrame(check_data)
    check_df.to_csv(check_file, index=False)
    
    print(f"✅ Файл для проверки создан: {check_file.relative_to(ROOT)}")
    print(f"   Всего статей: {len(check_data)}")
    
    display(check_df.head(20))
    
    print("\n✅ Все операции завершены")

## 4️⃣ Проверка выходных файлов <a id="section-4"></a>

In [23]:
display(Markdown("### 📊 Сводная таблица 3-Statement (последние 5 лет истории + прогноз)"))

# Маппинг метрик для поиска в прогнозе (короткие названия -> канонические)
METRIC_MAPPING_IS = {
    'depreciation_owned': ['dep_pp&e', 'dep_ppe', 'depreciation_owned'],
    'depreciation_rou': ['dep_rou', 'depreciation_rou'],
    'total_da': ['dep_total', 'total_da', 'depreciation_and_amortization'],
    'interest_expense': ['interest_expense_debt', 'interest_expense'],
    'tax': ['tax_expense', 'current_tax', 'tax'],
    'amortization': ['amortization']  # Может отсутствовать в прогнозе
}

METRIC_MAPPING_BS = {
    'ar': ['accounts_receivable', 'ar'],
    'ap': ['accounts_payable', 'ap'],
    'ppe': ['ppe_net', 'ppe'],
    'equity': ['total_equity', 'equity'],
    'debt': ['long_term_debt', 'lt_debt', 'debt'],  # Для debt нужно суммировать ST + LT
    'lease_liability': ['current_lease_liability', 'long_term_lease_liability', 'lease_liability'],
    'st_debt': ['short_term_debt', 'st_debt'],
    'dta': ['deferred_tax_assets', 'dta'],
    'dtl': ['deferred_tax_liabilities', 'dtl']
}

METRIC_MAPPING_CF = {
    'depreciation': ['depreciation', 'depreciation_owned', 'dep_pp&e'],
    'amortization': ['amortization'],
    'ar_delta': ['wc_accounts_receivable_change', 'ar_delta'],
    'inventory_delta': ['wc_inventory_change', 'inventory_delta'],
    'ap_delta': ['wc_accounts_payable_change', 'ap_delta'],
    'total_capex': ['capex', 'total_capex'],
    'maint_capex': ['maint_capex'],
    'growth_capex': ['growth_capex']
}

def find_metric_in_forecast(forecast_df, metric_name, statement_type='IS'):
    """Находит метрику в прогнозе с учетом маппинга"""
    if forecast_df.empty or 'metric' not in forecast_df.columns:
        return pd.DataFrame()
    
    # Прямой поиск
    result = forecast_df[forecast_df['metric'].str.lower() == metric_name.lower()]
    if not result.empty:
        return result
    
    # Поиск через маппинг
    mapping = {}
    if statement_type == 'IS':
        mapping = METRIC_MAPPING_IS
    elif statement_type == 'BS':
        mapping = METRIC_MAPPING_BS
    elif statement_type == 'CF':
        mapping = METRIC_MAPPING_CF
    
    if metric_name in mapping:
        for alias in mapping[metric_name]:
            result = forecast_df[forecast_df['metric'].str.lower() == alias.lower()]
            if not result.empty:
                return result
    
    # Частичное совпадение
    result = forecast_df[forecast_df['metric'].str.lower().str.contains(metric_name.lower(), na=False, regex=False)]
    if not result.empty:
        return result
    
    return pd.DataFrame()

# Загружаем историю и прогноз (БД имеет приоритет над CSV)
hist_is = load_history_from_db('IS')
hist_bs = load_history_from_db('BS')
hist_cf = load_history_from_db('CF')
is_forecast = load_model_forecast_from_db('IS', canonical=True)
bs_forecast = load_model_forecast_from_db('BS', canonical=True)
cf_forecast = load_model_forecast_from_db('CF', canonical=True)

# Последние 5 лет истории (поддержка wide format с метрикой как индекс)
if 'year' in hist_is.columns:
    history_years = sorted([y for y in hist_is['year'].values if pd.notna(y)])[-5:]
else:
    # Wide format: годы это колонки
    year_cols = [c for c in hist_is.columns if str(c).isdigit() and len(str(c)) == 4]
    history_years = sorted([int(c) for c in year_cols])[-5:]
forecast_years = sorted([int(c) for c in is_forecast.columns if str(c).isdigit() and len(str(c)) == 4])
all_years_display = list(history_years) + forecast_years

print(f"\n📅 Отчетные периоды: {history_years} (история) + {forecast_years} (прогноз)")

# === INCOME STATEMENT ===
display(Markdown("#### 📈 Income Statement (Отчет о прибылях и убытках)"))

# Собираем IS данные
is_data = []
metrics_is = ['revenue', 'cogs', 'sga', 'depreciation_owned', 'depreciation_rou', 'amortization', 'total_da', 'ebitda', 'ebit', 'gain_loss_on_disposal', 'interest_income', 'interest_expense', 'lease_interest', 'tax', 'net_income']

for metric in metrics_is:
    row_data = {'Metric': metric.upper().replace('_', ' ')}
    
    # История - рассчитываем отсутствующие метрики
    for y in history_years:
        hist_row = hist_is[hist_is['year'] == y]
        if not hist_row.empty:
            if metric == 'revenue':
                val = hist_row['revenue'].iloc[0] if 'revenue' in hist_is.columns else None
            elif metric == 'cogs':
                val = hist_row['cogs'].iloc[0] if 'cogs' in hist_is.columns else None
            elif metric == 'sga':
                val = hist_row['sga'].iloc[0] if 'sga' in hist_is.columns else None
            elif metric == 'dep':
                val = hist_row['depreciation'].iloc[0] if 'depreciation' in hist_is.columns else None
            elif metric == 'ebitda':
                # EBITDA = Revenue - COGS - SG&A
                rev = hist_row['revenue'].iloc[0] if 'revenue' in hist_is.columns else None
                cgs = hist_row['cogs'].iloc[0] if 'cogs' in hist_is.columns else None
                sg = hist_row['sga'].iloc[0] if 'sga' in hist_is.columns else None
                val = rev - cgs - sg if all(pd.notna(v) for v in [rev, cgs, sg]) else None
            elif metric == 'ebit':
                # EBIT = EBITDA - Depreciation
                rev = hist_row['revenue'].iloc[0] if 'revenue' in hist_is.columns else None
                cgs = hist_row['cogs'].iloc[0] if 'cogs' in hist_is.columns else None
                sg = hist_row['sga'].iloc[0] if 'sga' in hist_is.columns else None
                dep = hist_row['depreciation'].iloc[0] if 'depreciation' in hist_is.columns else None
                val = rev - cgs - sg - dep if all(pd.notna(v) for v in [rev, cgs, sg, dep]) else None
            elif metric == 'interest_income':
                val = hist_row.get('interest_income', hist_row.get('finance_income', None)).iloc[0] if 'interest_income' in hist_is.columns or 'finance_income' in hist_is.columns else None
            elif metric == 'interest_expense':
                val = hist_row.get('interest_expense', hist_row.get('finance_expense', None)).iloc[0] if 'interest_expense' in hist_is.columns or 'finance_expense' in hist_is.columns else None
            elif metric == 'lease_interest':
                val = 0.0  # В истории лизинг не учитывается
            elif metric == 'depreciation_owned':
                # Depreciation Owned (с fallback на dep или depreciation)
                if 'depreciation_owned' in hist_is.columns:
                    val = hist_row['depreciation_owned'].iloc[0]
                elif 'dep' in hist_is.columns:
                    dep_total = hist_row['dep'].iloc[0]
                    dep_rou = hist_row.get('depreciation_rou', pd.Series([0.0])).iloc[0] if 'depreciation_rou' in hist_is.columns else 0.0
                    val = dep_total - dep_rou
                else:
                    val = None
            elif metric == 'depreciation_rou':
                val = hist_row['depreciation_rou'].iloc[0] if 'depreciation_rou' in hist_is.columns else 0.0
            elif metric == 'amortization':
                val = hist_row['amortization'].iloc[0] if 'amortization' in hist_is.columns else None
            elif metric == 'total_da':
                # Total D&A = Dep Owned + Dep ROU + Amortization
                dep_owned_val = hist_row.get('depreciation_owned', pd.Series([0.0])).iloc[0] if 'depreciation_owned' in hist_is.columns else 0.0
                dep_rou_val = hist_row.get('depreciation_rou', pd.Series([0.0])).iloc[0] if 'depreciation_rou' in hist_is.columns else 0.0
                amort_val = hist_row.get('amortization', pd.Series([0.0])).iloc[0] if 'amortization' in hist_is.columns else 0.0
                val = dep_owned_val + dep_rou_val + amort_val
            elif metric == 'gain_loss_on_disposal':
                val = hist_row['gain_loss_on_disposal'].iloc[0] if 'gain_loss_on_disposal' in hist_is.columns else 0.0
            elif metric == 'tax':
                # Используем income_tax_expense из истории
                if 'income_tax_expense' in hist_is.columns:
                    val = hist_row['income_tax_expense'].iloc[0]
                elif 'tax' in hist_is.columns:
                    val = hist_row['tax'].iloc[0]
                else:
                    val = None
            elif metric == 'net_income':
                val = hist_row['net_income'].iloc[0] if 'net_income' in hist_is.columns else None
            else:
                val = None
        else:
            val = None
        row_data[y] = f"${val:,.0f}" if pd.notna(val) else "-"
    
    # Прогноз - используем функцию поиска с маппингом
    for y in forecast_years:
        forecast_row = find_metric_in_forecast(is_forecast, metric, statement_type='IS')
        if not forecast_row.empty:
            # Пробуем str и int колонку
            val = None
            if str(y) in forecast_row.columns:
                val = forecast_row[str(y)].iloc[0]
            elif y in forecast_row.columns:
                val = forecast_row[y].iloc[0]
            # Для total_da, если не найдено, вычисляем из компонентов
            if metric == 'total_da' and (pd.isna(val) or val == 0):
                dep_owned_row = find_metric_in_forecast(is_forecast, 'depreciation_owned', 'IS')
                dep_rou_row = find_metric_in_forecast(is_forecast, 'depreciation_rou', 'IS')
                amort_row = find_metric_in_forecast(is_forecast, 'amortization', 'IS')
                dep_owned_val = dep_owned_row[str(y)].iloc[0] if not dep_owned_row.empty and str(y) in dep_owned_row.columns else 0.0
                dep_rou_val = dep_rou_row[str(y)].iloc[0] if not dep_rou_row.empty and str(y) in dep_rou_row.columns else 0.0
                amort_val = amort_row[str(y)].iloc[0] if not amort_row.empty and str(y) in amort_row.columns else 0.0
                val = dep_owned_val + dep_rou_val + amort_val
            
            # Для EBIT вычисляем: EBITDA - Depreciation
            if metric == 'ebit' and (pd.isna(val) or val == 0):
                ebitda_row = find_metric_in_forecast(is_forecast, 'ebitda', 'IS')
                dep_row = find_metric_in_forecast(is_forecast, 'depreciation_owned', 'IS')
                ebitda_val = ebitda_row[str(y)].iloc[0] if not ebitda_row.empty and str(y) in ebitda_row.columns else None
                dep_val = dep_row[str(y)].iloc[0] if not dep_row.empty and str(y) in dep_row.columns else 0.0
                if pd.notna(ebitda_val):
                    val = ebitda_val - (dep_val if pd.notna(dep_val) else 0.0)
            
            # Для tax используем tax_expense
            if metric == 'tax' and (pd.isna(val) or val == 0):
                tax_row = find_metric_in_forecast(is_forecast, 'tax_expense', 'IS')
                if not tax_row.empty and str(y) in tax_row.columns:
                    val = tax_row[str(y)].iloc[0]
            
            # Для некоторых метрик нулевые значения валидны (tax может быть 0 при убытке)
            show_zero = metric in ['tax']
            if pd.notna(val):
                if val != 0 or show_zero:
                    row_data[y] = f"${val:,.0f}"
                else:
                    row_data[y] = "-"
            else:
                row_data[y] = "-"
        else:
            row_data[y] = "-"
    is_data.append(row_data)

is_df = pd.DataFrame(is_data)
display(is_df)

# === BALANCE SHEET ===
display(Markdown("\n#### 🏦 Balance Sheet (Баланс)"))
# Полная каноническая форма Balance Sheet
metrics_bs = [
    'cash', 
    'ar',
    'inventory',
    'other_current_assets',
    'ppe',
    'intangibles',
    'rou_asset',
    'dta',
    'other_non_current_assets',
    'total_assets',
    'st_debt',
    'ap',
    'accrued_liabilities',
    'other_current_liabilities',
    'debt',
    'lease_liability',
    'dtl',
    'provisions',
    'other_non_current_liabilities',
    'equity',
    'total_liab_equity'
]
bs_data = []
for metric in metrics_bs:
    row_data = {'Metric': metric.upper().replace('_', ' ')}

    # История - маппинг имен колонок
    metric_hist_mapping = {
        'ppe': 'ppe_net',
        'ar': 'accounts_receivable',
        'ap': 'accounts_payable',
        'equity': 'total_equity',
        'debt': 'lt_debt'  # debt = lt_debt + st_debt (только LT в истории, т.к. ST отдельно)
    }
    metric_hist = metric_hist_mapping.get(metric, metric)
    
    for y in history_years:
        hist_row = hist_bs[hist_bs['year'] == y]
        if metric_hist in hist_bs.columns:
            val = hist_row[metric_hist].iloc[0] if not hist_row.empty else None
            # Для debt добавляем st_debt если это debt metric
            if metric == 'debt' and 'st_debt' in hist_bs.columns:
                st_val = hist_row['st_debt'].iloc[0] if not hist_row.empty else 0.0
                val = (val or 0.0) + (st_val if pd.notna(st_val) else 0.0)
            row_data[y] = f"${val:,.0f}" if pd.notna(val) else "-"
        else:
            row_data[y] = "-"
    
    # Прогноз - используем функцию поиска с маппингом
    for y in forecast_years:
        forecast_row = find_metric_in_forecast(bs_forecast, metric, statement_type='BS')
        if not forecast_row.empty:
            # Пробуем str и int колонку
            val = None
            if str(y) in forecast_row.columns:
                val = forecast_row[str(y)].iloc[0]
            elif y in forecast_row.columns:
                val = forecast_row[y].iloc[0]
            
            # Для debt суммируем ST + LT
            if metric == 'debt':
                st_row = find_metric_in_forecast(bs_forecast, 'st_debt', 'BS')
                lt_row = find_metric_in_forecast(bs_forecast, 'long_term_debt', 'BS')
                st_val = st_row[str(y)].iloc[0] if not st_row.empty and str(y) in st_row.columns else 0.0
                lt_val = lt_row[str(y)].iloc[0] if not lt_row.empty and str(y) in lt_row.columns else 0.0
                val = (st_val if pd.notna(st_val) else 0.0) + (lt_val if pd.notna(lt_val) else 0.0)
            
            # Для lease_liability суммируем current + long_term
            if metric == 'lease_liability':
                cur_row = find_metric_in_forecast(bs_forecast, 'current_lease_liability', 'BS')
                lt_row = find_metric_in_forecast(bs_forecast, 'long_term_lease_liability', 'BS')
                cur_val = cur_row[str(y)].iloc[0] if not cur_row.empty and str(y) in cur_row.columns else 0.0
                lt_val = lt_row[str(y)].iloc[0] if not lt_row.empty and str(y) in lt_row.columns else 0.0
                val = (cur_val if pd.notna(cur_val) else 0.0) + (lt_val if pd.notna(lt_val) else 0.0)
            
            # Для total_assets вычисляем сумму всех активов
            if metric == 'total_assets':
                cash_row = find_metric_in_forecast(bs_forecast, 'cash', 'BS')
                ar_row = find_metric_in_forecast(bs_forecast, 'accounts_receivable', 'BS')
                inv_row = find_metric_in_forecast(bs_forecast, 'inventory', 'BS')
                oca_row = find_metric_in_forecast(bs_forecast, 'other_current_assets', 'BS')
                ppe_row = find_metric_in_forecast(bs_forecast, 'ppe_net', 'BS')
                int_row = find_metric_in_forecast(bs_forecast, 'intangibles', 'BS')
                rou_row = find_metric_in_forecast(bs_forecast, 'rou_asset', 'BS')
                dta_row = find_metric_in_forecast(bs_forecast, 'deferred_tax_assets', 'BS')
                onca_row = find_metric_in_forecast(bs_forecast, 'other_non_current_assets', 'BS')
                
                val = 0.0
                for row in [cash_row, ar_row, inv_row, oca_row, ppe_row, int_row, rou_row, dta_row, onca_row]:
                    if not row.empty and str(y) in row.columns:
                        v = row[str(y)].iloc[0]
                        if pd.notna(v):
                            val += float(v)
            
            # Для total_equity вычисляем сумму компонентов equity
            if metric == 'equity' or metric == 'total_equity':
                sc_row = find_metric_in_forecast(bs_forecast, 'share_capital', 'BS')
                apic_row = find_metric_in_forecast(bs_forecast, 'apic', 'BS')
                re_row = find_metric_in_forecast(bs_forecast, 'retained_earnings', 'BS')
                aoci_row = find_metric_in_forecast(bs_forecast, 'aoci', 'BS')
                nci_row = find_metric_in_forecast(bs_forecast, 'nci', 'BS')
                
                val = 0.0
                for row in [sc_row, apic_row, re_row, aoci_row, nci_row]:
                    if not row.empty and str(y) in row.columns:
                        v = row[str(y)].iloc[0]
                        if pd.notna(v):
                            val += float(v)
            
            # Для total_liab_equity суммируем liabilities + equity
            if metric == 'total_liab_equity':
                liab_row = find_metric_in_forecast(bs_forecast, 'total_liabilities', 'BS')
                equity_row = find_metric_in_forecast(bs_forecast, 'total_equity', 'BS')
                liab_val = liab_row[str(y)].iloc[0] if not liab_row.empty and str(y) in liab_row.columns else 0.0
                equity_val = equity_row[str(y)].iloc[0] if not equity_row.empty and str(y) in equity_row.columns else 0.0
                val = (liab_val if pd.notna(liab_val) else 0.0) + (equity_val if pd.notna(equity_val) else 0.0)
            
            row_data[y] = f"${val:,.0f}" if pd.notna(val) and val != 0 else "-"
        else:
            row_data[y] = "-"
    bs_data.append(row_data)

bs_df = pd.DataFrame(bs_data)
display(bs_df)

# === CASH FLOW ===
display(Markdown("\n#### 💰 Cash Flow Statement (Отчет о движении денежных средств)"))
# Полный канонический формат CF_detail
metrics_cf = [
    'net_income', 'depreciation', 'amortization', 'interest_income', 'interest_expense', 'tax_paid',
    'wc_delta', 'ar_delta', 'inventory_delta', 'ap_delta', 
    'lease_payments_cfo', 'lease_interest_cfo', 'cfo',
    'maint_capex', 'growth_capex', 'total_capex', 'disposals', 'ppe_balance', 'intangibles_balance', 'cfi',
    'debt_borrowings', 'debt_repayments', 'rc_draws', 'rc_repayments', 
    'lease_payments_cff', 'dividends', 'cff', 'net_change'
]
cf_data = []
for metric in metrics_cf:
    row_data = {'Metric': metric.upper().replace('_', ' ')}
    
    # История
    cf_col = metric.lower()

    for y in history_years:
        hist_row = hist_cf[hist_cf['year'] == y]
        if cf_col in hist_cf.columns:
            val = hist_row[cf_col].iloc[0] if not hist_row.empty else None
            row_data[y] = f"${val:,.0f}" if pd.notna(val) else "-"
        else:
            row_data[y] = "-"
    
    # Прогноз - используем функцию поиска с маппингом
    for y in forecast_years:
        forecast_row = find_metric_in_forecast(cf_forecast, metric, statement_type='CF')
        if not forecast_row.empty:
            # Пробуем str и int колонку
            val = None
            if str(y) in forecast_row.columns:
                val = forecast_row[str(y)].iloc[0]
            elif y in forecast_row.columns:
                val = forecast_row[y].iloc[0]
            
            # Для total_capex, если не найдено, используем capex
            if metric == 'total_capex' and (pd.isna(val) or val == 0):
                capex_row = find_metric_in_forecast(cf_forecast, 'capex', 'CF')
                if not capex_row.empty and str(y) in capex_row.columns:
                    val = capex_row[str(y)].iloc[0]
            
            row_data[y] = f"${val:,.0f}" if pd.notna(val) else "-"
        else:
            row_data[y] = "-"
    cf_data.append(row_data)

cf_df = pd.DataFrame(cf_data)
display(cf_df)

print(f"\n✅ Сводная таблица 3-Statement построена")
print(f"📊 Периоды: {min(all_years_display)}-{max(all_years_display)} ({len(all_years_display)} лет)")


### 📊 Сводная таблица 3-Statement (последние 5 лет истории + прогноз)

✅ История IS загружена из витрины
✅ История BS загружена из витрины
✅ История CF загружена из витрины
✅ Таблица IS загружена из витрины (версия v2-test)
✅ Таблица BS загружена из витрины (версия v2-test)
✅ Таблица CF загружена из витрины (версия v2-test)

📅 Отчетные периоды: [2020, 2021, 2022, 2023, 2024] (история) + [2025, 2026, 2027, 2028, 2029] (прогноз)


#### 📈 Income Statement (Отчет о прибылях и убытках)

,Metric,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029
0,REVENUE,"$8,765,000,000","$20,275,000,000","$21,065,000,000","$18,053,000,000","$15,640,000,000","$13,549,526,395","$11,738,469,662","$10,169,482,386","$8,810,209,080","$7,632,618,956"
1,COGS,"$9,558,000,000","$14,533,000,000","$16,777,000,000","$15,803,000,000","$14,060,000,000","$12,302,337,460","$10,657,982,489","$9,233,415,285","$7,999,258,575","$6,930,061,713"
2,SGA,"$277,000,000","$426,000,000","$422,000,000","$501,000,000","$435,000,000","$379,766,521","$329,006,170","$285,030,549","$246,932,797","$213,927,267"
3,DEPRECIATION OWNED,"$-5,880,930,233","$-8,674,372,093","$-12,644,000,000","$-18,524,930,233","$-26,758,232,558","$1,739,100,000","$1,647,764,079","$1,547,468,951","$1,442,587,224","$1,336,430,789"
4,DEPRECIATION ROU,"$14,000,000","$14,000,000","$14,000,000","$15,000,000","$14,000,000","$101,600,000","$81,280,000","$65,024,000","$52,019,200","$41,615,360"
5,AMORTIZATION,"$8,000,000","$26,000,000","$42,000,000","$42,000,000","$20,000,000",-,-,-,-,-
6,TOTAL DA,"$-5,858,930,233","$-8,634,372,093","$-12,588,000,000","$-18,467,930,233","$-26,724,232,558","$1,840,700,000","$1,729,044,079","$1,612,492,951","$1,494,606,424","$1,378,046,149"
7,EBITDA,"$-1,070,000,000","$5,316,000,000","$3,866,000,000","$1,749,000,000","$1,145,000,000","$867,422,413","$751,481,003","$651,036,552","$564,017,708","$488,629,976"
8,EBIT,-,-,-,-,-,"$-973,277,587","$-977,563,076","$-961,456,399","$-930,588,716","$-889,416,173"
9,GAIN LOSS ON DISPOSAL,"$-149,000,000","$-7,000,000","$-12,000,000","$-6,000,000","$-5,000,000",-,-,-,-,-



#### 🏦 Balance Sheet (Баланс)

,Metric,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029
0,CASH,"$1,985,000,000","$2,522,000,000","$3,504,000,000","$2,948,000,000","$1,367,000,000","$1,065,345,428","$1,083,664,386","$1,056,171,696","$1,030,029,639","$1,003,600,591"
1,AR,"$34,000,000","$31,000,000","$22,000,000","$19,000,000","$21,000,000","$38,633,555","$33,469,717","$28,996,088","$25,120,413","$21,762,769"
2,INVENTORY,"$1,402,000,000","$2,210,000,000","$2,359,000,000","$2,128,000,000","$2,168,000,000","$1,535,132,595","$1,329,943,709","$1,152,180,780","$998,177,998","$864,759,535"
3,OTHER CURRENT ASSETS,"$51,000,000","$331,000,000","$368,000,000","$319,000,000","$299,000,000","$1,676,000,000","$1,676,000,000","$1,676,000,000","$1,676,000,000","$1,676,000,000"
4,PPE,"$5,444,000,000","$7,254,000,000","$8,492,000,000","$10,393,000,000","$11,973,000,000","$10,985,093,860","$10,316,459,675","$9,617,248,162","$8,909,538,595","$8,209,760,248"
5,INTANGIBLES,"$54,000,000","$519,000,000","$478,000,000","$436,000,000","$416,000,000","$416,000,000","$416,000,000","$416,000,000","$416,000,000","$416,000,000"
6,ROU ASSET,"$327,000,000","$345,000,000","$358,000,000","$440,000,000","$508,000,000","$406,400,000","$325,120,000","$260,096,000","$208,076,800","$166,461,440"
7,DTA,"$387,000,000","$646,000,000","$-350,000,000","$-429,000,000","$-660,000,000",-,-,-,-,-
8,OTHER NON CURRENT ASSETS,"$511,000,000","$984,000,000","$675,000,000","$838,000,000","$830,000,000","$905,088,408","$905,088,408","$905,088,408","$905,088,408","$905,088,408"
9,TOTAL ASSETS,"$12,059,000,000","$17,816,000,000","$19,458,000,000","$20,451,000,000","$20,235,000,000","$17,027,693,846","$16,085,745,895","$15,111,781,133","$14,168,031,852","$13,263,432,991"



#### 💰 Cash Flow Statement (Отчет о движении денежных средств)

,Metric,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029
0,NET INCOME,"$-1,165,000,000","$4,174,000,000","$2,537,000,000","$908,000,000","$397,000,000","$-1,148,253,283","$-1,130,329,028","$-1,138,681,103","$-1,131,591,510","$-1,115,007,735"
1,DEPRECIATION,"$40,000,000","$59,000,000","$86,000,000","$126,000,000","$182,000,000","$1,840,700,000","$1,729,044,079","$1,612,492,951","$1,494,606,424","$1,378,046,149"
2,AMORTIZATION,"$8,000,000","$26,000,000","$42,000,000","$42,000,000","$20,000,000",$0,$0,$0,$0,$0
3,INTEREST INCOME,-,-,-,-,-,-,-,-,-,-
4,INTEREST EXPENSE,-,-,-,-,-,-,-,-,-,-
5,TAX PAID,"$45,000,000","$-75,000,000","$-242,000,000","$-86,000,000","$-53,000,000",$0,$0,$0,$0,$0
6,WC DELTA,"$-122,000,000","$-51,000,000","$-29,000,000","$-199,000,000","$-136,000,000","$1,758,723,042","$19,418,008","$16,822,558","$14,574,022","$12,626,029"
7,AR DELTA,-,-,-,-,-,"$-17,633,555","$5,163,838","$4,473,629","$3,875,675","$3,357,645"
8,INVENTORY DELTA,-,-,-,-,-,"$632,867,405","$205,188,886","$177,762,930","$154,002,782","$133,418,463"
9,AP DELTA,-,-,-,-,-,"$-1,143,489,193","$190,934,716","$165,414,001","$143,304,435","$124,150,079"



✅ Сводная таблица 3-Statement построена
📊 Периоды: 2020-2029 (10 лет)


### 💰 Детальный анализ Cash Flow Statement

Детальный разбор всех компонентов движения денежных средств с историей и прогнозом.


In [ ]:
display(Markdown("#### 💰 Cash Flow Statement - Detailed (последние 5 лет истории + прогноз)"))

hist_cf = load_history_from_db('CF')
hist_is = load_history_from_db('IS')
hist_bs = load_history_from_db('BS')

history_years = sorted(hist_cf['year'].unique()) if not hist_cf.empty and 'year' in hist_cf.columns else []
cf_detail_raw = load_model_table('CF_DETAIL', canonical=False, csv_filename='outputs/model/3statement_CF_detail.csv')

cf_detail_metrics = None
if isinstance(cf_detail_raw, pd.DataFrame) and not cf_detail_raw.empty:
    if 'metric' in cf_detail_raw.columns:
        cf_detail_metrics = cf_detail_raw.copy().set_index('metric')
    elif 'year' in cf_detail_raw.columns:
        cf_detail_metrics = cf_detail_raw.set_index('year').T
    if cf_detail_metrics is not None:
        cf_detail_metrics.columns = [str(c) for c in cf_detail_metrics.columns]

cf_summary_rows = []

def _fmt_currency(val):
    return f"${val:,.0f}" if pd.notna(val) else '-'

def _metric_value(metric_name, year):
    if cf_detail_metrics is None or metric_name not in cf_detail_metrics.index:
        return float('nan')
    col = str(year)
    if col not in cf_detail_metrics.columns:
        return float('nan')
    return pd.to_numeric(cf_detail_metrics.loc[metric_name, col], errors='coerce')

if not hist_cf.empty and not hist_is.empty and not hist_bs.empty and history_years:
    for y in history_years:
        hist_cf_row = hist_cf[hist_cf['year'] == y]
        hist_is_row = hist_is[hist_is['year'] == y]
        hist_bs_row = hist_bs[hist_bs['year'] == y]
        if hist_cf_row.empty or hist_is_row.empty or hist_bs_row.empty:
            continue
        ar_curr = hist_bs_row['accounts_receivable'].iloc[0] if 'accounts_receivable' in hist_bs_row else 0
        inv_curr = hist_bs_row['inventory'].iloc[0] if 'inventory' in hist_bs_row else 0
        ap_curr = hist_bs_row['accounts_payable'].iloc[0] if 'accounts_payable' in hist_bs_row else 0
        if y > min(history_years):
            hist_bs_prev = hist_bs[hist_bs['year'] == y-1]
            if not hist_bs_prev.empty:
                ar_prev = hist_bs_prev['accounts_receivable'].iloc[0] if 'accounts_receivable' in hist_bs_prev else ar_curr
                inv_prev = hist_bs_prev['inventory'].iloc[0] if 'inventory' in hist_bs_prev else inv_curr
                ap_prev = hist_bs_prev['accounts_payable'].iloc[0] if 'accounts_payable' in hist_bs_prev else ap_curr
            else:
                ar_prev, inv_prev, ap_prev = ar_curr, inv_curr, ap_curr
        else:
            ar_prev, inv_prev, ap_prev = ar_curr, inv_curr, ap_curr
        wc_delta_calc = (ar_curr - ar_prev) + (inv_curr - inv_prev) - (ap_curr - ap_prev)
        ni = hist_is_row['net_income'].iloc[0] if 'net_income' in hist_is_row else 0
        dep = hist_is_row['depreciation'].iloc[0] if 'depreciation' in hist_is_row else 0
        row = {
            'Year': y,
            'Period': 'History',
            'NetIncome': _fmt_currency(ni),
            'Depreciation': _fmt_currency(dep),
            'WCDelta': _fmt_currency(wc_delta_calc),
            'CFO': _fmt_currency(hist_cf_row['cfo'].iloc[0]) if 'cfo' in hist_cf_row else '-',
            'MaintCapex': '-',
            'GrowthCapex': '-',
            'TotalCapex': _fmt_currency(abs(hist_cf_row['capex_cf'].iloc[0])) if 'capex_cf' in hist_cf_row else '-',
            'CFI': _fmt_currency(hist_cf_row['cfi'].iloc[0]) if 'cfi' in hist_cf_row else '-',
            'DebtBorrowings': _fmt_currency(hist_cf_row['cff_borrowings'].iloc[0]) if 'cff_borrowings' in hist_cf_row else '-',
            'DebtRepayments': _fmt_currency(hist_cf_row['cff_repayments'].iloc[0]) if 'cff_repayments' in hist_cf_row else '-',
            'RCDraws': '-',
            'RCRepayments': '-',
            'CFF': _fmt_currency(hist_cf_row['cff'].iloc[0]) if 'cff' in hist_cf_row else '-',
            'NetChange': _fmt_currency(hist_cf_row['net_change'].iloc[0]) if 'net_change' in hist_cf_row else '-'
        }
        cf_summary_rows.append(row)

forecast_years = []
if cf_detail_metrics is not None:
    forecast_years = sorted([int(c) for c in cf_detail_metrics.columns if str(c).isdigit()])
    for y in forecast_years:
        row = {
            'Year': y,
            'Period': 'Forecast',
            'NetIncome': _fmt_currency(_metric_value('net_income', y)),
            'Depreciation': _fmt_currency(_metric_value('depreciation', y)),
            'WCDelta': _fmt_currency(_metric_value('wc_delta', y)),
            'CFO': _fmt_currency(_metric_value('cfo', y)),
            'MaintCapex': _fmt_currency(_metric_value('maint_capex', y)),
            'GrowthCapex': _fmt_currency(_metric_value('growth_capex', y)),
            'TotalCapex': _fmt_currency(_metric_value('total_capex', y)),
            'CFI': _fmt_currency(_metric_value('cfi', y)),
            'DebtBorrowings': _fmt_currency(_metric_value('debt_borrowings', y)),
            'DebtRepayments': _fmt_currency(_metric_value('debt_repayments', y)),
            'RCDraws': _fmt_currency(_metric_value('rc_draws', y)),
            'RCRepayments': _fmt_currency(_metric_value('rc_repayments', y)),
            'CFF': _fmt_currency(_metric_value('cff', y)),
            'NetChange': _fmt_currency(_metric_value('net_change', y))
        }
        cf_summary_rows.append(row)

cf_summary_df = pd.DataFrame(cf_summary_rows)
if not cf_summary_df.empty:
    display(cf_summary_df)
else:
    print("⚠️ Недостаточно данных для формирования CF обзора")

if cf_detail_metrics is not None and forecast_years:
    cf_detail_numeric = cf_detail_metrics.apply(pd.to_numeric, errors='coerce')
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    def _plot_series(ax, metric, title):
        if not hist_cf.empty and metric in hist_cf.columns and history_years:
            hist_pairs = [(y, hist_cf[hist_cf['year'] == y][metric].iloc[0]) for y in history_years if not hist_cf[hist_cf['year'] == y].empty and pd.notna(hist_cf[hist_cf['year'] == y][metric].iloc[0])]
            if hist_pairs:
                hist_years_plot, hist_values_plot = zip(*hist_pairs)
                ax.plot(hist_years_plot, hist_values_plot, 'o-', color='blue', linewidth=2, markersize=8, label='History')
        forecast_vals = [cf_detail_numeric.loc[metric, str(y)] for y in forecast_years if metric in cf_detail_numeric.index and str(y) in cf_detail_numeric.columns]
        if forecast_vals:
            ax.plot(forecast_years[:len(forecast_vals)], forecast_vals, '^-', color='red', linewidth=2, markersize=8, label='Forecast')
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=1)
        ax.set_xlabel('Year'); ax.set_ylabel(f'{metric} ($ millions)'); ax.set_title(title)
        ax.legend(); ax.grid(True, alpha=0.3)

    _plot_series(axes[0,0], 'cfo', 'Cash Flow from Operations')
    _plot_series(axes[0,1], 'cfi', 'Cash Flow from Investing')
    _plot_series(axes[1,0], 'cff', 'Cash Flow from Financing')
    _plot_series(axes[1,1], 'net_change', 'Cash Net Change')

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Детальные данные CF отсутствуют")



## 📊 Детальный анализ моделирования статей <a id="section-4-analysis"></a>

In [ ]:
### 📈 Анализ Revenue (Выручка)

display(Markdown("#### 🔍 Метод моделирования и параметры"))

reg_df = load_model_table('REVENUE_REGRESSION_SUMMARY', canonical=False)
reg_summary_path = croot / "outputs" / "model" / "revenue_regression_summary.csv"
if reg_df.empty and reg_summary_path.exists():
    print("⚠️ Таблица REVENUE_REGRESSION_SUMMARY отсутствует в витрине, используем legacy CSV.")
    reg_df = load_company_csv('outputs/model/revenue_regression_summary.csv')

if reg_df.empty:
    print("❌ Данные по регрессионной модели Revenue не найдены ни в витрине, ни в CSV.")
else:
    method_used = None
    if 'note' in reg_df.columns and 'fallback_growth_rule' in reg_df['note'].values:
        method_used = "Fallback (рост от GDP или 2%)"
        display(Markdown("**Метод**: Fallback (простой рост)"))
    elif 'method' in reg_df.columns and reg_df['method'].notna().any():
        method_used = reg_df['method'].dropna().iloc[0]
        display(Markdown(f"**Метод**: {method_used}"))
    else:
        method_used = "Elastic Net Regression (Ridge)"
        display(Markdown("**Метод**: Elastic Net Regression с макрофакторами"))

    print(f"\n📊 Параметры модели:")
    print(f"  Метод: {method_used}")

    if 'base' in reg_df.columns and 'note' in reg_df.columns:
        base_rows = reg_df[reg_df['note'] == 'fallback_growth_rule']
        if not base_rows.empty:
            base_rev = pd.to_numeric(base_rows['base'], errors='coerce').dropna().iloc[0] if 'base' in base_rows else None
            if base_rev is not None:
                print(f"  Базовая Revenue (anchor): ${base_rev:,.0f}")

    if 'feature' in reg_df.columns and 'coef' in reg_df.columns:
        coefs = reg_df[(reg_df['feature'].notna()) & (reg_df['feature'].astype(str).str.lower() != 'intercept') & (reg_df['coef'].notna())].copy()
        if not coefs.empty:
            coefs['coef'] = pd.to_numeric(coefs['coef'], errors='coerce')
            coefs['abs_coef'] = coefs['coef'].abs()
            coefs = coefs.sort_values('abs_coef', ascending=False)
            print(f"\n📈 Коэффициенты регрессии (standardized):")
            for _, row in coefs.iterrows():
                feat = str(row['feature']).strip()
                coef = row['coef']
                if pd.isna(coef):
                    continue
                sign = '+' if coef > 0 else ''
                abs_coef = abs(coef)
                importance = '🔴 высокий' if abs_coef > 0.1 else '🟡 средний' if abs_coef > 0.05 else '🟢 низкий'
                print(f"  {feat}: {sign}{coef:.6f} ({importance})")
            top3 = coefs.head(3)
            if not top3.empty:
                print(f"\n  🏆 Топ-3 фактора по важности:")
                for i, (_, row) in enumerate(top3.iterrows(), 1):
                    print(f"    {i}. {str(row['feature']).strip()}: {row['coef']:+.6f}")

        intercept_rows = reg_df[reg_df['feature'].astype(str).str.contains('intercept', case=False, na=False)] if 'feature' in reg_df.columns else pd.DataFrame()
        if not intercept_rows.empty and 'coef' in intercept_rows.columns:
            intercept_val = pd.to_numeric(intercept_rows['coef'], errors='coerce').dropna()
            if not intercept_val.empty:
                print(f"\n  Intercept: {intercept_val.iloc[0]:.6f}")

    if 'metric' in reg_df.columns and 'value' in reg_df.columns:
        metrics = reg_df[reg_df['metric'].notna()].copy()
        if not metrics.empty:
            print(f"\n📊 Метрики качества модели:")
            for _, row in metrics.iterrows():
                name = str(row['metric']).strip()
                val = pd.to_numeric(row.get('value'), errors='coerce')
                if pd.isna(val):
                    continue
                if name == 'R2':
                    print(f"  R2: {val:.4f} ({val*100:.1f}%) {'✅ Отличное качество' if val > 0.9 else '⚠️ Хорошее качество' if val > 0.7 else '❌ Требует улучшения'}")
                elif name == 'MAPE_pct':
                    print(f"  MAPE: {val:.2f}% {'⚠️ Высокий MAPE' if val > 100 else '✅ Приемлемый MAPE'}")
                else:
                    print(f"  {name}: {val}")

    if 'target_transform' in reg_df.columns or 'feature_transform' in reg_df.columns:
        target_tf = reg_df['target_transform'].dropna().iloc[0] if 'target_transform' in reg_df.columns and reg_df['target_transform'].notna().any() else None
        feature_tf = reg_df['feature_transform'].dropna().iloc[0] if 'feature_transform' in reg_df.columns and reg_df['feature_transform'].notna().any() else None
        if target_tf or feature_tf:
            tf_desc = {
                'dln': 'Δln (разность логарифмов, темп роста)',
                'dln_yoy': 'Δln YoY (годовой темп роста)',
                'ln': 'ln (натуральный логарифм)',
                'none': 'нет (уровни)',
                'pct': 'процент',
                'zscore': 'стандартизация'
            }
            print(f"\n🔄 Трансформации:")
            if target_tf:
                print(f"  Target transform: {target_tf} - {tf_desc.get(str(target_tf).lower(), target_tf)}")
            if feature_tf:
                print(f"  Feature transform: {feature_tf} - {tf_desc.get(str(feature_tf).lower(), feature_tf)}")
            if str(target_tf).lower() == 'dln' or str(feature_tf).lower() == 'dln':
                print("  ⚠️ Используются dln трансформации — MAPE может быть повышен")

    if 'info' in reg_df.columns:
        train_info = reg_df[reg_df['info'] == 'train_test_split'] if 'info' in reg_df.columns else pd.DataFrame()
        if not train_info.empty:
            train_end = pd.to_numeric(train_info.get('train_end_year'), errors='coerce').dropna()
            if not train_end.empty:
                print(f"\n🔬 Train/Test Split:")
                print(f"  Train end year: {int(train_end.iloc[0])}")
            train_years_info = reg_df[reg_df['info'] == 'train_years']
            if not train_years_info.empty and 'value' in train_years_info.columns:
                print(f"  Train years: {train_years_info['value'].iloc[0]}")
            test_years_info = reg_df[reg_df['info'] == 'test_years']
            if not test_years_info.empty and 'value' in test_years_info.columns:
                print(f"  Test years: {test_years_info['value'].iloc[0]}")

    if 'reconstruction_method' in reg_df.columns:
        recon = reg_df[reg_df['reconstruction_method'].notna()]
        if not recon.empty:
            method = recon['reconstruction_method'].iloc[0]
            anchor_year = pd.to_numeric(recon.get('reconstruction_anchor_year'), errors='coerce').dropna()
            anchor_value = pd.to_numeric(recon.get('reconstruction_anchor_value'), errors='coerce').dropna()
            print(f"\n🔗 Восстановление уровней:")
            print(f"  Метод: {method}")
            if not anchor_year.empty:
                print(f"  Anchor year: {int(anchor_year.iloc[0])}")
            if not anchor_value.empty:
                print(f"  Anchor value: ${anchor_value.iloc[0]:,.0f}")



In [ ]:
### 📊 График Revenue: Actual vs Fitted и Прогноз

display(Markdown("#### 🔍 Проверка истории и прогноза выручки"))

proj_yaml_path = croot / "configs" / "project.yaml"
if proj_yaml_path.exists():
    proj_yaml = yaml.safe_load(proj_yaml_path.read_text(encoding='utf-8'))
else:
    proj_yaml = {}

# Исторические данные Revenue

def _fmt_currency(val):
    if val is None or pd.isna(val):
        return '-'
    return f"${val:,.0f}"

hist_is = load_history_from_db('IS')
hist_revenue = {}
if not hist_is.empty and 'year' in hist_is.columns and 'revenue' in hist_is.columns:
    for _, row in hist_is.iterrows():
        year = row.get('year')
        revenue = pd.to_numeric(row.get('revenue'), errors='coerce')
        if pd.notna(year) and pd.notna(revenue):
            hist_revenue[int(year)] = float(revenue)
else:
    print("⚠️ Исторические данные Revenue недоступны")

# Прогноз выручки из витрины
is_forecast = load_model_forecast_from_db('IS', canonical=True)
forecast_revenue = {}
if not is_forecast.empty and 'metric' in is_forecast.columns:
    revenue_row = is_forecast[is_forecast['metric'].str.lower() == 'revenue']
    if not revenue_row.empty:
        for col in revenue_row.columns:
            if str(col).isdigit():
                val = pd.to_numeric(revenue_row.iloc[0][col], errors='coerce')
                if pd.notna(val):
                    forecast_revenue[int(col)] = float(val)
else:
    print("⚠️ Прогноз Revenue не найден ни в витрине, ни в CSV")

# Fitted значения (train/test)
fitted_values = {}
train_fitted = load_model_table('TRAIN_PERIOD_FITTED', csv_filename='outputs/model/train_period_fitted.csv')
test_forecasts = load_model_table('TEST_PERIOD_FORECASTS', csv_filename='outputs/model/test_period_forecasts.csv')
for df in [train_fitted, test_forecasts]:
    if df.empty:
        continue
    df_use = df.copy()
    if 'metric' in df_use.columns:
        df_use = df_use[df_use['metric'].str.lower() == 'revenue']
    for _, row in df_use.iterrows():
        year = pd.to_numeric(row.get('year'), errors='coerce')
        value = row.get('fitted', row.get('forecast'))
        value = pd.to_numeric(value, errors='coerce')
        if pd.notna(year) and pd.notna(value):
            fitted_values[int(year)] = float(value)

# Подготовка диапазонов лет
periods_config = proj_yaml.get('model', {}).get('standard', {}).get('periods', {})
forecast_start_year = periods_config.get('forecast_start_year')
forecast_end_year = periods_config.get('forecast_end_year')

history_years = sorted(hist_revenue.keys())
forecast_years = sorted(forecast_revenue.keys())
fitted_years = sorted(fitted_values.keys())

# Визуализация
if history_years or forecast_years:
    fig, ax = plt.subplots(figsize=(14, 6))
    if history_years:
        ax.plot(history_years, [hist_revenue[y] for y in history_years], 'o-', color='blue', linewidth=2, markersize=8, label='History')
    if fitted_years:
        ax.plot(fitted_years, [fitted_values[y] for y in fitted_years], 's--', color='green', linewidth=2, markersize=6, label='Fitted/Test')
    if forecast_years:
        ax.plot(forecast_years, [forecast_revenue[y] for y in forecast_years], '^-', color='red', linewidth=2, markersize=8, label='Forecast')
    if history_years and forecast_years:
        ax.axvline(x=max(history_years) + 0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Year')
    ax.set_ylabel('Revenue ($ millions)')
    ax.set_title('Revenue: History vs Fitted vs Forecast')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Недостаточно данных для построения графика Revenue")

# Сводная таблица
all_years = sorted(set(history_years + fitted_years + forecast_years))
if forecast_start_year and forecast_end_year:
    expected_years = range(forecast_start_year, forecast_end_year + 1)
    all_years = sorted(set(all_years) | set(expected_years))

rows = []
for y in all_years:
    rows.append({
        'Year': y,
        'Actual': _fmt_currency(hist_revenue.get(y)),
        'Fitted': _fmt_currency(fitted_values.get(y)),
        'Forecast': _fmt_currency(forecast_revenue.get(y))
    })
summary_df = pd.DataFrame(rows)

if not summary_df.empty:
    display(summary_df)
    if forecast_start_year and forecast_end_year:
        missing_years = [y for y in range(forecast_start_year, forecast_end_year + 1) if y not in forecast_revenue]
        if missing_years:
            print(f"⚠️ Отсутствуют значения прогноза Revenue для лет: {missing_years}")
        else:
            print(f"✅ Прогноз Revenue заполнен на весь горизонт {forecast_start_year}-{forecast_end_year}")
else:
    print("⚠️ Нет данных для сводной таблицы Revenue")


### 🔍 Детальный анализ сопоставимости истории и прогноза

Анализ каждой статьи с проверкой:
- Знаков (положительные/отрицательные)
- Порядка чисел (масштаб)
- Логики изменений
- Сопоставимости величин

In [ ]:
# Детальный анализ сопоставимости истории и прогноза
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

display(Markdown("### 🔍 Детальный анализ каждой статьи: История vs Прогноз"))

# Загружаем данные если еще не загружены
if 'hist_is' not in locals() or hist_is.empty:
    hist_is = load_history_from_db('IS', canonical=True)
if 'hist_bs' not in locals() or hist_bs.empty:
    hist_bs = load_history_from_db('BS', canonical=True)
if 'hist_cf' not in locals() or hist_cf.empty:
    hist_cf = load_history_from_db('CF', canonical=True)

if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)
if 'forecast_bs' not in locals() or forecast_bs.empty:
    forecast_bs = load_model_forecast_from_db('BS', canonical=True)
if 'forecast_cf' not in locals() or forecast_cf.empty:
    forecast_cf = load_model_forecast_from_db('CF', canonical=True)

# Преобразуем в wide format
def to_wide(df, metric_col='metric'):
    if df.empty:
        return pd.DataFrame()
    if 'year' in df.columns and metric_col in df.columns:
        return df.pivot(index=metric_col, columns='year', values='value')
    elif metric_col in df.columns:
        return df.set_index(metric_col)
    return df

hist_is_wide = to_wide(hist_is)
hist_bs_wide = to_wide(hist_bs)
hist_cf_wide = to_wide(hist_cf)
forecast_is_wide = to_wide(forecast_is)
forecast_bs_wide = to_wide(forecast_bs)
forecast_cf_wide = to_wide(forecast_cf)

# Получаем годы
hist_years = sorted([int(c) for c in hist_is_wide.columns if str(c).isdigit() and int(c) <= 2024])[-5:]
forecast_years = sorted([int(c) for c in forecast_is_wide.columns if str(c).isdigit() and int(c) > 2024])

def get_value(df, metric, year):
    """Безопасное получение значения"""
    if df.empty or metric not in df.index:
        return None
    year_str = str(year)
    if year_str not in df.columns:
        return None
    val = pd.to_numeric(df.loc[metric, year_str], errors='coerce')
    return float(val) if pd.notna(val) else None

def analyze_item(item_name, hist_df, forecast_df):
    """Анализирует одну статью"""
    issues = []
    warnings = []
    
    in_hist = item_name in hist_df.index if not hist_df.empty else False
    in_forecast = item_name in forecast_df.index if not forecast_df.empty else False
    
    if not in_hist and not in_forecast:
        return None
    
    # Получаем последние значения истории и первые прогноза
    hist_vals = {}
    forecast_vals = {}
    
    if in_hist:
        for y in hist_years:
            val = get_value(hist_df, item_name, y)
            if val is not None:
                hist_vals[y] = val
    
    if in_forecast:
        for y in forecast_years:
            val = get_value(forecast_df, item_name, y)
            if val is not None:
                forecast_vals[y] = val
    
    if not hist_vals and not forecast_vals:
        return None
    
    result = {
        'Статья': item_name,
        'В истории': '✅' if in_hist else '❌',
        'В прогнозе': '✅' if in_forecast else '❌',
        'Проблемы': [],
        'Предупреждения': []
    }
    
    if hist_vals and forecast_vals:
        # 1. Проверка знаков
        hist_nonzero = [v for v in hist_vals.values() if abs(v) > 1e-6]
        forecast_nonzero = [v for v in forecast_vals.values() if abs(v) > 1e-6]
        
        if hist_nonzero and forecast_nonzero:
            hist_signs = [1 if v > 0 else -1 for v in hist_nonzero]
            forecast_signs = [1 if v > 0 else -1 for v in forecast_nonzero]
            
            hist_typical = max(set(hist_signs), key=hist_signs.count) if hist_signs else 0
            forecast_typical = max(set(forecast_signs), key=forecast_signs.count) if forecast_signs else 0
            
            if hist_typical != 0 and forecast_typical != 0 and hist_typical != forecast_typical:
                result['Проблемы'].append(f"❌ Смена знака: история {('+' if hist_typical > 0 else '-')}, прогноз {('+' if forecast_typical > 0 else '-')}")
        
        # 2. Проверка масштаба
        hist_abs = [abs(v) for v in hist_vals.values() if abs(v) > 1e-6]
        forecast_abs = [abs(v) for v in forecast_vals.values() if abs(v) > 1e-6]
        
        if hist_abs and forecast_abs:
            hist_avg = np.mean(hist_abs)
            forecast_avg = np.mean(forecast_abs)
            
            if hist_avg > 0:
                ratio = forecast_avg / hist_avg
                if ratio > 10 or ratio < 0.1:
                    result['Проблемы'].append(f"❌ Разница в масштабе: {ratio:.1f}x (история ~${hist_avg:,.0f}, прогноз ~${forecast_avg:,.0f})")
                elif ratio > 3 or ratio < 0.33:
                    result['Предупреждения'].append(f"⚠️ Значительное изменение масштаба: {ratio:.1f}x")
        
        # 3. Проверка перехода
        if len(hist_vals) >= 1 and len(forecast_vals) >= 1:
            hist_last = list(hist_vals.values())[-1]
            forecast_first = list(forecast_vals.values())[0]
            
            jump = forecast_first - hist_last
            if abs(hist_last) > 1e-6:
                jump_pct = (jump / abs(hist_last)) * 100
                if abs(jump_pct) > 500:
                    result['Проблемы'].append(f"❌ Резкий скачок на границе: {jump:,.0f} ({jump_pct:+.0f}%)")
                elif abs(jump_pct) > 200:
                    result['Предупреждения'].append(f"⚠️ Большое изменение на границе: {jump:,.0f} ({jump_pct:+.0f}%)")
    
    elif not in_hist and in_forecast:
        result['Предупреждения'].append("⚠️ Статья отсутствует в истории, но есть в прогнозе")
    elif in_hist and not in_forecast:
        result['Предупреждения'].append("⚠️ Статья есть в истории, но отсутствует в прогнозе")
    
    return result

# Анализируем ключевые статьи
is_items = ['revenue', 'cogs', 'sga', 'depreciation_owned', 'depreciation_rou', 'amortization', 
            'ebitda', 'ebit', 'interest_income', 'interest_expense', 'lease_interest', 'tax', 'net_income']
bs_items = ['cash', 'ar', 'inventory', 'ppe', 'intangibles', 'rou_asset', 'total_assets',
            'st_debt', 'ap', 'debt', 'lease_liability', 'equity', 'total_liabilities', 'total_equity']
cf_items = ['net_income', 'depreciation', 'amortization', 'wc_delta', 'cfo', 'capex', 'cfi', 
            'debt_borrowings', 'debt_repayments', 'dividends', 'cff', 'net_change']

is_analysis = [analyze_item(item, hist_is_wide, forecast_is_wide) for item in is_items]
is_analysis = [a for a in is_analysis if a is not None]

bs_analysis = [analyze_item(item, hist_bs_wide, forecast_bs_wide) for item in bs_items]
bs_analysis = [a for a in bs_analysis if a is not None]

cf_analysis = [analyze_item(item, hist_cf_wide, forecast_cf_wide) for item in cf_items]
cf_analysis = [a for a in cf_analysis if a is not None]

# Выводим результаты
def print_analysis(analysis_list, title):
    display(Markdown(f"### {title}"))
    
    for item in analysis_list:
        if item['Проблемы'] or item['Предупреждения']:
            display(Markdown(f"#### {item['Статья']}"))
            print(f"  В истории: {item['В истории']} | В прогнозе: {item['В прогнозе']}")
            
            if item['Проблемы']:
                print(f"  \n  🔴 ПРОБЛЕМЫ:")
                for prob in item['Проблемы']:
                    print(f"    {prob}")
            
            if item['Предупреждения']:
                print(f"  \n  ⚠️ ПРЕДУПРЕЖДЕНИЯ:")
                for warn in item['Предупреждения']:
                    print(f"    {warn}")
            print()

print_analysis(is_analysis, '📊 Income Statement (IS)')
print_analysis(bs_analysis, '🏦 Balance Sheet (BS)')
print_analysis(cf_analysis, '💰 Cash Flow Statement (CF)')

# Сводная таблица
all_issues = []
for analysis in [is_analysis, bs_analysis, cf_analysis]:
    for item in analysis:
        if item['Проблемы'] or item['Предупреждения']:
            all_issues.append({
                'Статья': item['Статья'],
                'Проблем': len(item['Проблемы']),
                'Предупреждений': len(item['Предупреждения']),
                'Статус': '❌' if item['Проблемы'] else '⚠️'
            })

if all_issues:
    display(Markdown("### 📋 Сводная таблица проблем"))
    issues_df = pd.DataFrame(all_issues)
    display(issues_df)
    print(f"\nВсего статей с проблемами: {len(all_issues)}")
    print(f"Критических проблем: {sum(1 for i in all_issues if i['Проблем'] > 0)}")
    print(f"Предупреждений: {sum(1 for i in all_issues if i['Предупреждений'] > 0)}")
else:
    print("✅ Все статьи проверены, критических проблем не обнаружено")


### 💰 Анализ других статей (COGS, SG&A, Depreciation, Interest, Tax)


In [ ]:
# Анализ COGS
display(Markdown("#### 💸 COGS (Себестоимость)"))

cogs_config = proj_yaml.get("model", {}).get("standard", {}).get("cogs", {})
print(f"📋 Преднастройки COGS:")
print(f"  Clamp min: {cogs_config.get('clamp', {}).get('min', 0.55)}")
print(f"  Clamp max: {cogs_config.get('clamp', {}).get('max', 0.98)}")
print(f"  PPI uplift: {cogs_config.get('use_ppi_uplift', False)}")

hist_is = load_history_from_db('IS')
cogs_hist = {}
rev_hist_for_cogs = {}
if not hist_is.empty and 'year' in hist_is.columns:
    for _, row in hist_is.iterrows():
        year = row.get('year')
        if pd.notna(year):
            year = int(year)
            cogs_val = pd.to_numeric(row.get('cogs'), errors='coerce')
            rev_val = pd.to_numeric(row.get('revenue'), errors='coerce')
            if pd.notna(cogs_val) and pd.notna(rev_val) and rev_val != 0:
                cogs_hist[year] = float(cogs_val)
                rev_hist_for_cogs[year] = float(rev_val)

if cogs_hist:
    ratios = [cogs_hist[y]/rev_hist_for_cogs[y] for y in sorted(cogs_hist) if y in rev_hist_for_cogs and rev_hist_for_cogs[y] != 0]
    if ratios:
        print(f"\n📊 Исторические параметры COGS:")
        print(f"  Средний COGS/Revenue ratio: {np.mean(ratios):.2%}")
        print(f"  Минимальный ratio: {min(ratios):.2%}")
        print(f"  Максимальный ratio: {max(ratios):.2%}")
        print(f"  Используемый в модели (clamp): {max(0.55, min(0.98, np.mean(ratios))):.2%}")

is_forecast_df = load_model_forecast_from_db('IS')
if is_forecast_df.empty:
    legacy_path = croot / "outputs" / "model" / "3statement_IS.csv"
    if legacy_path.exists():
        print("⚠️ Прогноз IS отсутствует в витрине, используем legacy CSV.")
        is_forecast_df = load_company_csv('outputs/model/3statement_IS.csv')

sg_df = None
if not is_forecast_df.empty:
    if 'metric' in is_forecast_df.columns:
        sg_df = is_forecast_df.set_index('metric')
    else:
        sg_df = is_forecast_df

if sg_df is not None and 'cogs' in sg_df.index:
    forecast_years_all = [int(c) for c in sg_df.columns if str(c).isdigit()]
    forecast_years_all.sort()
    if forecast_years_all:
        print(f"\n📊 Прогноз COGS (первые 3 года):")
        for y in forecast_years_all[:3]:
            val = pd.to_numeric(sg_df.loc['cogs', str(y)], errors='coerce') if str(y) in sg_df.columns else pd.NA
            if pd.notna(val):
                print(f"  {y}: ${val:,.0f}")

    test_forecasts = load_model_table('TEST_PERIOD_FORECASTS', canonical=False)
    if test_forecasts.empty:
        legacy_test_path = croot / "outputs" / "model" / "test_period_forecasts.csv"
        if legacy_test_path.exists():
            print("⚠️ TEST_PERIOD_FORECASTS отсутствует в витрине, используем legacy CSV.")
            test_forecasts = load_company_csv('outputs/model/test_period_forecasts.csv')

    fig, ax = plt.subplots(figsize=(14, 6))
    if cogs_hist:
        hist_years = sorted(cogs_hist.keys())
        ax.plot(hist_years, [cogs_hist[y] for y in hist_years], 'o-', color='blue', linewidth=2, markersize=8, label='Actual (History)')

    if isinstance(test_forecasts, pd.DataFrame) and not test_forecasts.empty:
        tf = test_forecasts.copy()
        if 'metric' in tf.columns:
            tf = tf[tf['metric'] == 'cogs']
        if 'year' in tf.columns:
            fitted_pairs = [(int(row['year']), pd.to_numeric(row.get('fitted', row.get('forecast')), errors='coerce'))
                            for _, row in tf.iterrows() if pd.notna(row.get('year'))]
            fitted_pairs = [(y, v) for y, v in fitted_pairs if pd.notna(v)]
            fitted_pairs.sort(key=lambda x: x[0])
            if fitted_pairs:
                ax.plot([y for y, _ in fitted_pairs], [v for _, v in fitted_pairs], 's--', color='green', linewidth=2, markersize=6, label='Fitted (Test Period)', alpha=0.7)

    if sg_df is not None and 'cogs' in sg_df.index and forecast_years_all:
        forecast_vals = [pd.to_numeric(sg_df.loc['cogs', str(y)], errors='coerce') for y in forecast_years_all if str(y) in sg_df.columns]
        forecast_years_plot = [y for y, v in zip(forecast_years_all, forecast_vals) if pd.notna(v)]
        forecast_vals_plot = [v for v in forecast_vals if pd.notna(v)]
        if forecast_vals_plot:
            ax.plot(forecast_years_plot, forecast_vals_plot, '^-', color='red', linewidth=2, markersize=8, label='Forecast')

    if cogs_hist and forecast_years_all:
        ax.axvline(x=max(cogs_hist.keys()) + 0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)

    ax.set_xlabel('Year', fontsize=12, fontweight='bold')
    ax.set_ylabel('COGS ($ millions)', fontsize=12, fontweight='bold')
    ax.set_title('COGS: Actual vs Fitted vs Forecast', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Прогноз COGS не найден")



In [ ]:
# Анализ SG&A
display(Markdown("#### 📊 SG&A (Операционные расходы)"))

sgna_config = proj_yaml.get("model", {}).get("standard", {}).get("sgna", {})
print(f"📋 Преднастройки SG&A:")
print(f"  Метод: {sgna_config.get('base_method', 'ewa')}")
if sgna_config.get('base_method') == 'ewa':
    print(f"  EWA half-life: {sgna_config.get('ewa_halflife_years', 2)} лет")
print(f"  Transform: {sgna_config.get('transform', 'none')}")
print(f"  CPI индексация: {sgna_config.get('index_by_cpi', False)}")
if sgna_config.get('index_by_cpi', False):
    print(f"  CPI beta: {sgna_config.get('cpi_beta', 1.0)}")

hist_is = load_history_from_db('IS')
sgna_hist = {}
if not hist_is.empty and 'year' in hist_is.columns:
    for _, row in hist_is.iterrows():
        year = row.get('year')
        if pd.notna(year):
            year = int(year)
            sgna_val = row.get('sgna', row.get('sga', row.get('opex')))
            sgna_val = pd.to_numeric(sgna_val, errors='coerce')
            if pd.notna(sgna_val):
                sgna_hist[year] = float(sgna_val)

if sgna_hist:
    hist_years = sorted(sgna_hist.keys())
    sgna_values = [sgna_hist[y] for y in hist_years]
    print(f"\n📊 Исторические параметры SG&A:")
    print(f"  Период: {hist_years[0]}-{hist_years[-1]}")
    print(f"  Среднее SG&A: ${np.mean(sgna_values):,.0f}")
    print(f"  Минимум: ${min(sgna_values):,.0f}")
    print(f"  Максимум: ${max(sgna_values):,.0f}")

is_forecast_df = load_model_forecast_from_db('IS')
if is_forecast_df.empty:
    legacy_path = croot / "outputs" / "model" / "3statement_IS.csv"
    if legacy_path.exists():
        print("⚠️ Прогноз IS отсутствует в витрине, используем legacy CSV.")
        is_forecast_df = load_company_csv('outputs/model/3statement_IS.csv')

forecast_years_all = []
sgna_forecast = None
if not is_forecast_df.empty:
    if 'metric' in is_forecast_df.columns:
        sgna_forecast = is_forecast_df.set_index('metric')
    else:
        sgna_forecast = is_forecast_df
    if 'sga' in sgna_forecast.index:
        forecast_years_all = [int(c) for c in sgna_forecast.columns if str(c).isdigit()]
        forecast_years_all.sort()
        if forecast_years_all:
            print(f"\n📊 Прогноз SG&A (первые 3 года):")
            for y in forecast_years_all[:3]:
                val = pd.to_numeric(sgna_forecast.loc['sga', str(y)], errors='coerce') if str(y) in sgna_forecast.columns else pd.NA
                if pd.notna(val):
                    print(f"  {y}: ${val:,.0f}")

test_forecasts = load_model_table('TEST_PERIOD_FORECASTS', canonical=False)
if test_forecasts.empty:
    legacy_test_path = croot / "outputs" / "model" / "test_period_forecasts.csv"
    if legacy_test_path.exists():
        print("⚠️ TEST_PERIOD_FORECASTS отсутствует в витрине, используем legacy CSV.")
        test_forecasts = load_company_csv('outputs/model/test_period_forecasts.csv')

fig, ax = plt.subplots(figsize=(14, 6))
if sgna_hist:
    hist_years = sorted(sgna_hist.keys())
    ax.plot(hist_years, [sgna_hist[y] for y in hist_years], 'o-', color='blue', linewidth=2, markersize=8, label='Actual (History)')

if isinstance(test_forecasts, pd.DataFrame) and not test_forecasts.empty:
    tf = test_forecasts.copy()
    if 'metric' in tf.columns:
        tf = tf[tf['metric'].isin(['sga', 'sgna'])]
    if 'year' in tf.columns:
        fitted_pairs = [(int(row['year']), pd.to_numeric(row.get('fitted', row.get('forecast')), errors='coerce'))
                        for _, row in tf.iterrows() if pd.notna(row.get('year'))]
        fitted_pairs = [(y, v) for y, v in fitted_pairs if pd.notna(v)]
        fitted_pairs.sort(key=lambda x: x[0])
        if fitted_pairs:
            ax.plot([y for y, _ in fitted_pairs], [v for _, v in fitted_pairs], 's--', color='green', linewidth=2, markersize=6, label='Fitted (Test Period)', alpha=0.7)

if sgna_forecast is not None and 'sga' in sgna_forecast.index and forecast_years_all:
    forecast_vals = [pd.to_numeric(sgna_forecast.loc['sga', str(y)], errors='coerce') for y in forecast_years_all if str(y) in sgna_forecast.columns]
    forecast_years_plot = [y for y, v in zip(forecast_years_all, forecast_vals) if pd.notna(v)]
    forecast_vals_plot = [v for v in forecast_vals if pd.notna(v)]
    if forecast_vals_plot:
        ax.plot(forecast_years_plot, forecast_vals_plot, '^-', color='red', linewidth=2, markersize=8, label='Forecast')

if sgna_hist and forecast_years_all:
    ax.axvline(x=max(sgna_hist.keys()) + 0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)

ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('SG&A ($ millions)', fontsize=12, fontweight='bold')
ax.set_title('SG&A: Actual vs Fitted vs Forecast', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### 💰 Анализ CapEx


In [ ]:
# Анализ CapEx
display(Markdown("#### 💰 CapEx (Капитальные расходы)"))

capex_policy = proj_yaml.get("model", {}).get("standard", {}).get("capex_policy", {})
print("📋 Метод моделирования CapEx:")
print(f"  Method: {capex_policy.get('method', 'ratio_to_revenue')}")
print(f"  Ratio default: {capex_policy.get('ratio_default', 0.08):.1%}")

hist_cf = load_history_from_db('CF')
capex_hist = {}
if not hist_cf.empty and 'year' in hist_cf.columns:
    capex_col = None
    for col in hist_cf.columns:
        if 'capex' in str(col).lower():
            capex_col = col
            break
    if capex_col:
        for _, row in hist_cf.iterrows():
            year = row.get('year')
            val = pd.to_numeric(row.get(capex_col), errors='coerce')
            if pd.notna(year) and pd.notna(val):
                capex_hist[int(year)] = abs(float(val))

hist_is = load_history_from_db('IS')
rev_hist = {}
if not hist_is.empty and 'year' in hist_is.columns:
    for _, row in hist_is.iterrows():
        year = row.get('year')
        val = pd.to_numeric(row.get('revenue'), errors='coerce')
        if pd.notna(year) and pd.notna(val):
            rev_hist[int(year)] = float(val)

if capex_hist and rev_hist:
    ratios = [capex_hist[y] / rev_hist[y] for y in capex_hist if y in rev_hist and rev_hist[y] > 0]
    if ratios:
        print(" 📊 Исторический CapEx/Revenue:")
        print(f"  Среднее: {np.mean(ratios):.2%}")
        print(f"  Диапазон: {np.min(ratios):.2%} - {np.max(ratios):.2%}")

capex_breakdown = load_model_table('CAPEX_BREAKDOWN', csv_filename='outputs/model/capex_breakdown.csv')
is_forecast = load_model_forecast_from_db('IS', canonical=True)
if capex_breakdown.empty or is_forecast.empty:
    print("⚠️ CapEx breakdown или прогноз IS отсутствуют")
else:
    capex_metrics = capex_breakdown.set_index('metric') if 'metric' in capex_breakdown.columns else capex_breakdown
    rev_metrics = is_forecast.set_index('metric') if 'metric' in is_forecast.columns else is_forecast
    forecast_years = sorted([int(c) for c in capex_breakdown.columns if str(c).isdigit()])
    total_capex = [pd.to_numeric(capex_metrics.loc['TotalCapex', str(y)], errors='coerce') if 'TotalCapex' in capex_metrics.index else float('nan') for y in forecast_years]
    maint_capex = [pd.to_numeric(capex_metrics.loc.get('MaintCapex', {}).get(str(y)), errors='coerce') for y in forecast_years]
    growth_capex = [pd.to_numeric(capex_metrics.loc.get('GrowthCapex', {}).get(str(y)), errors='coerce') for y in forecast_years]
    revenue_vals = [pd.to_numeric(rev_metrics.loc['revenue', str(y)], errors='coerce') if 'revenue' in rev_metrics.index else float('nan') for y in forecast_years]
    ratios_forecast = []
    for capex_val, rev_val in zip(total_capex, revenue_vals):
        if pd.notna(capex_val) and pd.notna(rev_val) and rev_val != 0:
            ratios_forecast.append(float(capex_val)/float(rev_val)*100)
        else:
            ratios_forecast.append(float('nan'))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    hist_years_capex = sorted(capex_hist.keys())
    ax1.plot(hist_years_capex, [capex_hist[y] for y in hist_years_capex], 'o-', label='History', linewidth=2, markersize=8)
    ax1.plot(forecast_years, [float(v) if pd.notna(v) else 0 for v in total_capex], '^-', label='Forecast', linewidth=2, markersize=8)
    if maint_capex and growth_capex:
        ax1.plot(forecast_years, [float(v) if pd.notna(v) else 0 for v in maint_capex], 's--', label='Maint CapEx', linewidth=1.5)
        ax1.plot(forecast_years, [float(v) if pd.notna(v) else 0 for v in growth_capex], 's--', label='Growth CapEx', linewidth=1.5)
    ax1.set_title('CapEx: History vs Forecast'); ax1.set_xlabel('Year'); ax1.set_ylabel('$ millions'); ax1.legend(); ax1.grid(True, alpha=0.3)
    if hist_years_capex:
        ax1.axvline(x=max(hist_years_capex) + 0.5, color='gray', linestyle='--', alpha=0.5)
    hist_ratios = [capex_hist[y] / rev_hist[y] * 100 for y in hist_years_capex if y in rev_hist and rev_hist[y] > 0]
    ax2.plot(hist_years_capex, hist_ratios, 'o-', label='History %', linewidth=2, markersize=8)
    ax2.plot(forecast_years, ratios_forecast, '^-', label='Forecast %', linewidth=2, markersize=8)
    ax2.set_title('CapEx / Revenue Ratio'); ax2.set_xlabel('Year'); ax2.set_ylabel('%'); ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


In [ ]:
# Анализ PP&E
display(Markdown("#### 🏭 PP&E (Основные средства)"))

ppe_cfg = proj_yaml.get("model", {}).get("standard", {}).get("ppe", {})
print(f"📋 Настройки PP&E corkscrew: {ppe_cfg}")

bs_forecast = load_model_forecast_from_db('BS', canonical=True)
hist_bs = load_history_from_db('BS')
capex_breakdown = load_model_table('CAPEX_BREAKDOWN', csv_filename='outputs/model/capex_breakdown.csv')

if bs_forecast.empty:
    print("⚠️ Прогноз BS отсутствует")
else:
    bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns else bs_forecast
    forecast_years = sorted([int(c) for c in bs_forecast.columns if str(c).isdigit()])
    ppe_vals = [pd.to_numeric(bs_metrics.loc['ppe_net', str(y)], errors='coerce') if 'ppe_net' in bs_metrics.index else float('nan') for y in forecast_years]
    fig, ax = plt.subplots(figsize=(14, 6))
    if not hist_bs.empty and 'year' in hist_bs.columns and 'ppe_net' in hist_bs.columns:
        hist_years = sorted(hist_bs['year'].dropna().astype(int))
        hist_values = [pd.to_numeric(hist_bs[hist_bs['year']==y]['ppe_net'].iloc[0], errors='coerce') for y in hist_years]
        hist_values = [float(v) if pd.notna(v) else 0 for v in hist_values]
        ax.plot(hist_years, hist_values, 'o-', color='blue', linewidth=2, markersize=8, label='History')
    ax.plot(forecast_years, [float(v) if pd.notna(v) else 0 for v in ppe_vals], '^-', color='red', linewidth=2, markersize=8, label='Forecast')
    ax.set_xlabel('Year'); ax.set_ylabel('PP&E ($ millions)'); ax.set_title('PP&E: History vs Forecast'); ax.legend(); ax.grid(True, alpha=0.3)
    if hist_bs is not None and not hist_bs.empty:
        ax.axvline(x=hist_bs['year'].max() + 0.5, color='gray', linestyle='--', alpha=0.5)
    plt.tight_layout(); plt.show()
    if not capex_breakdown.empty and 'metric' in capex_breakdown.columns:
        capex_metrics = capex_breakdown.set_index('metric')
        growth = [pd.to_numeric(capex_metrics.loc['GrowthCapex', str(y)], errors='coerce') if 'GrowthCapex' in capex_metrics.index else float('nan') for y in forecast_years]
        maint = [pd.to_numeric(capex_metrics.loc['MaintCapex', str(y)], errors='coerce') if 'MaintCapex' in capex_metrics.index else float('nan') for y in forecast_years]
        print(" 📊 CapEx составляющие (первые 3 года):")
        for y, g, m in zip(forecast_years[:3], growth[:3], maint[:3]):
            print(f"  {y}: Maint=${float(m) if pd.notna(m) else 0:,.0f}, Growth=${float(g) if pd.notna(g) else 0:,.0f}")


In [ ]:
# Анализ Depreciation
display(Markdown("#### 💰 Depreciation (Амортизация)"))

print("📋 Метод: Depreciation является компонентом PP&E corkscrew")

is_forecast = load_model_forecast_from_db('IS', canonical=True)
bs_forecast = load_model_forecast_from_db('BS', canonical=True)
hist_is = load_history_from_db('IS')
if is_forecast.empty or bs_forecast.empty:
    print("⚠️ Прогноз IS/BS недоступен")
else:
    is_metrics = is_forecast.set_index('metric') if 'metric' in is_forecast.columns else is_forecast
    bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns else bs_forecast
    forecast_years = sorted([int(c) for c in is_forecast.columns if str(c).isdigit()])
    dep_vals = [pd.to_numeric(is_metrics.loc['depreciation', str(y)], errors='coerce') if 'depreciation' in is_metrics.index else float('nan') for y in forecast_years]
    ppe_vals = [pd.to_numeric(bs_metrics.loc['ppe_net', str(y)], errors='coerce') if 'ppe_net' in bs_metrics.index else float('nan') for y in forecast_years]
    fig, ax = plt.subplots(figsize=(14, 6))
    hist_dep = {}
    if not hist_is.empty and 'year' in hist_is.columns:
        for _, row in hist_is.iterrows():
            year = row.get('year')
            val = pd.to_numeric(row.get('depreciation', row.get('dep')), errors='coerce')
            if pd.notna(year) and pd.notna(val):
                hist_dep[int(year)] = float(val)
    if hist_dep:
        hist_years = sorted(hist_dep.keys())
        ax.plot(hist_years, [hist_dep[y] for y in hist_years], 'o-', color='blue', linewidth=2, markersize=8, label='History')
    dep_plot = [float(v) if pd.notna(v) else 0 for v in dep_vals]
    ax.plot(forecast_years, dep_plot, '^-', color='red', linewidth=2, markersize=8, label='Forecast')
    ax.set_xlabel('Year'); ax.set_ylabel('Depreciation ($ millions)'); ax.set_title('Depreciation: History vs Forecast'); ax.legend(); ax.grid(True, alpha=0.3)
    if hist_dep:
        ax.axvline(x=max(hist_dep.keys()) + 0.5, color='gray', linestyle='--', alpha=0.5)
    plt.tight_layout(); plt.show()
    if ppe_vals:
        print(" 📊 Depreciation / PP&E ratio (первые 3 года):")
        for y, dep_val, ppe_val in zip(forecast_years[:3], dep_vals[:3], ppe_vals[:3]):
            if pd.notna(dep_val) and pd.notna(ppe_val) and ppe_val != 0:
                print(f"  {y}: {float(dep_val)/float(ppe_val):.2%}")
            else:
                print(f"  {y}: N/A")


In [ ]:
# Анализ Interest Expense
display(Markdown("#### 💸 Interest Expense (Процентные расходы)"))

debt_cfg = proj_yaml.get("model", {}).get("standard", {}).get("debt", {})
print(f"📋 Interest treatment: {debt_cfg.get('interest_treatment', 'separate_line')}")

is_forecast = load_model_forecast_from_db('IS', canonical=True)
bs_forecast = load_model_forecast_from_db('BS', canonical=True)
if is_forecast.empty:
    print("⚠️ Прогноз IS отсутствует")
else:
    is_metrics = is_forecast.set_index('metric') if 'metric' in is_forecast.columns else is_forecast
    forecast_years = sorted([int(c) for c in is_forecast.columns if str(c).isdigit()])
    interest_vals = [pd.to_numeric(is_metrics.loc['interest_expense', str(y)], errors='coerce') if 'interest_expense' in is_metrics.index else float('nan') for y in forecast_years]
    lease_interest_vals = [pd.to_numeric(is_metrics.loc['lease_interest', str(y)], errors='coerce') if 'lease_interest' in is_metrics.index else 0 for y in forecast_years]
    hist_is = load_history_from_db('IS')
    interest_hist = {}
    if not hist_is.empty and 'year' in hist_is.columns:
        for _, row in hist_is.iterrows():
            year = row.get('year')
            val = pd.to_numeric(row.get('interest_expense', row.get('interest')), errors='coerce')
            if pd.notna(year) and pd.notna(val):
                interest_hist[int(year)] = float(val)
    fig, ax = plt.subplots(figsize=(14, 6))
    if interest_hist:
        hist_years = sorted(interest_hist.keys())
        ax.plot(hist_years, [interest_hist[y] for y in hist_years], 'o-', color='blue', linewidth=2, markersize=8, label='History')
    interest_plot = [float(v) if pd.notna(v) else 0 for v in interest_vals]
    ax.plot(forecast_years, interest_plot, '^-', color='red', linewidth=2, markersize=8, label='Interest Expense Forecast')
    if any(v != 0 for v in lease_interest_vals):
        ax.plot(forecast_years, [float(v) if pd.notna(v) else 0 for v in lease_interest_vals], 's--', color='green', linewidth=2, markersize=6, label='Lease Interest')
    ax.set_xlabel('Year'); ax.set_ylabel('Interest ($ millions)'); ax.set_title('Interest Expense: History vs Forecast'); ax.legend(); ax.grid(True, alpha=0.3)
    if interest_hist:
        ax.axvline(x=max(interest_hist.keys()) + 0.5, color='gray', linestyle='--', alpha=0.5)
    plt.tight_layout(); plt.show()
    print(" 📊 Прогноз Interest Expense (первые 3 года):")
    for y, interest_val, lease_val in zip(forecast_years[:3], interest_vals[:3], lease_interest_vals[:3]):
        print(f"  {y}: Interest=${float(interest_val) if pd.notna(interest_val) else float('nan'):,.0f}, Lease=${float(lease_val) if pd.notna(lease_val) else 0:,.0f}")


In [ ]:
# Анализ Tax
display(Markdown("#### 📊 Tax (Налог на прибыль)"))

taxes_cfg = proj_yaml.get("model", {}).get("standard", {}).get("taxes", {})
margins_cfg = proj_yaml.get("model", {}).get("standard", {}).get("margins", {})
print(f"📋 Политика: {taxes_cfg.get('policy', 'statutory_with_floor')}")
print(f"  Statutory rate: {margins_cfg.get('tax_rate_statutory', 0.21):.1%}")

is_forecast = load_model_forecast_from_db('IS', canonical=True)
if is_forecast.empty:
    print("⚠️ Прогноз IS отсутствует")
else:
    is_metrics = is_forecast.set_index('metric') if 'metric' in is_forecast.columns else is_forecast
    forecast_years = sorted([int(c) for c in is_forecast.columns if str(c).isdigit()])
    tax_vals = [pd.to_numeric(is_metrics.loc['tax', str(y)], errors='coerce') if 'tax' in is_metrics.index else float('nan') for y in forecast_years]
    ebt_vals = [pd.to_numeric(is_metrics.loc['ebt', str(y)], errors='coerce') if 'ebt' in is_metrics.index else float('nan') for y in forecast_years]
    hist_is = load_history_from_db('IS')
    tax_hist, ebt_hist = {}, {}
    if not hist_is.empty and 'year' in hist_is.columns:
        for _, row in hist_is.iterrows():
            year = row.get('year')
            tax_val = pd.to_numeric(row.get('tax', row.get('tax_expense')), errors='coerce')
            ebt_val = pd.to_numeric(row.get('ebt'), errors='coerce')
            if pd.notna(year) and pd.notna(tax_val):
                tax_hist[int(year)] = float(tax_val)
            if pd.notna(year) and pd.notna(ebt_val):
                ebt_hist[int(year)] = float(ebt_val)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    if tax_hist:
        hist_years = sorted(tax_hist.keys())
        axes[0].plot(hist_years, [tax_hist[y] for y in hist_years], 'o-', color='blue', linewidth=2, markersize=8, label='History')
    axes[0].plot(forecast_years, [float(v) if pd.notna(v) else 0 for v in tax_vals], '^-', color='red', linewidth=2, markersize=8, label='Forecast')
    axes[0].set_title('Tax Expense: History vs Forecast'); axes[0].set_xlabel('Year'); axes[0].set_ylabel('$ millions'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    statutory = margins_cfg.get('tax_rate_statutory', 0.21) * 100
    effective_rates_hist = []
    if tax_hist and ebt_hist:
        for y in sorted(tax_hist.keys()):
            if y in ebt_hist and ebt_hist[y] > 0:
                effective_rates_hist.append((y, tax_hist[y]/ebt_hist[y]*100))
    if effective_rates_hist:
        years_hist, rates_hist = zip(*effective_rates_hist)
        axes[1].plot(years_hist, rates_hist, 'o-', color='blue', linewidth=2, markersize=8, label='Effective (history)')
    effective_rates_forecast = []
    for y, tax_val, ebt_val in zip(forecast_years, tax_vals, ebt_vals):
        if pd.notna(tax_val) and pd.notna(ebt_val) and ebt_val != 0:
            effective_rates_forecast.append((y, float(tax_val)/float(ebt_val)*100))
    if effective_rates_forecast:
        years_fc, rates_fc = zip(*effective_rates_forecast)
        axes[1].plot(years_fc, rates_fc, '^-', color='red', linewidth=2, markersize=8, label='Effective (forecast)')
    axes[1].axhline(y=statutory, color='green', linestyle='--', linewidth=2, label=f'Statutory {statutory:.1f}%')
    axes[1].set_title('Effective Tax Rate'); axes[1].set_xlabel('Year'); axes[1].set_ylabel('%'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    print(" 📊 Прогноз Tax (первые 3 года):")
    for y, tax_val, ebt_val in zip(forecast_years[:3], tax_vals[:3], ebt_vals[:3]):
        if pd.notna(tax_val):
            if pd.notna(ebt_val) and ebt_val != 0:
                print(f"  {y}: Tax=${float(tax_val):,.0f}, Effective rate={float(tax_val)/float(ebt_val):.1%}")
            else:
                print(f"  {y}: Tax=${float(tax_val):,.0f} (EBT<=0)")


### 💼 Детальный анализ Working Capital, Cash и Debt/RC



In [ ]:
# Детальный анализ Working Capital
display(Markdown("#### 💼 Working Capital (Оборотный капитал)"))

wc_cfg = proj_yaml.get("model", {}).get("standard", {}).get("wc", {})
print(f"📋 Метод: {wc_cfg.get('method', 'days')} method")

wc_days = load_model_table('WC_DAYS', csv_filename='outputs/model/wc_days.csv')
if wc_days.empty:
    print("⚠️ Данные WC недоступны")
else:
    display(wc_days.head())


In [ ]:
# Анализ Cash
display(Markdown("#### 💵 Cash (Денежные средства)") )

rc_cfg = proj_yaml.get("model", {}).get("standard", {}).get("debt", {}).get('rc', {})
min_cash = rc_cfg.get('min_cash', 0.0)
print(f"📋 Минимальный уровень Cash: ${min_cash:,.0f}")

bs_forecast = load_model_forecast_from_db('BS', canonical=True)
cf_forecast = load_model_forecast_from_db('CF', canonical=True)
if bs_forecast.empty or cf_forecast.empty:
    print("⚠️ Прогноз BS или CF отсутствует")
else:
    bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns else bs_forecast
    cf_metrics = cf_forecast.set_index('metric') if 'metric' in cf_forecast.columns else cf_forecast
    forecast_years = sorted([int(c) for c in bs_forecast.columns if str(c).isdigit()])
    cash_vals = [pd.to_numeric(bs_metrics.loc['cash', str(y)], errors='coerce') if 'cash' in bs_metrics.index else float('nan') for y in forecast_years]
    netchange_vals = [pd.to_numeric(cf_metrics.loc['net_change', str(y)], errors='coerce') if 'net_change' in cf_metrics.index else float('nan') for y in forecast_years]
    hist_bs = load_history_from_db('BS')
    cash_hist = {}
    if not hist_bs.empty and 'year' in hist_bs.columns and 'cash' in hist_bs.columns:
        for _, row in hist_bs.iterrows():
            year = row.get('year')
            val = pd.to_numeric(row.get('cash'), errors='coerce')
            if pd.notna(year) and pd.notna(val):
                cash_hist[int(year)] = float(val)
    fig, ax = plt.subplots(figsize=(14, 6))
    if cash_hist:
        hist_years = sorted(cash_hist.keys())
        ax.plot(hist_years, [cash_hist[y] for y in hist_years], 'o-', color='blue', linewidth=2, markersize=8, label='History')
    cash_plot = [float(v) if pd.notna(v) else 0 for v in cash_vals]
    ax.plot(forecast_years, cash_plot, '^-', color='red', linewidth=2, markersize=8, label='Forecast')
    ax.axhline(y=min_cash, color='orange', linestyle='--', linewidth=2, label=f'Min Cash (${min_cash:,.0f})')
    ax.set_xlabel('Year'); ax.set_ylabel('Cash ($ millions)'); ax.set_title('Cash Balance: History vs Forecast'); ax.legend(); ax.grid(True, alpha=0.3)
    if cash_hist:
        ax.axvline(x=max(cash_hist.keys()) + 0.5, color='gray', linestyle='--', alpha=0.5)
    plt.tight_layout(); plt.show()
    print(" 📊 Cash Bridge (первые 3 года):")
    prev_cash = cash_hist.get(max(cash_hist.keys())) if cash_hist else 0.0
    for y, cash_val, net_val in zip(forecast_years[:3], cash_vals[:3], netchange_vals[:3]):
        if pd.notna(cash_val) and pd.notna(net_val):
            calc = prev_cash + float(net_val)
            status = '✅' if abs(calc - float(cash_val)) < 0.01 else '⚠️'
            min_status = ' (min_cash ⚠️)' if float(cash_val) < min_cash else ''
            print(f"  {y}: Cash=${float(cash_val):,.0f}, NetChange=${float(net_val):,.0f}, Cash_calc=${calc:,.0f} {status}{min_status}")
            prev_cash = float(cash_val)
        else:
            print(f"  {y}: данных недостаточно")


### 💳 Детальный анализ Debt/RC


In [ ]:
display(Markdown("#### 💳 Сводка долговых инструментов и RC"))

debt_candidates = [
    ("DEBT_SCHEDULE", "outputs/model/debt_schedule.csv"),
    ("DEBT_WATERFALL", "outputs/model/debt_waterfall.csv")
]

debt_df = pd.DataFrame()
for table_name, csv_name in debt_candidates:
    debt_df = load_model_table(table_name, canonical=False, csv_filename=csv_name)
    if not debt_df.empty:
        print(f"Источник данных: {table_name}")
        break

if debt_df.empty:
    print("⚠️ Данные долгового расписания недоступны в витрине и CSV")
else:
    df = debt_df.copy()
    instrument_col = next((col for col in ['instrument', 'name', 'debt_instrument', 'facility', 'tranche'] if col in df.columns), None)
    year_cols = [col for col in df.columns if str(col).isdigit()]
    if instrument_col and year_cols:
        df[year_cols] = df[year_cols].apply(pd.to_numeric, errors='coerce')
        grouped = df[[instrument_col] + year_cols].groupby(instrument_col).sum(min_count=1).reset_index()
        def _latest_year(row):
            years = [int(col) for col in year_cols if pd.notna(row[col])]
            return max(years) if years else None
        grouped['Последний год'] = grouped.apply(_latest_year, axis=1)
        def _latest_balance(row):
            year = row['Последний год']
            return row[str(year)] if year is not None and str(year) in row else pd.NA
        grouped['Баланс на последний год'] = grouped.apply(_latest_balance, axis=1)
        display(grouped.rename(columns={instrument_col: 'Инструмент'})[['Инструмент', 'Последний год', 'Баланс на последний год']])
    else:
        display(df.head())

rc_activity = load_model_table('RC_ACTIVITY', canonical=False, csv_filename='outputs/model/rc_activity.csv')
if not rc_activity.empty:
    print(" 📊 RC Activity (первые 5 строк):")
    display(rc_activity.head())
else:
    print(" ⚠️ RC Activity отсутствует")

rc_log = load_model_table('RC_CONVERGENCE_LOG', canonical=False, csv_filename='outputs/model/rc_convergence_log.csv')
if not rc_log.empty:
    print(f"\n📊 RC Convergence Log: {len(rc_log)} записей, последние 5:")
    display(rc_log.tail(5))
    if {'iteration', 'converged'}.issubset(rc_log.columns):
        last_row = rc_log.sort_values(by='iteration').tail(1)
        converged = bool(last_row['converged'].iloc[0])
        iteration = int(last_row['iteration'].iloc[0]) if pd.notna(last_row['iteration'].iloc[0]) else 'N/A'
        print(f"Статус последней итерации: {'✅ Сошлось' if converged else '❌ Не сошлось'} (итерация {iteration})")
else:
    print("⚠️ RC Convergence Log отсутствует")


In [ ]:
# Детальный анализ Debt и Revolving Credit
display(Markdown("#### 💳 Debt и RC (Долг и револьверный кредит)"))

debt_cfg = proj_yaml.get("model", {}).get("standard", {}).get("debt", {})
rc_cfg = debt_cfg.get("rc", {})
refinancing_cfg = debt_cfg.get("refinancing", {})

print(f"📋 Конфигурация Debt/RC:")
print(f"  RC enabled: {rc_cfg.get('enable', False)}")
if rc_cfg.get('enable', False):
    print(f"  RC limit: ${rc_cfg.get('limit', 0):,.0f}")
    print(f"  Min cash: ${rc_cfg.get('min_cash', 0):,.0f}")
print(f"  Refinancing enabled: {refinancing_cfg.get('enable', False)}")
if refinancing_cfg.get('enable', False):
    print(f"  Default tenor: {refinancing_cfg.get('default_tenor_years', 5)} лет")
    print(f"  Bonds refinance tenor: {refinancing_cfg.get('bonds_refinance_tenor_years', 3)} лет")
print(f"  Iterative convergence: max_iter={debt_cfg.get('iter_max', 50)}, tol={debt_cfg.get('tol', 1e-6)}")

# Debt waterfall
debt_waterfall_df = load_model_table('DEBT_WATERFALL', csv_filename='outputs/model/debt_waterfall.csv')
if not debt_waterfall_df.empty:
    print(f"\n✅ Debt Waterfall загружен ({len(debt_waterfall_df)} строк)")
    if 'instrument' in debt_waterfall_df.columns:
        instruments = debt_waterfall_df['instrument'].unique()
        print(f"📊 Инструменты долга (первые 5):")
        for inst in instruments[:5]:
            print(f"  - {inst}")
else:
    print("⚠️ Debt Waterfall отсутствует (пока доступен только legacy CSV)")

# Прогнозный баланс
bs_forecast = load_model_forecast_from_db('BS', canonical=True)
bs_metrics = bs_forecast.set_index('metric') if ('metric' in bs_forecast.columns and not bs_forecast.empty) else pd.DataFrame()
forecast_years = sorted([int(c) for c in bs_forecast.columns if str(c).isdigit()]) if not bs_forecast.empty else []

if not bs_metrics.empty and forecast_years:
    debt_row = bs_metrics.loc['debt'] if 'debt' in bs_metrics.index else None
    cash_row = bs_metrics.loc['cash'] if 'cash' in bs_metrics.index else None
    if debt_row is not None:
        print(f"\n📊 Общий долг по годам (первые 3):")
        for y in forecast_years[:3]:
            debt_val = pd.to_numeric(debt_row.get(str(y)), errors='coerce')
            if pd.notna(debt_val):
                print(f"  {y}: ${float(debt_val):,.0f}")
        fig, ax = plt.subplots(figsize=(14, 6))
        debt_values = [pd.to_numeric(debt_row.get(str(y)), errors='coerce') for y in forecast_years]
        debt_values = [float(v) if pd.notna(v) else 0 for v in debt_values]
        cash_values = []
        if cash_row is not None:
            cash_values = [pd.to_numeric(cash_row.get(str(y)), errors='coerce') for y in forecast_years]
            cash_values = [float(v) if pd.notna(v) else 0 for v in cash_values]
        net_debt_values = [d - c if cash_values else d for d, c in zip(debt_values, cash_values or [0]*len(debt_values))]
        ax.plot(forecast_years, debt_values, 'o-', label='Total Debt', linewidth=2, markersize=8)
        if cash_values:
            ax.plot(forecast_years, net_debt_values, 's-', label='Net Debt', linewidth=2, markersize=8)
        ax.set_xlabel('Year')
        ax.set_ylabel('Debt ($ millions)')
        ax.set_title('Debt Balance: Total vs Net')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='black', linewidth=0.8)
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ Прогнозный баланс недоступен")

# RC convergence log
rc_log_df = load_model_table('RC_CONVERGENCE_LOG', csv_filename='outputs/model/rc_convergence_log.csv')
if not rc_log_df.empty:
    print(f"\n📊 RC Convergence: {len(rc_log_df)} итераций")
    if 'iteration' in rc_log_df.columns:
        print(f"  Последняя итерация: {int(rc_log_df['iteration'].max())}")
    if 'converged' in rc_log_df.columns:
        converged = bool(rc_log_df['converged'].iloc[-1])
        print(f"  Статус: {'✅ Сошлось' if converged else '❌ Не сошлось'}")
    if {'cash_after_rc', 'min_cash'}.issubset(rc_log_df.columns):
        violations = rc_log_df[rc_log_df['cash_after_rc'] < rc_log_df['min_cash']]
        if not violations.empty:
            print(f"  ⚠️ Нарушение min_cash в {len(violations)} итерациях")
        else:
            print("  ✅ min_cash соблюден во всех итерациях")
else:
    print("⚠️ RC Convergence Log отсутствует")

# Проверяем, какая версия движка подхвачена в текущей сессии ноутбука
import inspect
import engine.model3.core as core_module

print("\n🔍 Проверка версии модуля engine.model3.core")
print(f"  Файл: {core_module.__file__}")
has_dividends_fix = 'self.re[y] = self.re[prev_year] + ni2 - dividends2' in inspect.getsource(core_module.ThreeStatementModel._solve_year)
print(f"  Используется фикс с dividends2: {'✅ Да' if has_dividends_fix else '❌ Нет'}")


### 📈 Анализ Retained Earnings


In [ ]:
# Анализ Retained Earnings
display(Markdown("#### 📈 Retained Earnings (Нераспределенная прибыль)"))

print("📋 Метод: Retained Earnings Corkscrew")
print("  Формула: RE_t = RE_{t-1} + Net Income - Dividends")

bs_forecast = load_model_forecast_from_db('BS', canonical=True)
is_forecast = load_model_forecast_from_db('IS', canonical=True)
hist_bs = load_history_from_db('BS')

if bs_forecast.empty or is_forecast.empty:
    print("⚠️ Прогнозные данные BS/IS недоступны")
else:
    bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns else bs_forecast
    is_metrics = is_forecast.set_index('metric') if 'metric' in is_forecast.columns else is_forecast
    if 'retained_earnings' not in bs_metrics.index or 'net_income' not in is_metrics.index:
        print("⚠️ Метрики Retained Earnings или Net Income отсутствуют в прогнозе")
    else:
        forecast_years = sorted([int(c) for c in bs_forecast.columns if str(c).isdigit()])
        re_row = bs_metrics.loc['retained_earnings']
        ni_row = is_metrics.loc['net_income']
        re_opening = None
        if not hist_bs.empty and 'year' in hist_bs.columns:
            re_hist_col = None
            for col in hist_bs.columns:
                if 'retained' in str(col).lower():
                    re_hist_col = col
                    break
            if re_hist_col:
                latest_year = hist_bs['year'].max()
                re_val = hist_bs.loc[hist_bs['year'] == latest_year, re_hist_col]
                if not re_val.empty:
                    re_opening = float(pd.to_numeric(re_val.iloc[0], errors='coerce'))
        print(f"\n📊 Проверка Retained Earnings Corkscrew (первые 3 года):")
        prev_re = re_opening if re_opening is not None else 0.0
        for y in forecast_years[:3]:
            re_val = pd.to_numeric(re_row.get(str(y)), errors='coerce')
            ni_val = pd.to_numeric(ni_row.get(str(y)), errors='coerce')
            dividends = 0.0
            if pd.notna(re_val) and pd.notna(ni_val):
                calc_re = prev_re + float(ni_val) - dividends
                diff = abs(float(re_val) - calc_re)
                if diff < 0.01:
                    print(f"  {y}: ✅ RE=${float(re_val):,.0f} (RE_prev={prev_re:,.0f}, NI={float(ni_val):,.0f})")
                else:
                    print(f"  {y}: ⚠️ RE=${float(re_val):,.0f}, ожидалось {calc_re:,.0f}, diff=${diff:,.0f}")
                prev_re = float(re_val)
        fig, ax = plt.subplots(figsize=(14, 6))
        re_values = [float(pd.to_numeric(re_row.get(str(y)), errors='coerce')) if pd.notna(pd.to_numeric(re_row.get(str(y)), errors='coerce')) else 0 for y in forecast_years]
        ni_values = [float(pd.to_numeric(ni_row.get(str(y)), errors='coerce')) if pd.notna(pd.to_numeric(ni_row.get(str(y)), errors='coerce')) else 0 for y in forecast_years]
        ax.plot(forecast_years, re_values, 'o-', label='Retained Earnings', linewidth=2, markersize=8)
        ax2 = ax.twinx()
        ax2.plot(forecast_years, ni_values, 's--', color='green', linewidth=2, markersize=6, label='Net Income')
        ax.set_xlabel('Year')
        ax.set_ylabel('Retained Earnings ($ millions)')
        ax2.set_ylabel('Net Income ($ millions)', color='green')
        ax.set_title('Retained Earnings: Forecast')
        ax.legend(loc='upper left')
        ax2.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


### 💼 Детальный анализ Working Capital (WC)


In [ ]:
# Детальный анализ Working Capital - улучшенная версия
display(Markdown("#### 💼 Working Capital: Детальный анализ компонентов и изменений"))

wc_cfg = proj_yaml.get("model", {}).get("standard", {}).get("wc", {})
print(f"📋 Метод моделирования: {wc_cfg.get('method', 'days')} method")
print(f"  DSO: {wc_cfg.get('dso_days', 30)} дней")
print(f"  DIO: {wc_cfg.get('dio_days', 40)} дней")
print(f"  DPO: {wc_cfg.get('dpo_days', 45)} дней")
print(f"  Transform days: {wc_cfg.get('transform_days', 'none')}")

floors = wc_cfg.get("floors", {})
if floors:
    print(f"  Floors: AR={floors.get('ar_days', 5)}, INV={floors.get('inv_days', 10)}, AP={floors.get('ap_days', 5)}")

wc_days_df = load_model_table('WC_DAYS', csv_filename='outputs/model/wc_days.csv')
bs_forecast = load_model_forecast_from_db('BS', canonical=True)
is_forecast = load_model_forecast_from_db('IS', canonical=True)
cf_forecast = load_model_forecast_from_db('CF', canonical=True)

if wc_days_df.empty or bs_forecast.empty or is_forecast.empty:
    print("⚠️ Недостаточно данных WC в витрине — используем legacy CSV где возможно")

wc_metrics = wc_days_df.copy()
if 'metric' in wc_metrics.columns:
    wc_metrics = wc_metrics.set_index('metric')
bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns else bs_forecast
is_metrics = is_forecast.set_index('metric') if 'metric' in is_forecast.columns else is_forecast
cf_metrics = cf_forecast.set_index('metric') if 'metric' in cf_forecast.columns else cf_forecast

forecast_years = sorted([int(c) for c in wc_days_df.columns if str(c).isdigit()]) if not wc_days_df.empty else None

wc_history = load_model_table('WC_HISTORY_DAYS', csv_filename='outputs/model/wc_history_days.csv')
if not wc_history.empty and 'year' in wc_history.columns:
    hist_stats = []
    for metric in ['dso', 'dio', 'dpo']:
        if metric in wc_history.columns:
            values = pd.to_numeric(wc_history[metric], errors='coerce').dropna()
            if not values.empty:
                hist_stats.append((metric.upper(), values.mean(), values.min(), values.max()))
    if hist_stats:
        print(" 📊 Исторические параметры WC Days:")
        for metric, mean_val, min_val, max_val in hist_stats:
            print(f"  {metric}: среднее={mean_val:.1f}, минимум={min_val:.1f}, максимум={max_val:.1f}")

if wc_metrics is not None and not wc_metrics.empty and forecast_years:
    revenue_row = is_metrics.loc['revenue'] if 'revenue' in is_metrics.index else None
    wc_delta_cf_row = cf_metrics.loc['wc_delta'] if 'wc_delta' in cf_metrics.index else None
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    def _plot_metric(ax, metric, label):
        if metric in wc_metrics.index:
            values = [pd.to_numeric(wc_metrics.loc[metric, str(y)], errors='coerce') for y in forecast_years]
            values = [float(v) if pd.notna(v) else 0 for v in values]
            ax.plot(forecast_years, values, 'o-', linewidth=2, markersize=6, label=label)

    ax1 = axes[0,0]
    _plot_metric(ax1, 'AR', 'Accounts Receivable')
    _plot_metric(ax1, 'AP', 'Accounts Payable')
    _plot_metric(ax1, 'INV', 'Inventory')
    _plot_metric(ax1, 'WC_balance', 'Total WC')
    ax1.set_title('Working Capital Balances')
    ax1.set_xlabel('Year'); ax1.set_ylabel('Balance ($ millions)'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = axes[0,1]
    if not wc_history.empty:
        hist_years = sorted(wc_history['year'].dropna().unique())
        for metric, color in [('dso', 'blue'), ('dio', 'green'), ('dpo', 'orange')]:
            if metric in wc_history.columns:
                ax2.plot(hist_years, pd.to_numeric(wc_history[metric], errors='coerce'), 'o-', color=color, label=metric.upper())
        if floors:
            ax2.axhline(y=floors.get('ar_days', 5), color='blue', linestyle='--', alpha=0.5)
            ax2.axhline(y=floors.get('inv_days', 10), color='green', linestyle='--', alpha=0.5)
            ax2.axhline(y=floors.get('ap_days', 5), color='orange', linestyle='--', alpha=0.5)
    ax2.set_title('WC Days History'); ax2.set_xlabel('Year'); ax2.set_ylabel('Days'); ax2.legend(); ax2.grid(True, alpha=0.3)

    ax3 = axes[1,0]
    if 'WC_delta' in wc_metrics.index and wc_delta_cf_row is not None:
        wc_delta_values = [pd.to_numeric(wc_metrics.loc['WC_delta', str(y)], errors='coerce') for y in forecast_years]
        cf_delta_values = [pd.to_numeric(wc_delta_cf_row.get(str(y)), errors='coerce') for y in forecast_years]
        wc_delta_values = [float(v) if pd.notna(v) else 0 for v in wc_delta_values]
        cf_delta_values = [float(v) if pd.notna(v) else 0 for v in cf_delta_values]
        colors = ['red' if v < 0 else 'green' for v in wc_delta_values]
        ax3.bar(forecast_years, wc_delta_values, color=colors, alpha=0.7, edgecolor='black', label='WC Δ (Model)')
        ax3.plot(forecast_years, cf_delta_values, 's--', color='black', label='WC Δ (CF)')
        ax3.legend(); ax3.grid(True, alpha=0.3, axis='y'); ax3.axhline(y=0, color='black', linewidth=0.8)
        ax3.set_title('Working Capital Delta'); ax3.set_xlabel('Year'); ax3.set_ylabel('$ millions')

    ax4 = axes[1,1]
    if revenue_row is not None and 'WC_balance' in wc_metrics.index:
        wc_values = [pd.to_numeric(wc_metrics.loc['WC_balance', str(y)], errors='coerce') for y in forecast_years]
        rev_values = [pd.to_numeric(revenue_row.get(str(y)), errors='coerce') for y in forecast_years]
        wc_pct = []
        for wc_val, rev_val in zip(wc_values, rev_values):
            if pd.notna(wc_val) and pd.notna(rev_val) and rev_val:
                wc_pct.append((float(wc_val)/float(rev_val))*100)
            else:
                wc_pct.append(float('nan'))
        ax4.plot(forecast_years, wc_pct, 'o-', linewidth=2, markersize=6, label='WC/Revenue %')
        ax4.set_xlabel('Year'); ax4.set_ylabel('%'); ax4.set_title('Working Capital as % of Revenue'); ax4.legend(); ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(" 📊 Детальная таблица Working Capital (первые 3 года):")
    rows = []
    for y in forecast_years[:3]:
        ar_val = pd.to_numeric(wc_metrics.loc['AR', str(y)], errors='coerce') if 'AR' in wc_metrics.index else float('nan')
        ap_val = pd.to_numeric(wc_metrics.loc['AP', str(y)], errors='coerce') if 'AP' in wc_metrics.index else float('nan')
        inv_val = pd.to_numeric(wc_metrics.loc['INV', str(y)], errors='coerce') if 'INV' in wc_metrics.index else float('nan')
        wc_val = pd.to_numeric(wc_metrics.loc['WC_balance', str(y)], errors='coerce') if 'WC_balance' in wc_metrics.index else float('nan')
        wc_delta_val = pd.to_numeric(wc_metrics.loc['WC_delta', str(y)], errors='coerce') if 'WC_delta' in wc_metrics.index else float('nan')
        rev_val = pd.to_numeric(revenue_row.get(str(y)), errors='coerce') if revenue_row is not None else float('nan')
        rows.append({
            'Year': y,
            'AR': _fmt_currency(ar_val),
            'AP': _fmt_currency(ap_val),
            'Inventory': _fmt_currency(inv_val),
            'WC': _fmt_currency(wc_val),
            'WC Δ': _fmt_currency(wc_delta_val),
            'WC/Revenue %': f"{(float(wc_val)/float(rev_val)*100):.1f}%" if pd.notna(wc_val) and pd.notna(rev_val) and rev_val else 'N/A'
        })
    display(pd.DataFrame(rows))
else:
    print("⚠️ Данные WC недоступны")


### 🏛️ Анализ Intangible Assets (Нематериальные активы)


In [ ]:
# Анализ Intangible Assets (Corkscrew метод)
display(Markdown("#### 🏛️ Intangible Assets (Нематериальные активы) - Corkscrew метод"))

print("📋 Метод: Corkscrew (аналогично PP&E)")
print("  Формула: Intangibles_t = Intangibles_{t-1} + Additions - Amortization")

intangibles_cfg = proj_yaml.get("model", {}).get("standard", {}).get("intangibles", {})
print(" 📋 Конфигурация:")
print(f"  Additions method: {intangibles_cfg.get('additions_method', 'pct_of_revenue')}")
print(f"  Amortization method: {intangibles_cfg.get('amortization_method', 'pct_of_balance')}")

bs_forecast = load_model_forecast_from_db('BS', canonical=True)
hist_bs = load_history_from_db('BS')

if bs_forecast.empty:
    print("⚠️ Прогноз BS недоступен")
else:
    bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns else bs_forecast
    intang_row = None
    for name in ['intangibles', 'intangible_assets', 'intangible']:
        if name in bs_metrics.index:
            intang_row = bs_metrics.loc[name]
            break
    if intang_row is None:
        print("⚠️ Intangible Assets отсутствуют в прогнозе")
    else:
        forecast_years = sorted([int(c) for c in bs_forecast.columns if str(c).isdigit()])
        hist_intang = {}
        if not hist_bs.empty and 'year' in hist_bs.columns:
            for _, row in hist_bs.iterrows():
                year = row.get('year')
                value = row.get('intangibles', row.get('intangible_assets'))
                value = pd.to_numeric(value, errors='coerce')
                if pd.notna(year) and pd.notna(value):
                    hist_intang[int(year)] = float(value)
        if hist_intang:
            hist_years = sorted(hist_intang.keys())
            print(" 📊 Исторические параметры Intangibles:")
            print(f"  Период: {hist_years[0]}-{hist_years[-1]}")
            print(f"  Среднее: ${np.mean(list(hist_intang.values())):,.0f}")
        fig, ax = plt.subplots(figsize=(14, 6))
        if hist_intang:
            ax.plot(hist_years, [hist_intang[y] for y in hist_years], 'o-', color='blue', linewidth=2, markersize=8, label='History')
        forecast_vals = [pd.to_numeric(intang_row.get(str(y)), errors='coerce') for y in forecast_years]
        forecast_vals = [float(v) if pd.notna(v) else 0 for v in forecast_vals]
        ax.plot(forecast_years, forecast_vals, '^-', color='red', linewidth=2, markersize=8, label='Forecast')
        ax.set_xlabel('Year'); ax.set_ylabel('Intangible Assets ($ millions)'); ax.set_title('Intangible Assets: History vs Forecast')
        ax.legend(); ax.grid(True, alpha=0.3)
        if hist_intang and forecast_years:
            ax.axvline(x=max(hist_years) + 0.5, color='gray', linestyle='--', alpha=0.5)
        plt.tight_layout(); plt.show()
        print(" 📊 Прогноз Intangibles (первые 3 года):")
        for y in forecast_years[:3]:
            print(f"  {y}: {_fmt_currency(pd.to_numeric(intang_row.get(str(y)), errors='coerce'))}")


### 💰 Анализ Finance Income и Finance Expense


In [ ]:
if '_fmt_currency' not in globals():
    def _fmt_currency(val):
        if val is None or pd.isna(val):
            return 'N/A'
        try:
            return f"${float(val):,.0f}"
        except Exception:
            return str(val)

display(Markdown("#### 💰 Finance Income vs Finance Expense — ключевые коэффициенты"))

is_forecast = load_model_forecast_from_db('IS', canonical=True)
bs_forecast = load_model_forecast_from_db('BS', canonical=True)
hist_bs = load_history_from_db('BS')

if is_forecast.empty or bs_forecast.empty:
    print("⚠️ Прогнозы IS/BS недоступны — пропускаем анализ финансовых доходов и расходов")
else:
    is_metrics = is_forecast.set_index('metric') if 'metric' in is_forecast.columns else is_forecast
    bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns else bs_forecast
    forecast_years = sorted(int(c) for c in bs_forecast.columns if str(c).isdigit())

    def _series(metrics, keys):
        for key in keys:
            if key in metrics.index:
                return [pd.to_numeric(metrics.loc[key, str(y)], errors='coerce') for y in forecast_years]
        return [pd.NA] * len(forecast_years)

    cash_vals = _series(bs_metrics, ['cash'])
    lt_debt_vals = _series(bs_metrics, ['debt', 'lt_debt'])
    st_debt_vals = _series(bs_metrics, ['st_debt'])
    fi_vals = _series(is_metrics, ['finance_income', 'interest_income'])
    fe_vals = _series(is_metrics, ['finance_expense', 'interest_expense'])

    total_debt_vals = []
    for lt, st in zip(lt_debt_vals, st_debt_vals):
        lt_val = float(lt) if pd.notna(lt) else 0.0
        st_val = float(st) if pd.notna(st) else 0.0
        total_debt_vals.append(lt_val + st_val)

    prev_cash = None
    prev_debt = None
    if not hist_bs.empty:
        last_year = hist_bs['year'].max() if 'year' in hist_bs.columns else None
        if last_year is not None:
            if 'cash' in hist_bs.columns:
                val = hist_bs.loc[hist_bs['year'] == last_year, 'cash']
                if not val.empty:
                    prev_cash = float(pd.to_numeric(val.iloc[0], errors='coerce'))
            debt_hist = 0.0
            if 'debt' in hist_bs.columns:
                val = hist_bs.loc[hist_bs['year'] == last_year, 'debt']
                if not val.empty:
                    debt_hist += float(pd.to_numeric(val.iloc[0], errors='coerce'))
            if 'st_debt' in hist_bs.columns:
                val = hist_bs.loc[hist_bs['year'] == last_year, 'st_debt']
                if not val.empty:
                    debt_hist += float(pd.to_numeric(val.iloc[0], errors='coerce'))
            prev_debt = debt_hist if debt_hist else prev_debt

    rows = []
    for idx, year in enumerate(forecast_years):
        cash = float(cash_vals[idx]) if pd.notna(cash_vals[idx]) else None
        debt = float(total_debt_vals[idx]) if pd.notna(total_debt_vals[idx]) else None
        fi = float(fi_vals[idx]) if pd.notna(fi_vals[idx]) else None
        fe = float(fe_vals[idx]) if pd.notna(fe_vals[idx]) else None
        avg_cash = ((prev_cash if prev_cash is not None else cash) + cash) / 2 if cash is not None else None
        avg_debt = ((prev_debt if prev_debt is not None else debt) + debt) / 2 if debt is not None else None
        rows.append({
            'Year': year,
            'Finance Income': fi,
            'Avg Cash': avg_cash,
            'Income Yield %': (fi / avg_cash * 100) if fi is not None and avg_cash else None,
            'Finance Expense': fe,
            'Avg Debt': avg_debt,
            'Expense Rate %': (fe / avg_debt * 100) if fe is not None and avg_debt else None,
            'Net Finance': (fi or 0) - (fe or 0)
        })
        prev_cash = cash if cash is not None else prev_cash
        prev_debt = debt if debt is not None else prev_debt

    summary_df = pd.DataFrame(rows)
    if summary_df.empty:
        print("⚠️ Нет данных для анализа финансовых доходов и расходов")
    else:
        formatted = summary_df.copy()
        for col in ['Finance Income', 'Avg Cash', 'Finance Expense', 'Avg Debt', 'Net Finance']:
            formatted[col] = formatted[col].apply(lambda v: _fmt_currency(v) if pd.notna(v) else 'N/A')
        for col in ['Income Yield %', 'Expense Rate %']:
            formatted[col] = formatted[col].apply(lambda v: f"{v:.2f}%" if v is not None and pd.notna(v) else 'N/A')
        display(formatted)
        if summary_df[['Income Yield %','Expense Rate %']].notna().any().any():
            fig, ax = plt.subplots(figsize=(10, 4))
            if summary_df['Income Yield %'].notna().any():
                ax.plot(summary_df['Year'], summary_df['Income Yield %'], 'o-', label='Yield на денежные средства')
            if summary_df['Expense Rate %'].notna().any():
                ax.plot(summary_df['Year'], summary_df['Expense Rate %'], 's--', label='Стоимость долга')
            ax.set_ylabel('%'); ax.set_xlabel('Year'); ax.set_title('Ставки по Finance Income / Expense'); ax.grid(True, alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()


### 🏢 Анализ лизинга (Operating & Finance Leases)


In [ ]:
# Анализ лизинга
display(Markdown("#### 🏢 Лизинг (Operating & Finance Leases)"))

leases_cfg = proj_yaml.get("model", {}).get("standard", {}).get("leases", {})
leases_enabled = leases_cfg.get("enabled", False)

print("📋 Настройки лизинга:")
print(f"  Включен: {leases_enabled}")
if leases_enabled:
    print(f"  Ставка дисконтирования: {leases_cfg.get('default_discount_rate', 0.05):.2%}")
    print(f"  Частота платежей: {leases_cfg.get('default_payment_frequency', 'annual')}")

lease_schedule = load_company_csv('data/leases/lease_schedule.csv')
if lease_schedule.empty:
    lease_schedule = load_company_csv('data/operational/lease_schedule.csv')

if not lease_schedule.empty:
    print(f"✅ Найдено {len(lease_schedule)} записей в расписании лизинга")
    display(lease_schedule.head())
else:
    print("⚠️ Расписание лизинга не найдено (legacy CSV)")

bs_forecast = load_model_forecast_from_db('BS', canonical=True)
cf_forecast = load_model_forecast_from_db('CF', canonical=True)

if leases_enabled and (bs_forecast.empty or cf_forecast.empty):
    print("⚠️ Прогнозы BS/CF отсутствуют — анализ лизинга ограничен")

bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns and not bs_forecast.empty else pd.DataFrame()
cf_metrics = cf_forecast.set_index('metric') if 'metric' in cf_forecast.columns and not cf_forecast.empty else pd.DataFrame()
forecast_years = sorted([int(c) for c in bs_forecast.columns if str(c).isdigit()]) if not bs_forecast.empty else []

if not bs_metrics.empty and forecast_years:
    lease_liab = bs_metrics.loc['lease_liability'] if 'lease_liability' in bs_metrics.index else None
    rou_asset = bs_metrics.loc['rou_asset'] if 'rou_asset' in bs_metrics.index else None
    if lease_liab is not None:
        fig, ax = plt.subplots(figsize=(14, 6))
        lease_vals = [pd.to_numeric(lease_liab.get(str(y)), errors='coerce') for y in forecast_years]
        lease_vals = [float(v) if pd.notna(v) else 0 for v in lease_vals]
        ax.plot(forecast_years, lease_vals, 'o-', linewidth=2, markersize=8, label='Lease Liability')
        if rou_asset is not None:
            rou_vals = [pd.to_numeric(rou_asset.get(str(y)), errors='coerce') for y in forecast_years]
            rou_vals = [float(v) if pd.notna(v) else 0 for v in rou_vals]
            ax.plot(forecast_years, rou_vals, 's--', linewidth=2, markersize=6, label='ROU Asset')
        ax.set_title('Lease Liability & ROU Asset'); ax.set_xlabel('Year'); ax.set_ylabel('$ millions'); ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

if not cf_metrics.empty and forecast_years:
    lease_cff = cf_metrics.loc['lease_payments_cff'] if 'lease_payments_cff' in cf_metrics.index else None
    lease_interest = cf_metrics.loc['lease_interest_cfo'] if 'lease_interest_cfo' in cf_metrics.index else None
    if lease_cff is not None or lease_interest is not None:
        fig, ax = plt.subplots(figsize=(14, 6))
        if lease_cff is not None:
            cff_vals = [pd.to_numeric(lease_cff.get(str(y)), errors='coerce') for y in forecast_years]
            cff_vals = [float(v) if pd.notna(v) else 0 for v in cff_vals]
            ax.plot(forecast_years, cff_vals, 'o-', linewidth=2, markersize=8, label='Lease Principal (CFF)')
        if lease_interest is not None:
            interest_vals = [pd.to_numeric(lease_interest.get(str(y)), errors='coerce') for y in forecast_years]
            interest_vals = [float(v) if pd.notna(v) else 0 for v in interest_vals]
            ax.plot(forecast_years, interest_vals, 's--', linewidth=2, markersize=6, label='Lease Interest (CFO)')
        ax.set_title('Lease Payments & Interest'); ax.set_xlabel('Year'); ax.set_ylabel('$ millions'); ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

print(" 📊 Итоговая сводка лизинга (первые 3 года):")
rows = []
for y in forecast_years[:3]:
    liab = pd.to_numeric(bs_metrics.loc['lease_liability', str(y)], errors='coerce') if 'lease_liability' in bs_metrics.index else None
    rou = pd.to_numeric(bs_metrics.loc['rou_asset', str(y)], errors='coerce') if 'rou_asset' in bs_metrics.index else None
    cff = pd.to_numeric(cf_metrics.loc['lease_payments_cff', str(y)], errors='coerce') if 'lease_payments_cff' in cf_metrics.index else None
    interest = pd.to_numeric(cf_metrics.loc['lease_interest_cfo', str(y)], errors='coerce') if 'lease_interest_cfo' in cf_metrics.index else None
    rows.append({
        'Year': y,
        'Lease Liability': _fmt_currency(liab),
        'ROU Asset': _fmt_currency(rou),
        'Lease Principal (CFF)': _fmt_currency(cff),
        'Lease Interest (CFO)': _fmt_currency(interest)
    })
if rows:
    display(pd.DataFrame(rows))
else:
    print("⚠️ Нет данных по лизингу для отображения")


In [ ]:
# Анализ Finance Income и Finance Expense
display(Markdown("#### 💰 Finance Income & Expense"))

finance_cfg = proj_yaml.get("model", {}).get("standard", {}).get("finance", {})
print(f"📋 Finance income enabled: {finance_cfg.get('income', {}).get('enabled', True)}")
print(f"📋 Finance expense addon: {finance_cfg.get('expense', {}).get('addon_to_interest_pct', 0.0)}%")

hist_is = load_history_from_db('IS')
hist_bs = load_history_from_db('BS')
is_forecast = load_model_forecast_from_db('IS', canonical=True)
bs_forecast = load_model_forecast_from_db('BS', canonical=True)

if is_forecast.empty or bs_forecast.empty:
    print("⚠️ Прогноз IS/BS отсутствует")
else:
    is_metrics = is_forecast.set_index('metric') if 'metric' in is_forecast.columns else is_forecast
    bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns else bs_forecast
    forecast_years = sorted([int(c) for c in is_forecast.columns if str(c).isdigit()])
    finance_income_vals = [pd.to_numeric(is_metrics.loc['finance_income', str(y)], errors='coerce') if 'finance_income' in is_metrics.index else float('nan') for y in forecast_years]
    finance_expense_vals = [pd.to_numeric(is_metrics.loc['finance_expense', str(y)], errors='coerce') if 'finance_expense' in is_metrics.index else float('nan') for y in forecast_years]
    interest_vals = [pd.to_numeric(is_metrics.loc['interest_expense', str(y)], errors='coerce') if 'interest_expense' in is_metrics.index else float('nan') for y in forecast_years]
    cash_vals = [pd.to_numeric(bs_metrics.loc['cash', str(y)], errors='coerce') if 'cash' in bs_metrics.index else float('nan') for y in forecast_years]
    hist_fi, hist_fe, hist_cash = {}, {}, {}
    if not hist_is.empty and 'year' in hist_is.columns:
        for _, row in hist_is.iterrows():
            year = row.get('year')
            if pd.notna(year):
                hist_fi[int(year)] = pd.to_numeric(row.get('finance_income'), errors='coerce')
                hist_fe[int(year)] = pd.to_numeric(row.get('finance_expense'), errors='coerce')
    if not hist_bs.empty and 'year' in hist_bs.columns:
        for _, row in hist_bs.iterrows():
            year = row.get('year')
            if pd.notna(year):
                hist_cash[int(year)] = pd.to_numeric(row.get('cash'), errors='coerce')
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    if hist_fi:
        years = sorted(y for y, v in hist_fi.items() if pd.notna(v))
        axes[0].plot(years, [hist_fi[y] for y in years], 'o-', color='green', linewidth=2, markersize=8, label='History')
    axes[0].plot(forecast_years, [float(v) if pd.notna(v) else 0 for v in finance_income_vals], '^-', color='green', linewidth=2, markersize=8, label='Forecast')
    axes[0].set_title('Finance Income'); axes[0].set_xlabel('Year'); axes[0].set_ylabel('$ millions'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    if hist_fe:
        years = sorted(y for y, v in hist_fe.items() if pd.notna(v))
        axes[1].plot(years, [hist_fe[y] for y in years], 'o-', color='red', linewidth=2, markersize=8, label='History')
    axes[1].plot(forecast_years, [float(v) if pd.notna(v) else 0 for v in finance_expense_vals], '^-', color='red', linewidth=2, markersize=8, label='Forecast')
    axes[1].plot(forecast_years, [float(v) if pd.notna(v) else 0 for v in interest_vals], 's--', color='orange', linewidth=2, markersize=6, label='Interest')
    axes[1].set_title('Finance Expense vs Interest'); axes[1].set_xlabel('Year'); axes[1].set_ylabel('$ millions'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    print(" 📊 Прогноз Finance Income / Expense (первые 3 года):")
    prev_cash = hist_cash.get(max(hist_cash.keys())) if hist_cash else 0.0
    addon = finance_cfg.get('expense', {}).get('addon_to_interest_pct', 0.0)
    for y, fi, fe, interest, cash_val in zip(forecast_years[:3], finance_income_vals[:3], finance_expense_vals[:3], interest_vals[:3], cash_vals[:3]):
        avg_cash = (prev_cash + float(cash_val)) / 2.0 if pd.notna(cash_val) else float('nan')
        calc_fi = avg_cash * (finance_cfg.get('income', {}).get('rate_pct', 0.0) / 100.0) if pd.notna(avg_cash) else float('nan')
        calc_fe = float(interest) * (1 + addon/100.0) if pd.notna(interest) else float('nan')
        print(f"  {y}: FI=${float(fi) if pd.notna(fi) else float('nan'):,.0f} (calc={calc_fi:,.0f}), FE=${float(fe) if pd.notna(fe) else float('nan'):,.0f} (calc={calc_fe:,.0f})")
        if pd.notna(cash_val):
            prev_cash = float(cash_val)


### 💰 Анализ Cash Flow Statement


In [ ]:
if '_fmt_currency' not in globals():
    def _fmt_currency(val):
        if val is None or pd.isna(val):
            return 'N/A'
        try:
            return f"${float(val):,.0f}"
        except Exception:
            return str(val)

display(Markdown("#### 💰 Основные компоненты движения денежных средств"))

cf_forecast = load_model_forecast_from_db('CF', canonical=True)
if cf_forecast.empty:
    print("⚠️ Прогноз CF недоступен")
else:
    cf_metrics = cf_forecast.set_index('metric') if 'metric' in cf_forecast.columns else cf_forecast
    years = sorted(int(c) for c in cf_forecast.columns if str(c).isdigit())
    key_metrics = [('CFO', 'Operating'), ('CFI', 'Investing'), ('CFF', 'Financing'), ('net_change', 'Net Change')]
    summary_rows = []
    for metric, label in key_metrics:
        if metric in cf_metrics.index:
            values = [pd.to_numeric(cf_metrics.loc[metric, str(y)], errors='coerce') for y in years]
            summary_rows.append({'Component': label, **{y: _fmt_currency(v) for y, v in zip(years, values)}})
    if summary_rows:
        display(pd.DataFrame(summary_rows))
    else:
        print("⚠️ Ключевые метрики CF не найдены")
    df_plot = pd.DataFrame({label: [pd.to_numeric(cf_metrics.loc[metric, str(y)], errors='coerce') if metric in cf_metrics.index else pd.NA for y in years] for metric, label in key_metrics}, index=years)
    if not df_plot.empty:
        ax = df_plot.plot(kind='bar', figsize=(12, 5), edgecolor='black')
        ax.set_ylabel('$ millions')
        ax.set_title('CFO / CFI / CFF / Net Change')
        plt.xticks(rotation=0)
        ax.grid(True, axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()


### ✅ Улучшенная валидация финансовых связей


In [ ]:
# Улучшенная валидация финансовых связей
display(Markdown("#### ✅ Детальная проверка финансовых связей"))

bs_df = load_model_forecast_from_db('BS', canonical=True)
is_df = load_model_forecast_from_db('IS', canonical=True)
cf_df = load_model_forecast_from_db('CF', canonical=True)
if bs_df.empty or is_df.empty or cf_df.empty:
    print("❌ Не удалось загрузить прогнозы BS/IS/CF из витрины или CSV fallback")
else:
    bs_idx = bs_df.set_index('metric') if 'metric' in bs_df.columns else bs_df
    is_idx = is_df.set_index('metric') if 'metric' in is_df.columns else is_df
    cf_idx = cf_df.set_index('metric') if 'metric' in cf_df.columns else cf_df
    forecast_years = sorted([int(c) for c in bs_df.columns if str(c).isdigit()])
    validation_rows = []

    def _append(check, year, ok, details='OK'):
        validation_rows.append({'Check': check, 'Year': year, 'Status': '✅' if ok else '❌', 'Details': details})

    # 1. BS Identity
    for y in forecast_years:
        assets = pd.to_numeric(bs_idx.get(str(y), pd.Series(dtype=float)).get('total_assets'), errors='coerce') if 'total_assets' in bs_idx.index else pd.NA
        eq_liab = pd.to_numeric(bs_idx.get(str(y), pd.Series(dtype=float)).get('total_liab_equity'), errors='coerce') if 'total_liab_equity' in bs_idx.index else pd.NA
        if pd.notna(assets) and pd.notna(eq_liab):
            diff = abs(float(assets) - float(eq_liab))
            _append('BS Identity', y, diff <= 0.01, 'OK' if diff <= 0.01 else f'Δ=${diff:,.0f}')

    # 2. Cash Bridge
    hist_bs = load_history_from_db('BS')
    prev_cash = float(hist_bs.loc[hist_bs['year'] == hist_bs['year'].max(), 'cash'].iloc[0]) if not hist_bs.empty and 'cash' in hist_bs.columns else 0.0
    for y in forecast_years:
        cash = pd.to_numeric(bs_idx.loc['cash', str(y)], errors='coerce') if 'cash' in bs_idx.index else pd.NA
        net_change = pd.to_numeric(cf_idx.loc['net_change', str(y)], errors='coerce') if 'net_change' in cf_idx.index else pd.NA
        if pd.notna(cash) and pd.notna(net_change):
            calc = prev_cash + float(net_change)
            diff = abs(float(cash) - calc)
            _append('Cash Bridge', y, diff <= 0.01, 'OK' if diff <= 0.01 else f'Δ=${diff:,.0f}')
            prev_cash = float(cash)

    # 3. WC Delta
    prev_wc = None
    for y in forecast_years:
        wc = pd.to_numeric(bs_idx.loc['wc', str(y)], errors='coerce') if 'wc' in bs_idx.index else pd.NA
        wc_delta_cf = pd.to_numeric(cf_idx.loc['wc_delta', str(y)], errors='coerce') if 'wc_delta' in cf_idx.index else pd.NA
        if pd.notna(wc):
            if prev_wc is not None and pd.notna(wc_delta_cf):
                diff = abs((float(wc) - prev_wc) - float(wc_delta_cf))
                _append('WC Delta', y, diff <= 0.01, 'OK' if diff <= 0.01 else f'Δ=${diff:,.0f}')
            prev_wc = float(wc)

    # 4. Retained Earnings
    prev_re = float(hist_bs.loc[hist_bs['year'] == hist_bs['year'].max(), 'retained_earnings'].iloc[0]) if not hist_bs.empty and 'retained_earnings' in hist_bs.columns else 0.0
    for y in forecast_years:
        re_val = pd.to_numeric(bs_idx.loc['retained_earnings', str(y)], errors='coerce') if 'retained_earnings' in bs_idx.index else pd.NA
        ni_val = pd.to_numeric(is_idx.loc['net_income', str(y)], errors='coerce') if 'net_income' in is_idx.index else pd.NA
        if pd.notna(re_val) and pd.notna(ni_val):
            calc = prev_re + float(ni_val)
            diff = abs(float(re_val) - calc)
            _append('Retained Earnings', y, diff <= 0.01, 'OK' if diff <= 0.01 else f'Δ=${diff:,.0f}')
            prev_re = float(re_val)

    # 5. Intangibles corkscrew
    intangibles_table = load_model_table('INTANGIBLES_CORKSCREW', canonical=False, csv_filename='outputs/model/intangibles_corkscrew.csv')
    if not intangibles_table.empty and 'metric' in intangibles_table.columns and 'intangibles' in bs_idx.index:
        int_idx = intangibles_table.set_index('metric')
        prev_int = float(hist_bs.loc[hist_bs['year'] == hist_bs['year'].max(), 'intangibles'].iloc[0]) if not hist_bs.empty and 'intangibles' in hist_bs.columns else 0.0
        for y in forecast_years:
            balance = pd.to_numeric(bs_idx.loc['intangibles', str(y)], errors='coerce')
            additions = pd.to_numeric(int_idx.loc['Additions', str(y)], errors='coerce') if 'Additions' in int_idx.index else 0.0
            amort = pd.to_numeric(int_idx.loc['Amortization', str(y)], errors='coerce') if 'Amortization' in int_idx.index else 0.0
            if pd.notna(balance):
                calc = max(0.0, prev_int + (float(additions) if pd.notna(additions) else 0.0) - (float(amort) if pd.notna(amort) else 0.0))
                diff = abs(float(balance) - calc)
                _append('Intangibles Corkscrew', y, diff <= 0.01, 'OK' if diff <= 0.01 else f'Δ=${diff:,.0f}')
                prev_int = float(balance)
    else:
        print("ℹ️ Информация по intangibles corkscrew отсутствует")

    if validation_rows:
        val_df = pd.DataFrame(validation_rows)
        display(val_df)
        total = len(val_df)
        passed = (val_df['Status'] == '✅').sum()
        failed = total - passed
        print(f"\n📈 Проверок: {total} | ✅ {passed} | ❌ {failed}")


In [ ]:
# Анализ Cash Flow Statement
display(Markdown("#### 💰 Cash Flow Statement (Отчет о движении денежных средств)"))

cf_forecast = load_model_forecast_from_db('CF', canonical=True)
if cf_forecast.empty:
    print("⚠️ Прогноз CF недоступен в витрине")
else:
    cf_metrics = cf_forecast.set_index('metric') if 'metric' in cf_forecast.columns else cf_forecast
    forecast_years = sorted([int(c) for c in cf_forecast.columns if str(c).isdigit()])
    if forecast_years:
        def _metric_series(metric_name):
            if metric_name in cf_metrics.index:
                return [pd.to_numeric(cf_metrics.loc[metric_name, str(y)], errors='coerce') for y in forecast_years]
            return []
        cfo_vals = _metric_series('CFO')
        cfi_vals = _metric_series('CFI')
        cff_vals = _metric_series('CFF')
        net_vals = _metric_series('NetChange')
        print(" 📊 Проверка связей CF (первые 3 года):")
        for y, cfo, cfi, cff, net in zip(forecast_years[:3], cfo_vals[:3], cfi_vals[:3], cff_vals[:3], net_vals[:3]):
            if pd.notna(cfo) and pd.notna(cfi) and pd.notna(cff) and pd.notna(net):
                calc = float(cfo) + float(cfi) + float(cff)
                diff = abs(calc - float(net))
                if diff < 0.01:
                    print(f"  {y}: ✅ NetChange = CFO + CFI + CFF = {calc:,.0f}")
                else:
                    print(f"  {y}: ⚠️ NetChange ожидалось {calc:,.0f}, получено {net:,.0f}, diff={diff:,.0f}")
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.flatten()
        for ax, (name, values) in zip(axes, [('CFO', cfo_vals), ('CFI', cfi_vals), ('CFF', cff_vals), ('Net Change', net_vals)]):
            if values:
                values = [float(v) if pd.notna(v) else 0 for v in values]
                colors = ['green' if v >= 0 else 'red' for v in values]
                ax.bar(forecast_years, values, color=colors, alpha=0.7, edgecolor='black')
                ax.axhline(y=0, color='black', linewidth=0.8)
                ax.set_title(f'{name}: Forecast'); ax.set_xlabel('Year'); ax.set_ylabel('$ millions'); ax.grid(True, alpha=0.3)
                for year, val in zip(forecast_years, values):
                    ax.text(year, val, f'{val:,.0f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=8)
        plt.tight_layout(); plt.show()
        print(" 📊 Детализация CFO (первые 3 года):")
        ni_vals = _metric_series('net_income') or [float('nan')]*len(forecast_years)
        dep_vals = _metric_series('depreciation') or [float('nan')]*len(forecast_years)
        wc_vals = _metric_series('wc_delta') or [float('nan')]*len(forecast_years)
        fi_vals = []
        is_forecast = load_model_forecast_from_db('IS', canonical=True)
        if not is_forecast.empty and 'metric' in is_forecast.columns:
            is_metrics = is_forecast.set_index('metric')
            if 'finance_income' in is_metrics.index:
                fi_vals = [pd.to_numeric(is_metrics.loc['finance_income', str(y)], errors='coerce') for y in forecast_years]
        if not fi_vals:
            fi_vals = [0.0]*len(forecast_years)
        for y, cfo, ni, dep, wc, fi in zip(forecast_years[:3], cfo_vals[:3], ni_vals[:3], dep_vals[:3], wc_vals[:3], fi_vals[:3]):
            if pd.notna(cfo) and pd.notna(ni) and pd.notna(dep) and pd.notna(wc):
                cfo_calc = float(ni) + float(dep) - float(wc) + (float(fi) if pd.notna(fi) else 0.0)
                diff = abs(float(cfo) - cfo_calc)
                status = '✅' if diff < 0.01 else '⚠️'
                print(f"  {y}: CFO={cfo:,.0f}, calc={cfo_calc:,.0f} {status}")
else:
    print("⚠️ Прогноз CF отсутствует")


### 📊 Анализ EBITDA, EBIT, EBT, Net Income и их связей


In [ ]:
# Анализ связей между статьями Income Statement
display(Markdown("#### 📊 Проверка связей IS: Revenue → COGS → SG&A → Dep → EBITDA → EBIT → EBT → Tax → NI"))

is_forecast = load_model_forecast_from_db('IS', canonical=True)
if is_forecast.empty:
    print("⚠️ Прогноз IS недоступен")
else:
    is_metrics = is_forecast.setIndex('metric') if 'metric' in is_forecast.columns else is_forecast
    forecast_years = sorted([int(c) for c in is_forecast.columns if str(c).isdigit()])
    def _get(metric):
        return [pd.to_numeric(is_metrics.loc[metric, str(y)], errors='coerce') if metric in is_metrics.index else float('nan') for y in forecast_years]
    revenue_vals = _get('revenue')
    cogs_vals = _get('cogs')
    sgna_vals = _get('sga')
    dep_vals = _get('depreciation') or _get('dep')
    ebitda_vals = _get('ebitda')
    ebit_vals = _get('ebit')
    interest_vals = _get('interest_expense')
    lease_interest_vals = _get('lease_interest')
    ebt_vals = _get('ebt')
    tax_vals = _get('tax')
    ni_vals = _get('net_income')
    print("📊 Проверка связей (первые 3 года):")
    for y, idx in zip(forecast_years[:3], range(3)):
        rev, cogs, sgna, dep, ebitda, ebit, interest, lease_interest, ebt_val, tax, ni = [vals[idx] for vals in [revenue_vals, cogs_vals, sgna_vals, dep_vals, ebitda_vals, ebit_vals, interest_vals, lease_interest_vals, ebt_vals, tax_vals, ni_vals]]
        if all(pd.notna(v) for v in [rev, cogs, sgna, dep]) and pd.notna(ebitda):
            calc_ebitda = rev - cogs - sgna - dep
            diff = abs(calc_ebitda - ebitda)
            print(f"  {y}: EBITDA {'✅' if diff < 0.01 else '⚠️'} (расчет={calc_ebitda:,.0f}, модель={ebitda:,.0f})")
        if pd.notna(ebitda) and pd.notna(dep) and pd.notna(ebit):
            calc_ebit = ebitda - dep
            diff = abs(calc_ebit - ebit)
            print(f"    EBIT {'✅' if diff < 0.01 else '⚠️'} (расчет={calc_ebit:,.0f}, модель={ebit:,.0f})")
        net_interest = 0.0
        if pd.notna(interest):
            net_interest += float(interest)
        if pd.notna(lease_interest):
            net_interest += float(lease_interest)
        if pd.notna(ebit) and pd.notna(net_interest) and pd.notna(ebt_val):
            calc_ebt = ebit - net_interest
            diff = abs(calc_ebt - ebt_val)
            print(f"    EBT {'✅' if diff < 0.01 else '⚠️'} (расчет={calc_ebt:,.0f}, модель={ebt_val:,.0f})")
        if pd.notna(ebt_val) and pd.notna(tax) and pd.notna(ni):
            calc_ni = ebt_val - tax
            diff = abs(calc_ni - ni)
            print(f"    Net Income {'✅' if diff < 0.01 else '⚠️'} (расчет={calc_ni:,.0f}, модель={ni:,.0f})")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    for ax, (name, values) in zip(axes, [('EBITDA', ebitda_vals), ('EBIT', ebit_vals), ('EBT', ebt_vals), ('Net Income', ni_vals)]):
        clean_vals = [float(v) if pd.notna(v) else 0 for v in values]
        ax.plot(forecast_years, clean_vals, 'o-', linewidth=2, markersize=8)
        ax.set_title(f'{name}: Forecast'); ax.set_xlabel('Year'); ax.set_ylabel('$ millions'); ax.grid(True, alpha=0.3); ax.axhline(y=0, color='black', linewidth=0.8)
    plt.tight_layout(); plt.show()


### 💼 Анализ Working Capital, PP&E, CapEx


In [ ]:
if '_fmt_currency' not in globals():
    def _fmt_currency(val):
        if val is None or pd.isna(val):
            return 'N/A'
        try:
            return f"${float(val):,.0f}"
        except Exception:
            return str(val)

display(Markdown("#### 💼 Сводка по Working Capital, PP&E и CapEx"))

bs_forecast = load_model_forecast_from_db('BS', canonical=True)
is_forecast = load_model_forecast_from_db('IS', canonical=True)
cf_forecast = load_model_forecast_from_db('CF', canonical=True)

if bs_forecast.empty or cf_forecast.empty:
    print("⚠️ Недостаточно данных BS/CF для анализа")
else:
    bs_metrics = bs_forecast.set_index('metric') if 'metric' in bs_forecast.columns else bs_forecast
    cf_metrics = cf_forecast.set_index('metric') if 'metric' in cf_forecast.columns else cf_forecast
    is_metrics = is_forecast.set_index('metric') if not is_forecast.empty and 'metric' in is_forecast.columns else is_forecast
    years = sorted(int(c) for c in bs_forecast.columns if str(c).isdigit())
    def _first_available(metrics, keys):
        for key in keys:
            if key in metrics.index:
                return key
        return None
    wc_key = _first_available(bs_metrics, ['wc', 'wc_balance', 'working_capital'])
    ppe_key = _first_available(bs_metrics, ['ppe', 'ppe_net'])
    debt_keys = [key for key in ['debt', 'lt_debt'] if key in bs_metrics.index]
    st_debt_key = 'st_debt' if 'st_debt' in bs_metrics.index else None
    capex_key = _first_available(cf_metrics, ['total_capex', 'capex'])
    dep_key = _first_available(is_metrics, ['depreciation', 'dep']) if not is_forecast.empty else None
    rows = []
    for y in years:
        wc_val = pd.to_numeric(bs_metrics.loc[wc_key, str(y)], errors='coerce') if wc_key else pd.NA
        ppe_val = pd.to_numeric(bs_metrics.loc[ppe_key, str(y)], errors='coerce') if ppe_key else pd.NA
        capex_val = pd.to_numeric(cf_metrics.loc[capex_key, str(y)], errors='coerce') if capex_key else pd.NA
        dep_val = pd.to_numeric(is_metrics.loc[dep_key, str(y)], errors='coerce') if dep_key and not is_forecast.empty else pd.NA
        cash_val = pd.to_numeric(bs_metrics.loc['cash', str(y)], errors='coerce') if 'cash' in bs_metrics.index else pd.NA
        debt_val = 0.0
        if debt_keys:
            for key in debt_keys:
                debt_val += float(pd.to_numeric(bs_metrics.loc[key, str(y)], errors='coerce')) if pd.notna(pd.to_numeric(bs_metrics.loc[key, str(y)], errors='coerce')) else 0.0
        if st_debt_key:
            st_val = pd.to_numeric(bs_metrics.loc[st_debt_key, str(y)], errors='coerce')
            debt_val += float(st_val) if pd.notna(st_val) else 0.0
        net_debt = debt_val - (float(cash_val) if pd.notna(cash_val) else 0.0)
        rows.append({
            'Year': y,
            'Working Capital': wc_val,
            'PP&E': ppe_val,
            'CapEx': capex_val,
            'Depreciation': dep_val,
            'Cash': cash_val,
            'Total Debt': debt_val,
            'Net Debt': net_debt
        })
    summary_df = pd.DataFrame(rows)
    if summary_df.empty:
        print("⚠️ Не удалось сформировать сводную таблицу")
    else:
        formatted = summary_df.copy()
        for col in ['Working Capital', 'PP&E', 'CapEx', 'Depreciation', 'Cash', 'Total Debt', 'Net Debt']:
            formatted[col] = formatted[col].apply(lambda v: _fmt_currency(v) if pd.notna(v) else 'N/A')
        display(formatted)


### 📊 Сводная таблица всех методов и параметров моделирования


In [ ]:
# Создаем сводную таблицу методов моделирования
modeling_summary = []

# Revenue
revenue_cfg = proj_yaml.get("model", {}).get("standard", {}).get("revenue", {})
method = "Elastic Net Regression" if revenue_cfg.get("use_elastic_net", False) else revenue_cfg.get("fallback_growth", "N/A")
modeling_summary.append({
    "Статья": "Revenue",
    "Метод": method,
    "Преднастройки": f"Driver: {revenue_cfg.get('driver', 'N/A')}, Transform: {revenue_cfg.get('target_transform', 'none')}",
    "Исторические параметры": "Расчет из истории revenue (2010-2024)",
    "Предпосылки": f"Chain-link anchor: {revenue_cfg.get('chainlink_anchor', 'history_last')}"
})

# COGS
cogs_cfg = proj_yaml.get("model", {}).get("standard", {}).get("cogs", {})
clamp = cogs_cfg.get("clamp", {})
modeling_summary.append({
    "Статья": "COGS",
    "Метод": "Ratio от Revenue (из истории, с clamp)",
    "Преднастройки": f"Clamp: [{clamp.get('min', 0.55)}, {clamp.get('max', 0.98)}]",
    "Исторические параметры": "Средний COGS/Revenue ratio из истории",
    "Предпосылки": f"PPI uplift: {cogs_cfg.get('use_ppi_uplift', False)}"
})

# SG&A
sgna_cfg = proj_yaml.get("model", {}).get("standard", {}).get("sgna", {})
modeling_summary.append({
    "Статья": "SG&A",
    "Метод": sgna_cfg.get("base_method", "ratio"),
    "Преднастройки": f"EWA half-life: {sgna_cfg.get('ewa_halflife_years', 2)} лет, Transform: {sgna_cfg.get('transform', 'none')}",
    "Исторические параметры": "EWA сглаживание истории SG&A",
    "Предпосылки": f"CPI indexation: {sgna_cfg.get('index_by_cpi', False)} (beta={sgna_cfg.get('cpi_beta', 1.0)})"
})

# Depreciation
modeling_summary.append({
    "Статья": "Depreciation",
    "Метод": "PP&E Corkscrew",
    "Преднастройки": "PP&E_t = PP&E_{t-1} + CapEx - Disposals - Dep",
    "Исторические параметры": "Начальное PP&E из BS history (2024)",
    "Предпосылки": "Disposals = ratio * CapEx"
})

# Interest
debt_cfg = proj_yaml.get("model", {}).get("standard", {}).get("debt", {})
modeling_summary.append({
    "Статья": "Interest",
    "Метод": "Из Debt Schedule (waterfall)",
    "Преднастройки": f"RC: {debt_cfg.get('rc', {}).get('enable', False)}, Refinancing: {debt_cfg.get('refinancing', {}).get('enable', False)}",
    "Исторические параметры": "Debt schedule из data/debt/",
    "Предпосылки": f"Iterative convergence (max_iter={debt_cfg.get('iter_max', 50)})"
})

# Tax
margins_cfg = proj_yaml.get("model", {}).get("standard", {}).get("margins", {})
taxes_cfg = proj_yaml.get("model", {}).get("standard", {}).get("taxes", {})
modeling_summary.append({
    "Статья": "Tax",
    "Метод": taxes_cfg.get("policy", "statutory_with_floor"),
    "Преднастройки": f"Statutory rate: {margins_cfg.get('tax_rate_statutory', 0.21):.1%}",
    "Исторические параметры": "Effective tax rate из истории",
    "Предпосылки": "Tax floor для отрицательного EBT"
})

# WC
wc_cfg = proj_yaml.get("model", {}).get("standard", {}).get("wc", {})
modeling_summary.append({
    "Статья": "Working Capital",
    "Метод": f"{wc_cfg.get('method', 'days')} method",
    "Преднастройки": f"DSO={wc_cfg.get('dso_days', 30)}, DIO={wc_cfg.get('dio_days', 40)}, DPO={wc_cfg.get('dpo_days', 45)}",
    "Исторические параметры": "Расчет дней из истории WC balances",
    "Предпосылки": f"Floors применяются, Transform: {wc_cfg.get('transform_days', 'none')}"
})

# CapEx
capex_cfg = proj_yaml.get("model", {}).get("standard", {}).get("capex", {})
modeling_summary.append({
    "Статья": "CapEx",
    "Метод": capex_cfg.get("method", "ratio_to_revenue"),
    "Преднастройки": f"Ratio: {capex_cfg.get('ratio_default', 0.08):.1%}, Maint=Dep*{capex_cfg.get('maint_as_dep_ratio', 1.0)}",
    "Исторические параметры": "Maint CapEx из Dep, Growth из % revenue",
    "Предпосылки": "MaintCapEx >= Dep, Growth = % of Revenue"
})

# PP&E
modeling_summary.append({
    "Статья": "PP&E",
    "Метод": "Corkscrew roll-forward",
    "Преднастройки": "PP&E_t = PP&E_{t-1} + CapEx - Disposals - Dep",
    "Исторические параметры": "Начальное PP&E из BS (2024)",
    "Предпосылки": "PP&E >= 0, Disposals = ratio * CapEx"
})

# Finance Income
finance_cfg_table = proj_yaml.get("model", {}).get("standard", {}).get("finance", {})
finance_income_cfg_table = finance_cfg_table.get("income", {})
modeling_summary.append({
    "Статья": "Finance Income",
    "Метод": "% от среднего Cash balance",
    "Преднастройки": f"Rate: {finance_income_cfg_table.get('rate_pct', 'авторасчет')}%",
    "Исторические параметры": "Рассчитывается из истории (Finance Income / Avg Cash) если rate_pct=null",
    "Предпосылки": "Avg Cash = (Cash_{t-1} + Cash_t) / 2, включен в CFO"
})

# Finance Expense
finance_expense_cfg_table = finance_cfg_table.get("expense", {})
modeling_summary.append({
    "Статья": "Finance Expense",
    "Метод": "Interest Expense + addon",
    "Преднастройки": f"Addon: {finance_expense_cfg_table.get('addon_to_interest_pct', 0.0)}% от Interest",
    "Исторические параметры": "Исторический Finance Expense из IS",
    "Предпосылки": "FE = Interest * (1 + addon_pct / 100), влияет на Net Income"
})

# Intangible Assets
intangibles_cfg_table = proj_yaml.get("model", {}).get("standard", {}).get("intangibles", {})
additions_method_table = intangibles_cfg_table.get("additions_method", "pct_of_revenue")
amortization_method_table = intangibles_cfg_table.get("amortization_method", "pct_of_balance")
additions_desc = f"{additions_method_table}"
if additions_method_table == "pct_of_revenue":
    additions_desc += f" ({intangibles_cfg_table.get('additions_pct_of_revenue', 0.0)}%)"
elif additions_method_table == "fixed":
    additions_desc += f" (${intangibles_cfg_table.get('additions_fixed', 0.0):,.0f})"

amortization_desc = f"{amortization_method_table}"
if amortization_method_table == "pct_of_balance":
    amortization_desc += f" ({intangibles_cfg_table.get('amortization_pct_of_balance', 10.0)}%)"
elif amortization_method_table == "fixed_rate":
    amortization_desc += f" (${intangibles_cfg_table.get('amortization_fixed_rate', 0.0):,.0f})"

modeling_summary.append({
    "Статья": "Intangible Assets",
    "Метод": "Corkscrew (аналогично PP&E)",
    "Преднастройки": f"Additions: {additions_desc}, Amortization: {amortization_desc}",
    "Исторические параметры": "Начальные Intangibles из BS (2024)",
    "Предпосылки": "Intangibles_t = Intangibles_{t-1} + Additions - Amortization, включены в total_assets"
})

summary_df = pd.DataFrame(modeling_summary)
display(summary_df)

print(f"\n📊 Всего статей: {len(modeling_summary)}")


In [ ]:
### 📊 График коэффициентов регрессии

reg_df = load_model_table('REVENUE_REGRESSION_SUMMARY', csv_filename='outputs/model/revenue_regression_summary.csv')
if reg_df.empty:
    print("⚠️ Данные регрессии не найдены в витрине")
else:
    if 'feature' in reg_df.columns and 'coef' in reg_df.columns:
        coefs = reg_df[(reg_df['feature'].notna()) & (reg_df['feature'].astype(str).str.lower() != 'intercept') & (reg_df['coef'].notna())].copy()
        if not coefs.empty:
            coefs['coef'] = pd.to_numeric(coefs['coef'], errors='coerce')
            coefs['abs_coef'] = coefs['coef'].abs()
            coefs = coefs.sort_values('abs_coef', ascending=True)
            fig, ax = plt.subplots(figsize=(12, 8))
            y_pos = range(len(coefs))
            colors = ['red' if c > 0 else 'blue' for c in coefs['coef']]
            ax.barh(list(y_pos), coefs['coef'], color=colors, alpha=0.7)
            ax.set_yticks(list(y_pos))
            ax.set_yticklabels(coefs['feature'].str.replace('_', ' ').str.title())
            ax.set_xlabel('Коэффициент регрессии (standardized)')
            ax.set_title('Важность макро-факторов для Revenue')
            ax.axvline(x=0, color='black', linewidth=0.8)
            ax.grid(True, alpha=0.3, axis='x')
            for i, (_, row) in enumerate(coefs.iterrows()):
                coef_val = float(row['coef']) if pd.notna(row['coef']) else 0
                ax.text(coef_val, i, f' {coef_val:+.4f}', va='center', fontsize=9)
            plt.tight_layout(); plt.show()
        else:
            print("⚠️ Коэффициенты регрессии отсутствуют")


In [ ]:
display(Markdown("### Проверка выходных данных модели"))

outputs = {
    '3-Statement IS': {'stmt': 'IS', 'canonical': True, 'csv': 'outputs/model/3statement_IS.csv'},
    '3-Statement BS': {'stmt': 'BS', 'canonical': True, 'csv': 'outputs/model/3statement_BS.csv'},
    '3-Statement CF': {'stmt': 'CF', 'canonical': True, 'csv': 'outputs/model/3statement_CF.csv'},
    'WC Days': {'stmt': 'WC_DAYS', 'canonical': False, 'csv': 'outputs/model/wc_days.csv'},
    'PP&E Corkscrew': {'stmt': 'PPE_CORKSCREW', 'canonical': False, 'csv': 'outputs/model/ppe_corkscrew.csv'},
    'CapEx Breakdown': {'stmt': 'CAPEX_BREAKDOWN', 'canonical': False, 'csv': 'outputs/model/capex_breakdown.csv'},
    'Debt Waterfall': {'stmt': 'DEBT_WATERFALL', 'canonical': False, 'csv': 'outputs/model/debt_waterfall.csv'},
    'RC Convergence Log': {'stmt': 'RC_CONVERGENCE_LOG', 'canonical': False, 'csv': 'outputs/model/rc_convergence_log.csv'},
    'Regression Summary': {'stmt': 'REVENUE_REGRESSION_SUMMARY', 'canonical': False, 'csv': 'outputs/model/revenue_regression_summary.csv'}
}

rows = []
for name, meta in outputs.items():
    if meta['stmt'] in ['IS', 'BS', 'CF']:
        df = load_model_forecast_from_db(meta['stmt'], canonical=meta['canonical'])
    else:
        df = load_model_table(meta['stmt'], canonical=meta['canonical'], csv_filename=meta['csv'])
    rows.append({
        'table': name,
        'source': 'Data Mart' if not df.empty else 'CSV fallback',
        'rows': len(df) if isinstance(df, pd.DataFrame) else 0,
        'cols': len(df.columns) if isinstance(df, pd.DataFrame) and not df.empty else 0
    })

overview = pd.DataFrame(rows)
display(overview)

if rows[0]['rows'] > 0:
    display(Markdown("#### Пример: Income Statement"))
    display(load_model_forecast_from_db('IS', canonical=True).head())
if rows[1]['rows'] > 0:
    display(Markdown("#### Пример: Balance Sheet"))
    display(load_model_forecast_from_db('BS', canonical=True).head())


## 8️⃣ Анализ результатов VECM <a id="section-8"></a>

### 8.1 Анализ макропрогноза (витрина данных) <a id="section-8-1"></a>

### 8.3 Анализ SVAR модели (если используется) <a id="section-8-3"></a>


In [ ]:
display(Markdown("#### 8.3.1 Проверка использования SVAR модели"))

# Проверяем конфигурацию
cfg_path = croot / "configs" / "macro_ecm.yaml"
if not cfg_path.exists():
    cfg_path = ROOT / "configs" / "forecast" / "macro_ecm.yaml"

svar_enabled = False
if cfg_path.exists():
    try:
        with open(cfg_path, 'r', encoding='utf-8') as f:
            cfg = yaml.safe_load(f)
        svar_enabled = cfg.get('svar', {}).get('enabled', False)
    except Exception as exc:
        print(f"⚠️ Не удалось прочитать конфигурацию: {exc}")

if svar_enabled:
    display(Markdown("**✅ SVAR модель включена в конфигурации**"))
    
    # Загружаем диагностику SVAR из БД
    with get_data_mart(ROOT, COMPANY) as mart:
        # Проверяем наличие прогнозов с методом SVAR
        macro_forecasts_svar = {}
        for factor in factors:
            forecast_data = mart.get_macro_forecast(factor)
            if forecast_data:
                # Проверяем метод прогнозирования
                diag_rows = mart.get_ecm_diagnostics()
                if not diag_rows.empty:
                    factor_diag = diag_rows[diag_rows['factor_name'] == factor]
                    if not factor_diag.empty:
                        method = factor_diag['method'].iloc[-1]
                        if 'SVAR' in str(method).upper():
                            macro_forecasts_svar[factor] = forecast_data
        
        if macro_forecasts_svar:
            display(Markdown(f"**📊 Найдено {len(macro_forecasts_svar)} факторов, прогнозируемых SVAR:**"))
            for factor, forecast in macro_forecasts_svar.items():
                years = sorted(forecast.keys())
                display(Markdown(f"- **{factor}**: прогноз на {len(years)} лет ({min(years)}-{max(years)})"))
            
            display(Markdown("**💡 Для детального анализа IRF и Variance Decomposition используйте модуль `engine.ecm.svar`**"))
        else:
            display(Markdown("**ℹ️ SVAR включена в конфигурации, но прогнозы еще не сгенерированы. Запустите `run_ecm_all()` для генерации.**"))
else:
    display(Markdown("**ℹ️ SVAR модель не включена в конфигурации. Для использования SVAR установите `svar.enabled: true` в `macro_ecm.yaml`**"))


In [ ]:
if '_fmt_currency' not in globals():
    def _fmt_currency(val):
        if val is None or pd.isna(val):
            return 'N/A'
        try:
            return f"${float(val):,.0f}"
        except Exception:
            return str(val)

display(Markdown("#### 8.1.1 Сводка результатов VECM"))

selection_log_path = ROOT / "companies" / COMPANY / "outputs" / "logs" / "vecm_group_selection.json"
if selection_log_path.exists():
    try:
        selection_data = json.loads(selection_log_path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"⚠️ Не удалось прочитать vecm_group_selection.json: {exc}")
    else:
        if selection_data:
            log_df = pd.DataFrame(selection_data)
            if 'factors' in log_df.columns:
                log_df['factors'] = log_df['factors'].apply(lambda v: ', '.join(v) if isinstance(v, list) else v)
            display(Markdown("##### 🤖 Лог автоподбора групп"))
            display(log_df)
        else:
            print("ℹ️ Лог автоподбора пуст")
else:
    print("ℹ️ Файл vecm_group_selection.json еще не создан — запустите VECM для генерации лога")

ecm_diag_df: pd.DataFrame = pd.DataFrame()
ecm_forecast_diag_df: pd.DataFrame = pd.DataFrame()
ecm_avf_df: pd.DataFrame = pd.DataFrame()
macro_forecasts: dict[str, dict[int, float]] = {}

with get_data_mart(ROOT, COMPANY) as mart:
    diag_raw = mart.get_ecm_diagnostics()
    ecm_diag_df = diag_raw if isinstance(diag_raw, pd.DataFrame) else pd.DataFrame()

    forecast_diag_raw = mart.get_ecm_forecast_diag()
    ecm_forecast_diag_df = forecast_diag_raw if isinstance(forecast_diag_raw, pd.DataFrame) else pd.DataFrame()

    avf_raw = mart.get_ecm_actual_vs_fitted()
    ecm_avf_df = avf_raw if isinstance(avf_raw, pd.DataFrame) else pd.DataFrame()

    macro_forecasts = {
        factor: mart.get_macro_forecast(factor) or {}
        for factor in factors
    }

forecast_summary = []
example_factor = None
example_series = pd.Series(dtype=float)
example_span_end = None

for factor in factors:
    diag_row = ecm_diag_df[ecm_diag_df['factor_name'] == factor].tail(1) if not ecm_diag_df.empty else pd.DataFrame()
    span_end = int(diag_row['span_end'].iloc[0]) if not diag_row.empty and pd.notna(diag_row['span_end'].iloc[0]) else None
    method = diag_row['method'].iloc[0] if not diag_row.empty else 'N/A'
    note = diag_row['note'].iloc[0] if not diag_row.empty else ''

    forecast_series = macro_forecasts.get(factor, {})
    years = sorted(forecast_series.keys())
    forecast_horizon = len([y for y in years if span_end is not None and y > span_end])
    forecast_end = max(years) if years else None
    has_forecast = forecast_horizon > 0

    forecast_summary.append({
        'factor': factor,
        'method': method,
        'span_end': span_end,
        'forecast_years': forecast_horizon,
        'forecast_end': forecast_end,
        'note': note
    })

    if example_factor is None and has_forecast:
        example_factor = factor
        example_series = pd.Series(forecast_series).sort_index()
        example_span_end = span_end

summary_df = pd.DataFrame(forecast_summary)
if not summary_df.empty:
    display(summary_df)
else:
    print("⚠️ В витрине нет данных о макропрогнозе.")

if example_factor and not example_series.empty:
    display(Markdown(f"#### 8.1.2 Пример прогноза ({example_factor})"))
    df_display = pd.DataFrame({'year': example_series.index, 'ln_value': example_series.values})
    display(df_display.tail(10))
    try:
        history_mask = example_series.index <= example_span_end if example_span_end is not None else example_series.index <= example_series.index.max()
        history_series = example_series[history_mask]
        forecast_part = example_series[~history_mask] if example_span_end is not None else pd.Series(dtype=float)
        plt.figure(figsize=(12, 6))
        if not history_series.empty:
            plt.plot(history_series.index, history_series.values, 'o-', label='История (ln)', linewidth=2, markersize=6)
        if not forecast_part.empty:
            plt.plot(forecast_part.index, forecast_part.values, 's--', label='Прогноз (ln)', linewidth=2, markersize=6, color='green')
        plt.xlabel('Год'); plt.ylabel('ln(значение)'); plt.title(f'Прогноз для {example_factor}'); plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
    except Exception as exc:
        print(f"⚠️ Не удалось построить график прогноза: {exc}")
else:
    print("⚠️ Не найден фактор с прогнозным горизонтом для примера")

EXAMPLE_FACTOR_VECM = example_factor
EXAMPLE_SERIES_VECM = example_series
EXAMPLE_SPAN_END_VECM = example_span_end
ECM_AVF_VECM = ecm_avf_df.copy()


### 8.2 Анализ точности прогнозов <a id="section-8-2"></a>

In [ ]:
display(Markdown("#### 8.2 Анализ точности прогнозов"))

if 'EXAMPLE_FACTOR_VECM' not in globals() or EXAMPLE_FACTOR_VECM is None:
    print("⚠️ Сначала выполните предыдущую ячейку со сводкой VECM")
else:
    avf_rows = pd.DataFrame()
    if 'ECM_AVF_VECM' in globals() and isinstance(ECM_AVF_VECM, pd.DataFrame):
        avf_rows = ECM_AVF_VECM[ECM_AVF_VECM['factor_name'] == EXAMPLE_FACTOR_VECM] if 'factor_name' in ECM_AVF_VECM.columns else pd.DataFrame()
    if not avf_rows.empty:
        display(Markdown(f"#### Actual vs Fitted для {EXAMPLE_FACTOR_VECM}"))
        display(avf_rows.head(10))
        try:
            plt.figure(figsize=(12, 6))
            plt.plot(avf_rows['year'], avf_rows['actual_ln'], 'o-', label='Actual (ln)', linewidth=2, markersize=6)
            plt.plot(avf_rows['year'], avf_rows['fitted_ln'], 's--', label='Fitted (ln)', linewidth=2, markersize=6, color='orange')
            plt.xlabel('Год'); plt.ylabel('ln(значение)'); plt.title(f'Actual vs Fitted для {EXAMPLE_FACTOR_VECM}'); plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
        except Exception as exc:
            print(f"⚠️ Не удалось построить график Actual vs Fitted: {exc}")
    else:
        print(f"ℹ️ Для {EXAMPLE_FACTOR_VECM} нет записей actual vs fitted в витрине.")


## 5️⃣ Валидация финансовых связей <a id="section-5"></a>

In [ ]:
display(Markdown("### Проверка BS Identity, Cash Bridge, Retained Earnings"))

from engine.acceptance.checks import run_acceptance

print("Запуск acceptance checks...")
try:
    run_acceptance(croot)
    print(" ✅ Acceptance checks выполнены")
    mart = _open_data_mart()
    df = pd.DataFrame()
    if mart is not None:
        try:
            df = pd.read_sql_query(
                "SELECT metric, year, value FROM outputs WHERE company = ? AND output_type = ? ORDER BY metric, year",
                mart.db.conn,
                params=(COMPANY, 'acceptance_checks')
            )
            mart.close()
        except Exception as exc:
            print(f"⚠️ Не удалось прочитать acceptance_checks из витрины: {exc}")
            mart.close()
    if df.empty:
        df = load_company_csv('outputs/checks/acceptance_checks.csv')
    if df.empty:
        print("⚠️ Acceptance checks не найдены")
    else:
        if {'metric','year','value'}.issubset(df.columns):
            pivot = df.pivot_table(index='metric', columns='year', values='value', aggfunc='first')
            display(pivot)
        else:
            display(df)
except Exception as exc:
    print(f"\n❌ Ошибка выполнения acceptance checks: {exc}")
    import traceback
    traceback.print_exc()


In [ ]:
display(Markdown("### Анализ финансовых ковенант"))

mart = _open_data_mart()
covenants_df = pd.DataFrame()
if mart is not None:
    try:
        covenants_df = mart.get_output('covenants')
        if not covenants_df.empty:
            print("✅ Ковенанты загружены из витрины")
    except Exception as exc:
        print(f"⚠️ Не удалось загрузить ковенанты из витрины: {exc}")
    finally:
        mart.close()

if covenants_df.empty:
    covenants_df = load_company_csv('outputs/checks/covenants.csv')

if covenants_df.empty:
    print("⚠️ Данные ковенант не найдены. Убедитесь, что acceptance checks выполнены.")
else:
    proj_yaml_path = croot / 'configs' / 'project.yaml'
    thresholds = {}
    if proj_yaml_path.exists():
        proj_yaml = yaml.safe_load(proj_yaml_path.read_text(encoding='utf-8'))
        covenant_config = proj_yaml.get('covenants', {})
        if isinstance(covenant_config, dict) and covenant_config.get('enabled', True):
            thresholds = covenant_config.get('thresholds', {})
    if 'metric' in covenants_df.columns and any(str(c).isdigit() for c in covenants_df.columns):
        year_cols = [c for c in covenants_df.columns if str(c).isdigit()]
        covenants_wide = covenants_df.copy()
    elif {'metric','year','value'}.issubset(covenants_df.columns):
        covenants_wide = covenants_df.pivot(index='metric', columns='year', values='value').reset_index()
        year_cols = [c for c in covenants_wide.columns if str(c).isdigit()]
    else:
        covenants_wide = covenants_df.copy()
        year_cols = [c for c in covenants_wide.columns if str(c).isdigit()]
    years = sorted(int(y) for y in year_cols)
    if not years:
        print("⚠️ Не найдены годы в данных ковенант")
    else:
        print(f"\n📊 Ковенанты рассчитаны для годов: {years[0]}-{years[-1]}")
        covenant_metrics = [
            ('NetDebt/EBITDA', 'net_debt_to_ebitda_max', 'max', 4.0),
            ('Interest_Coverage_Ratio', 'interest_coverage_min', 'min', 2.0),
            ('DSCR', 'dscr_min', 'min', 1.2),
            ('Debt/Equity', 'debt_to_equity_max', 'max', 1.0),
            ('FFO/NetDebt', 'ffo_to_debt_min', 'min', 0.15),
            ('Debt/FFO', 'debt_to_ffo_max', 'max', 6.0)
        ]
        summary_rows = []
        for metric_name, threshold_key, threshold_type, default_threshold in covenant_metrics:
            threshold = thresholds.get(threshold_key, default_threshold)
            metric_row = covenants_wide[covenants_wide['metric'].astype(str).str.lower().str.contains(metric_name.lower())] if 'metric' in covenants_wide.columns else pd.DataFrame()
            if not metric_row.empty:
                row = metric_row.iloc[0]
                values = [pd.to_numeric(row.get(str(y)), errors='coerce') for y in years[:3]]
                statuses = []
                for val in values:
                    if pd.isna(val):
                        statuses.append(False)
                    elif threshold_type == 'max':
                        statuses.append(val <= threshold)
                    else:
                        statuses.append(val >= threshold)
                summary_rows.append({
                    'Ковенант': metric_name,
                    'Порог': f"{threshold_type.upper()} {threshold:.2f}",
                    '2025': f"{values[0]:.2f}" if len(values) > 0 and pd.notna(values[0]) else 'N/A',
                    '2026': f"{values[1]:.2f}" if len(values) > 1 and pd.notna(values[1]) else 'N/A',
                    '2027': f"{values[2]:.2f}" if len(values) > 2 and pd.notna(values[2]) else 'N/A',
                    'Статус': '✅ Соблюдено' if all(statuses) else '❌ Нарушение'
                })
        if summary_rows:
            display(pd.DataFrame(summary_rows))
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        for idx, (metric_name, threshold_key, threshold_type, default_threshold) in enumerate(covenant_metrics):
            ax = axes[idx]
            threshold = thresholds.get(threshold_key, default_threshold)
            metric_row = covenants_wide[covenants_wide['metric'].astype(str).str.lower().str.contains(metric_name.lower())] if 'metric' in covenants_wide.columns else pd.DataFrame()
            if metric_row.empty:
                ax.set_visible(False)
                continue
            row = metric_row.iloc[0]
            plot_years = []
            values = []
            for y in years:
                val = pd.to_numeric(row.get(str(y)), errors='coerce')
                if pd.notna(val):
                    plot_years.append(y)
                    values.append(float(val))
            if values:
                ax.plot(plot_years, values, 'o-', linewidth=2, markersize=8, label='Значение')
                ax.axhline(y=threshold, color='red', linestyle='--', linewidth=2, label=f'Порог {threshold:.2f}')
                if threshold_type == 'max':
                    ax.fill_between(plot_years, threshold, max(values+[threshold]), color='red', alpha=0.2)
                else:
                    ax.fill_between(plot_years, min(values+[threshold]), threshold, color='red', alpha=0.2)
                ax.set_title(metric_name)
                ax.set_xlabel('Год')
                ax.set_ylabel('Значение')
                ax.legend()
                ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


## 📊 Детальный анализ моделирования статей: История vs Прогноз <a id="section-4-analysis"></a>

Этот раздел содержит:
- Сравнение истории и прогноза для всех ключевых статей IS/BS/CF
- Графики Actual vs Forecast
- Анализ точности моделирования
- Проверка соответствия историческим трендам


In [ ]:
display(Markdown("### 📈 Анализ Income Statement: История vs Прогноз"))

# Загружаем историю и прогноз IS
hist_is = load_history_from_db('IS', canonical=True)
forecast_is = load_model_forecast_from_db('IS', canonical=True)

if hist_is.empty or forecast_is.empty:
    print("⚠️ Данные IS не найдены")
else:
    # Преобразуем в wide format если нужно
    if 'year' in hist_is.columns and 'metric' in hist_is.columns:
        hist_is_wide = hist_is.pivot(index='metric', columns='year', values='value')
    else:
        hist_is_wide = hist_is.set_index('metric') if 'metric' in hist_is.columns else hist_is
    
    if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
        forecast_is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
    else:
        forecast_is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
    
    # Ключевые метрики IS для анализа
    is_metrics = ['revenue', 'cogs', 'sga', 'ebitda', 'ebit', 'depreciation_owned', 'amortization', 
                  'interest_expense', 'interest_income', 'tax_expense', 'net_income']
    
    # Определяем годы
    hist_year_cols = [c for c in hist_is_wide.columns if str(c).isdigit() and int(c) <= 2024]
    forecast_year_cols = [c for c in forecast_is_wide.columns if str(c).isdigit() and int(c) > 2024]
    all_years = sorted([int(c) for c in hist_year_cols + forecast_year_cols])
    
    # Создаем графики для каждой метрики
    n_metrics = len([m for m in is_metrics if m in hist_is_wide.index or m in forecast_is_wide.index])
    n_cols = 3
    n_rows = (n_metrics + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    plot_idx = 0
    for metric in is_metrics:
        if plot_idx >= len(axes):
            break
        
        ax = axes[plot_idx]
        hist_values = []
        forecast_values = []
        hist_years_plot = []
        forecast_years_plot = []
        
        # История
        if metric in hist_is_wide.index:
            for y in sorted(hist_year_cols):
                val = pd.to_numeric(hist_is_wide.loc[metric, str(y)], errors='coerce')
                if pd.notna(val):
                    hist_values.append(float(val))
                    hist_years_plot.append(int(y))
        
        # Прогноз
        if metric in forecast_is_wide.index:
            for y in sorted(forecast_year_cols):
                val = pd.to_numeric(forecast_is_wide.loc[metric, str(y)], errors='coerce')
                if pd.notna(val):
                    forecast_values.append(float(val))
                    forecast_years_plot.append(int(y))
        
        if hist_values or forecast_values:
            if hist_values:
                ax.plot(hist_years_plot, hist_values, 'o-', linewidth=2, markersize=6, label='История', color='blue')
            if forecast_values:
                ax.plot(forecast_years_plot, forecast_values, 's--', linewidth=2, markersize=6, label='Прогноз', color='red')
            
            # Вертикальная линия разделения истории и прогноза
            if hist_years_plot and forecast_years_plot:
                transition_year = max(hist_years_plot)
                ax.axvline(x=transition_year, color='gray', linestyle=':', linewidth=1, alpha=0.5)
            
            ax.set_title(metric.replace('_', ' ').title())
            ax.set_xlabel('Год')
            ax.set_ylabel('$ millions')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plot_idx += 1
    
    # Скрываем пустые subplots
    for idx in range(plot_idx, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Создаем сводную таблицу сравнения
    display(Markdown("#### 📊 Сводная таблица: История vs Прогноз (последние 3 года истории + прогноз)"))
    
    comparison_data = []
    last_3_years = sorted([int(y) for y in hist_year_cols])[-3:]
    first_3_forecast_years = sorted([int(y) for y in forecast_year_cols])[:3]
    
    for metric in is_metrics:
        row = {'Метрика': metric.replace('_', ' ').title()}
        
        # История
        for y in last_3_years:
            if metric in hist_is_wide.index:
                val = pd.to_numeric(hist_is_wide.loc[metric, str(y)], errors='coerce')
                row[f'{y} (история)'] = f"${val:,.0f}" if pd.notna(val) else "N/A"
            else:
                row[f'{y} (история)'] = "N/A"
        
        # Прогноз
        for y in first_3_forecast_years:
            if metric in forecast_is_wide.index:
                val = pd.to_numeric(forecast_is_wide.loc[metric, str(y)], errors='coerce')
                row[f'{y} (прогноз)'] = f"${val:,.0f}" if pd.notna(val) else "N/A"
            else:
                row[f'{y} (прогноз)'] = "N/A"
        
        comparison_data.append(row)
    
    comparison_df = pd.DataFrame(comparison_data)
    display(comparison_df)


In [ ]:
display(Markdown("### 🏦 Анализ Balance Sheet: История vs Прогноз"))

# Загружаем историю и прогноз BS
hist_bs = load_history_from_db('BS', canonical=True)
forecast_bs = load_model_forecast_from_db('BS', canonical=True)

if hist_bs.empty or forecast_bs.empty:
    print("⚠️ Данные BS не найдены")
else:
    # Преобразуем в wide format если нужно
    if 'year' in hist_bs.columns and 'metric' in hist_bs.columns:
        hist_bs_wide = hist_bs.pivot(index='metric', columns='year', values='value')
    else:
        hist_bs_wide = hist_bs.set_index('metric') if 'metric' in hist_bs.columns else hist_bs
    
    if 'year' in forecast_bs.columns and 'metric' in forecast_bs.columns:
        forecast_bs_wide = forecast_bs.pivot(index='metric', columns='year', values='value')
    else:
        forecast_bs_wide = forecast_bs.set_index('metric') if 'metric' in forecast_bs.columns else forecast_bs
    
    # Ключевые метрики BS для анализа
    bs_metrics = ['cash', 'accounts_receivable', 'inventory', 'ppe_net', 'ppe_gross', 'ppe_accum_dep',
                  'intangibles', 'total_assets', 'accounts_payable', 'short_term_debt', 'long_term_debt',
                  'total_liabilities', 'retained_earnings', 'total_equity', 'total_liab_equity']
    
    # Определяем годы
    hist_year_cols = [c for c in hist_bs_wide.columns if str(c).isdigit() and int(c) <= 2024]
    forecast_year_cols = [c for c in forecast_bs_wide.columns if str(c).isdigit() and int(c) > 2024]
    
    # Создаем графики для каждой метрики
    n_metrics = len([m for m in bs_metrics if m in hist_bs_wide.index or m in forecast_bs_wide.index])
    n_cols = 3
    n_rows = (n_metrics + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    plot_idx = 0
    for metric in bs_metrics:
        if plot_idx >= len(axes):
            break
        
        ax = axes[plot_idx]
        hist_values = []
        forecast_values = []
        hist_years_plot = []
        forecast_years_plot = []
        
        # История
        if metric in hist_bs_wide.index:
            for y in sorted(hist_year_cols):
                val = pd.to_numeric(hist_bs_wide.loc[metric, str(y)], errors='coerce')
                if pd.notna(val):
                    hist_values.append(float(val))
                    hist_years_plot.append(int(y))
        
        # Прогноз
        if metric in forecast_bs_wide.index:
            for y in sorted(forecast_year_cols):
                val = pd.to_numeric(forecast_bs_wide.loc[metric, str(y)], errors='coerce')
                if pd.notna(val):
                    forecast_values.append(float(val))
                    forecast_years_plot.append(int(y))
        
        if hist_values or forecast_values:
            if hist_values:
                ax.plot(hist_years_plot, hist_values, 'o-', linewidth=2, markersize=6, label='История', color='blue')
            if forecast_values:
                ax.plot(forecast_years_plot, forecast_values, 's--', linewidth=2, markersize=6, label='Прогноз', color='red')
            
            # Вертикальная линия разделения истории и прогноза
            if hist_years_plot and forecast_years_plot:
                transition_year = max(hist_years_plot)
                ax.axvline(x=transition_year, color='gray', linestyle=':', linewidth=1, alpha=0.5)
            
            ax.set_title(metric.replace('_', ' ').title())
            ax.set_xlabel('Год')
            ax.set_ylabel('$ millions')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plot_idx += 1
    
    # Скрываем пустые subplots
    for idx in range(plot_idx, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Проверка баланса BS
    display(Markdown("#### ✅ Проверка баланса BS (Assets = Liabilities + Equity)"))
    
    balance_check = []
    for y in sorted(forecast_year_cols)[:5]:  # Первые 5 лет прогноза
        y_str = str(y)
        if 'total_assets' in forecast_bs_wide.index and 'total_liab_equity' in forecast_bs_wide.index:
            assets = pd.to_numeric(forecast_bs_wide.loc['total_assets', y_str], errors='coerce')
            liab_equity = pd.to_numeric(forecast_bs_wide.loc['total_liab_equity', y_str], errors='coerce')
            if pd.notna(assets) and pd.notna(liab_equity):
                diff = abs(assets - liab_equity)
                diff_pct = (diff / assets * 100) if assets != 0 else 0
                balance_check.append({
                    'Год': y,
                    'Assets': f"${assets:,.0f}",
                    'Liab+Equity': f"${liab_equity:,.0f}",
                    'Разница': f"${diff:,.0f}",
                    'Разница %': f"{diff_pct:.4f}%",
                    'Статус': '✅' if diff < 1.0 else '⚠️'
                })
    
    if balance_check:
        balance_df = pd.DataFrame(balance_check)
        display(balance_df)


In [ ]:
display(Markdown("### 💰 Анализ Cash Flow Statement: История vs Прогноз"))

# Загружаем историю и прогноз CF
hist_cf = load_history_from_db('CF', canonical=True)
forecast_cf = load_model_forecast_from_db('CF', canonical=True)

if hist_cf.empty or forecast_cf.empty:
    print("⚠️ Данные CF не найдены")
else:
    # Преобразуем в wide format если нужно
    if 'year' in hist_cf.columns and 'metric' in hist_cf.columns:
        hist_cf_wide = hist_cf.pivot(index='metric', columns='year', values='value')
    else:
        hist_cf_wide = hist_cf.set_index('metric') if 'metric' in hist_cf.columns else hist_cf
    
    if 'year' in forecast_cf.columns and 'metric' in forecast_cf.columns:
        forecast_cf_wide = forecast_cf.pivot(index='metric', columns='year', values='value')
    else:
        forecast_cf_wide = forecast_cf.set_index('metric') if 'metric' in forecast_cf.columns else forecast_cf
    
    # Ключевые метрики CF для анализа
    cf_metrics = ['cfo', 'cfi', 'cff', 'net_change', 'capex', 'depreciation', 'amortization',
                  'wc_delta', 'tax_paid', 'interest_paid']
    
    # Определяем годы
    hist_year_cols = [c for c in hist_cf_wide.columns if str(c).isdigit() and int(c) <= 2024]
    forecast_year_cols = [c for c in forecast_cf_wide.columns if str(c).isdigit() and int(c) > 2024]
    
    # Создаем графики для каждой метрики
    n_metrics = len([m for m in cf_metrics if m in hist_cf_wide.index or m in forecast_cf_wide.index])
    n_cols = 3
    n_rows = (n_metrics + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    plot_idx = 0
    for metric in cf_metrics:
        if plot_idx >= len(axes):
            break
        
        ax = axes[plot_idx]
        hist_values = []
        forecast_values = []
        hist_years_plot = []
        forecast_years_plot = []
        
        # История
        if metric in hist_cf_wide.index:
            for y in sorted(hist_year_cols):
                val = pd.to_numeric(hist_cf_wide.loc[metric, str(y)], errors='coerce')
                if pd.notna(val):
                    hist_values.append(float(val))
                    hist_years_plot.append(int(y))
        
        # Прогноз
        if metric in forecast_cf_wide.index:
            for y in sorted(forecast_year_cols):
                val = pd.to_numeric(forecast_cf_wide.loc[metric, str(y)], errors='coerce')
                if pd.notna(val):
                    forecast_values.append(float(val))
                    forecast_years_plot.append(int(y))
        
        if hist_values or forecast_values:
            if hist_values:
                ax.plot(hist_years_plot, hist_values, 'o-', linewidth=2, markersize=6, label='История', color='blue')
            if forecast_values:
                ax.plot(forecast_years_plot, forecast_values, 's--', linewidth=2, markersize=6, label='Прогноз', color='red')
            
            # Вертикальная линия разделения истории и прогноза
            if hist_years_plot and forecast_years_plot:
                transition_year = max(hist_years_plot)
                ax.axvline(x=transition_year, color='gray', linestyle=':', linewidth=1, alpha=0.5)
            
            # Горизонтальная линия на нуле для CF метрик
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
            
            ax.set_title(metric.replace('_', ' ').title())
            ax.set_xlabel('Год')
            ax.set_ylabel('$ millions')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plot_idx += 1
    
    # Скрываем пустые subplots
    for idx in range(plot_idx, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Проверка Cash Bridge
    display(Markdown("#### ✅ Проверка Cash Bridge (Opening + CFO + CFI + CFF = Closing)"))
    
    # Загружаем BS для проверки Cash Bridge
    hist_bs_temp = load_history_from_db('BS', canonical=True)
    forecast_bs_temp = load_model_forecast_from_db('BS', canonical=True)
    
    if not forecast_bs_temp.empty:
        if 'year' in forecast_bs_temp.columns and 'metric' in forecast_bs_temp.columns:
            forecast_bs_wide_temp = forecast_bs_temp.pivot(index='metric', columns='year', values='value')
        else:
            forecast_bs_wide_temp = forecast_bs_temp.set_index('metric') if 'metric' in forecast_bs_temp.columns else forecast_bs_temp
        
        if 'year' in hist_bs_temp.columns and 'metric' in hist_bs_temp.columns:
            hist_bs_wide_temp = hist_bs_temp.pivot(index='metric', columns='year', values='value')
        else:
            hist_bs_wide_temp = hist_bs_temp.set_index('metric') if 'metric' in hist_bs_temp.columns else hist_bs_temp
        
        hist_year_cols_bs = [c for c in hist_bs_wide_temp.columns if str(c).isdigit() and int(c) <= 2024]
    
    cash_bridge_check = []
    if not forecast_bs_temp.empty and 'cash_opening' in forecast_bs_wide_temp.index and 'cash' in forecast_bs_wide_temp.index:
        for i, y in enumerate(sorted(forecast_year_cols)[:5]):
            y_str = str(y)
            prev_y_str = str(y - 1) if i > 0 else str(max([int(c) for c in hist_year_cols_bs]) if hist_year_cols_bs else y - 1)
            
            opening = pd.to_numeric(forecast_bs_wide_temp.loc['cash_opening', y_str] if 'cash_opening' in forecast_bs_wide_temp.index else 
                                    forecast_bs_wide_temp.loc['cash', prev_y_str] if prev_y_str in forecast_bs_wide_temp.columns else None, errors='coerce')
            cfo = pd.to_numeric(forecast_cf_wide.loc['cfo', y_str], errors='coerce') if 'cfo' in forecast_cf_wide.index else None
            cfi = pd.to_numeric(forecast_cf_wide.loc['cfi', y_str], errors='coerce') if 'cfi' in forecast_cf_wide.index else None
            cff = pd.to_numeric(forecast_cf_wide.loc['cff', y_str], errors='coerce') if 'cff' in forecast_cf_wide.index else None
            closing = pd.to_numeric(forecast_bs_wide_temp.loc['cash', y_str], errors='coerce')
            
            if all(pd.notna(v) for v in [opening, cfo, cfi, cff, closing] if v is not None):
                calculated_closing = opening + (cfo or 0) + (cfi or 0) + (cff or 0)
                diff = abs(closing - calculated_closing)
                cash_bridge_check.append({
                    'Год': y,
                    'Opening': f"${opening:,.0f}",
                    'CFO': f"${cfo:,.0f}" if cfo else "N/A",
                    'CFI': f"${cfi:,.0f}" if cfi else "N/A",
                    'CFF': f"${cff:,.0f}" if cff else "N/A",
                    'Calculated Closing': f"${calculated_closing:,.0f}",
                    'Actual Closing': f"${closing:,.0f}",
                    'Разница': f"${diff:,.0f}",
                    'Статус': '✅' if diff < 1.0 else '⚠️'
                })
    
    if cash_bridge_check:
        bridge_df = pd.DataFrame(cash_bridge_check)
        display(bridge_df)


In [ ]:
display(Markdown("### 📊 Анализ ключевых финансовых метрик и маржинальности"))

# Анализ маржинальности и ключевых коэффициентов
if not hist_is.empty and not forecast_is.empty:
    # Преобразуем в wide format если нужно
    if 'year' in hist_is.columns and 'metric' in hist_is.columns:
        hist_is_wide = hist_is.pivot(index='metric', columns='year', values='value')
    else:
        hist_is_wide = hist_is.set_index('metric') if 'metric' in hist_is.columns else hist_is
    
    if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
        forecast_is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
    else:
        forecast_is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
    
    hist_year_cols = [c for c in hist_is_wide.columns if str(c).isdigit() and int(c) <= 2024]
    forecast_year_cols = [c for c in forecast_is_wide.columns if str(c).isdigit() and int(c) > 2024]
    all_years = sorted([int(c) for c in hist_year_cols + forecast_year_cols])
    
    # Рассчитываем маржинальность
    metrics_data = []
    
    for y in all_years:
        y_str = str(y)
        is_hist = y <= 2024
        
        revenue = pd.to_numeric(hist_is_wide.loc['revenue', y_str] if is_hist and 'revenue' in hist_is_wide.index else 
                               forecast_is_wide.loc['revenue', y_str] if not is_hist and 'revenue' in forecast_is_wide.index else None, errors='coerce')
        cogs = pd.to_numeric(hist_is_wide.loc['cogs', y_str] if is_hist and 'cogs' in hist_is_wide.index else 
                             forecast_is_wide.loc['cogs', y_str] if not is_hist and 'cogs' in forecast_is_wide.index else None, errors='coerce')
        ebitda = pd.to_numeric(hist_is_wide.loc['ebitda', y_str] if is_hist and 'ebitda' in hist_is_wide.index else 
                              forecast_is_wide.loc['ebitda', y_str] if not is_hist and 'ebitda' in forecast_is_wide.index else None, errors='coerce')
        ebit = pd.to_numeric(hist_is_wide.loc['ebit', y_str] if is_hist and 'ebit' in hist_is_wide.index else 
                            forecast_is_wide.loc['ebit', y_str] if not is_hist and 'ebit' in forecast_is_wide.index else None, errors='coerce')
        net_income = pd.to_numeric(hist_is_wide.loc['net_income', y_str] if is_hist and 'net_income' in hist_is_wide.index else 
                                   forecast_is_wide.loc['net_income', y_str] if not is_hist and 'net_income' in forecast_is_wide.index else None, errors='coerce')
        
        if pd.notna(revenue) and revenue > 0:
            gross_margin = ((revenue - cogs) / revenue * 100) if pd.notna(cogs) else None
            ebitda_margin = (ebitda / revenue * 100) if pd.notna(ebitda) else None
            ebit_margin = (ebit / revenue * 100) if pd.notna(ebit) else None
            net_margin = (net_income / revenue * 100) if pd.notna(net_income) else None
            
            metrics_data.append({
                'Год': y,
                'Тип': 'История' if is_hist else 'Прогноз',
                'Revenue': f"${revenue:,.0f}",
                'Gross Margin %': f"{gross_margin:.1f}%" if gross_margin else "N/A",
                'EBITDA Margin %': f"{ebitda_margin:.1f}%" if ebitda_margin else "N/A",
                'EBIT Margin %': f"{ebit_margin:.1f}%" if ebit_margin else "N/A",
                'Net Margin %': f"{net_margin:.1f}%" if net_margin else "N/A"
            })
    
    if metrics_data:
        metrics_df = pd.DataFrame(metrics_data)
        display(metrics_df)
        
        # Графики маржинальности
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # Gross Margin
        hist_years_m = [r['Год'] for r in metrics_data if r['Тип'] == 'История']
        forecast_years_m = [r['Год'] for r in metrics_data if r['Тип'] == 'Прогноз']
        gross_margins_hist = [float(r['Gross Margin %'].replace('%', '')) for r in metrics_data if r['Тип'] == 'История' and r['Gross Margin %'] != 'N/A']
        gross_margins_frc = [float(r['Gross Margin %'].replace('%', '')) for r in metrics_data if r['Тип'] == 'Прогноз' and r['Gross Margin %'] != 'N/A']
        
        axes[0, 0].plot(hist_years_m[:len(gross_margins_hist)], gross_margins_hist, 'o-', linewidth=2, markersize=6, label='История', color='blue')
        if gross_margins_frc:
            axes[0, 0].plot(forecast_years_m[:len(gross_margins_frc)], gross_margins_frc, 's--', linewidth=2, markersize=6, label='Прогноз', color='red')
        if hist_years_m and forecast_years_m:
            axes[0, 0].axvline(x=max(hist_years_m), color='gray', linestyle=':', linewidth=1, alpha=0.5)
        axes[0, 0].set_title('Gross Margin %')
        axes[0, 0].set_xlabel('Год')
        axes[0, 0].set_ylabel('%')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # EBITDA Margin
        ebitda_margins_hist = [float(r['EBITDA Margin %'].replace('%', '')) for r in metrics_data if r['Тип'] == 'История' and r['EBITDA Margin %'] != 'N/A']
        ebitda_margins_frc = [float(r['EBITDA Margin %'].replace('%', '')) for r in metrics_data if r['Тип'] == 'Прогноз' and r['EBITDA Margin %'] != 'N/A']
        
        axes[0, 1].plot(hist_years_m[:len(ebitda_margins_hist)], ebitda_margins_hist, 'o-', linewidth=2, markersize=6, label='История', color='blue')
        if ebitda_margins_frc:
            axes[0, 1].plot(forecast_years_m[:len(ebitda_margins_frc)], ebitda_margins_frc, 's--', linewidth=2, markersize=6, label='Прогноз', color='red')
        if hist_years_m and forecast_years_m:
            axes[0, 1].axvline(x=max(hist_years_m), color='gray', linestyle=':', linewidth=1, alpha=0.5)
        axes[0, 1].set_title('EBITDA Margin %')
        axes[0, 1].set_xlabel('Год')
        axes[0, 1].set_ylabel('%')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # EBIT Margin
        ebit_margins_hist = [float(r['EBIT Margin %'].replace('%', '')) for r in metrics_data if r['Тип'] == 'История' and r['EBIT Margin %'] != 'N/A']
        ebit_margins_frc = [float(r['EBIT Margin %'].replace('%', '')) for r in metrics_data if r['Тип'] == 'Прогноз' and r['EBIT Margin %'] != 'N/A']
        
        axes[1, 0].plot(hist_years_m[:len(ebit_margins_hist)], ebit_margins_hist, 'o-', linewidth=2, markersize=6, label='История', color='blue')
        if ebit_margins_frc:
            axes[1, 0].plot(forecast_years_m[:len(ebit_margins_frc)], ebit_margins_frc, 's--', linewidth=2, markersize=6, label='Прогноз', color='red')
        if hist_years_m and forecast_years_m:
            axes[1, 0].axvline(x=max(hist_years_m), color='gray', linestyle=':', linewidth=1, alpha=0.5)
        axes[1, 0].set_title('EBIT Margin %')
        axes[1, 0].set_xlabel('Год')
        axes[1, 0].set_ylabel('%')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Net Margin
        net_margins_hist = [float(r['Net Margin %'].replace('%', '')) for r in metrics_data if r['Тип'] == 'История' and r['Net Margin %'] != 'N/A']
        net_margins_frc = [float(r['Net Margin %'].replace('%', '')) for r in metrics_data if r['Тип'] == 'Прогноз' and r['Net Margin %'] != 'N/A']
        
        axes[1, 1].plot(hist_years_m[:len(net_margins_hist)], net_margins_hist, 'o-', linewidth=2, markersize=6, label='История', color='blue')
        if net_margins_frc:
            axes[1, 1].plot(forecast_years_m[:len(net_margins_frc)], net_margins_frc, 's--', linewidth=2, markersize=6, label='Прогноз', color='red')
        if hist_years_m and forecast_years_m:
            axes[1, 1].axvline(x=max(hist_years_m), color='gray', linestyle=':', linewidth=1, alpha=0.5)
        axes[1, 1].set_title('Net Margin %')
        axes[1, 1].set_xlabel('Год')
        axes[1, 1].set_ylabel('%')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ Данные IS не найдены для анализа маржинальности")


## ✅ Комплексное тестирование модели: Полнота, Корректность, Логика <a id="section-testing"></a>

Этот раздел содержит систематические тесты для проверки:
- ✅ Полнота данных: все метрики присутствуют в прогнозе
- ✅ Корректность corkscrews: PP&E, Intangibles, Debt, WC, Tax, Lease
- ✅ Финансовые связи: BS Identity, Cash Bridge, Retained Earnings
- ✅ Соответствие историческим трендам
- ✅ Наличие драйверов для всех статей
- ✅ Логика моделирования работает корректно
- ✅ Контрольные точки пройдены


In [ ]:
display(Markdown("### ✅ Тест 1: Полнота данных - проверка наличия всех метрик в прогнозе"))

# Загружаем прогнозы
forecast_is = load_model_forecast_from_db('IS', canonical=True)
forecast_bs = load_model_forecast_from_db('BS', canonical=True)
forecast_cf = load_model_forecast_from_db('CF', canonical=True)

# Критические метрики, которые должны быть в прогнозе
critical_metrics = {
    'IS': ['revenue', 'cogs', 'sga', 'ebitda', 'ebit', 'depreciation_owned', 'amortization', 
           'interest_expense', 'interest_income', 'tax_expense', 'net_income'],
    'BS': ['cash', 'accounts_receivable', 'inventory', 'ppe_net', 'ppe_gross', 'ppe_accum_dep',
           'intangibles', 'total_assets', 'accounts_payable', 'short_term_debt', 'long_term_debt',
           'total_liabilities', 'retained_earnings', 'total_equity', 'total_liab_equity'],
    'CF': ['cfo', 'cfi', 'cff', 'net_change', 'capex', 'depreciation', 'amortization', 'wc_delta']
}

completeness_results = []

for stmt_type, metrics in critical_metrics.items():
    forecast_df = forecast_is if stmt_type == 'IS' else (forecast_bs if stmt_type == 'BS' else forecast_cf)
    
    if forecast_df.empty:
        completeness_results.append({
            'Отчет': stmt_type,
            'Статус': '❌',
            'Детали': 'Прогноз отсутствует'
        })
        continue
    
    # Преобразуем в wide format если нужно
    if 'year' in forecast_df.columns and 'metric' in forecast_df.columns:
        forecast_wide = forecast_df.pivot(index='metric', columns='year', values='value')
    else:
        forecast_wide = forecast_df.set_index('metric') if 'metric' in forecast_df.columns else forecast_df
    
    forecast_year_cols = [c for c in forecast_wide.columns if str(c).isdigit() and int(c) > 2024]
    
    if not forecast_year_cols:
        completeness_results.append({
            'Отчет': stmt_type,
            'Статус': '❌',
            'Детали': 'Нет прогнозных лет'
        })
        continue
    
    missing_metrics = []
    partial_metrics = []
    complete_metrics = []
    
    for metric in metrics:
        if metric not in forecast_wide.index:
            missing_metrics.append(metric)
        else:
            # Проверяем наличие значений для всех прогнозных лет
            values_count = 0
            for y in forecast_year_cols:
                val = pd.to_numeric(forecast_wide.loc[metric, str(y)], errors='coerce')
                if pd.notna(val) and val != 0:
                    values_count += 1
            
            if values_count == 0:
                missing_metrics.append(metric)
            elif values_count < len(forecast_year_cols):
                partial_metrics.append(f"{metric} ({values_count}/{len(forecast_year_cols)} лет)")
            else:
                complete_metrics.append(metric)
    
    status = '✅' if not missing_metrics and not partial_metrics else ('⚠️' if partial_metrics else '❌')
    details = f"Полных: {len(complete_metrics)}, Частичных: {len(partial_metrics)}, Отсутствует: {len(missing_metrics)}"
    if missing_metrics:
        details += f" | Отсутствуют: {', '.join(missing_metrics[:5])}"
    
    completeness_results.append({
        'Отчет': stmt_type,
        'Статус': status,
        'Детали': details
    })

completeness_df = pd.DataFrame(completeness_results)
display(completeness_df)

# Детальная таблица по метрикам
display(Markdown("#### 📊 Детальная проверка метрик"))
detailed_check = []
for stmt_type, metrics in critical_metrics.items():
    forecast_df = forecast_is if stmt_type == 'IS' else (forecast_bs if stmt_type == 'BS' else forecast_cf)
    if forecast_df.empty:
        continue
    
    if 'year' in forecast_df.columns and 'metric' in forecast_df.columns:
        forecast_wide = forecast_df.pivot(index='metric', columns='year', values='value')
    else:
        forecast_wide = forecast_df.set_index('metric') if 'metric' in forecast_df.columns else forecast_df
    
    forecast_year_cols = sorted([int(c) for c in forecast_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    for metric in metrics:
        row = {'Отчет': stmt_type, 'Метрика': metric}
        if metric in forecast_wide.index:
            for y in forecast_year_cols[:3]:  # Первые 3 года прогноза
                val = pd.to_numeric(forecast_wide.loc[metric, str(y)], errors='coerce')
                row[f'{y}'] = '✅' if pd.notna(val) else '❌'
        else:
            for y in forecast_year_cols[:3]:
                row[f'{y}'] = '❌'
        detailed_check.append(row)

if detailed_check:
    detailed_df = pd.DataFrame(detailed_check)
    display(detailed_df.head(30))  # Показываем первые 30 строк


In [ ]:
display(Markdown("### ✅ Тест 2: Корректность Corkscrews - проверка формул"))

corkscrew_tests = []

# 1. PP&E Corkscrew: PP&E_Net(t) = PP&E_Gross(t) - PP&E_AccumDep(t)
if not forecast_bs.empty:
    if 'year' in forecast_bs.columns and 'metric' in forecast_bs.columns:
        bs_wide = forecast_bs.pivot(index='metric', columns='year', values='value')
    else:
        bs_wide = forecast_bs.set_index('metric') if 'metric' in forecast_bs.columns else forecast_bs
    
    forecast_year_cols = sorted([int(c) for c in bs_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    if 'ppe_net' in bs_wide.index and 'ppe_gross' in bs_wide.index and 'ppe_accum_dep' in bs_wide.index:
        ppe_errors = []
        for y in forecast_year_cols[:5]:
            y_str = str(y)
            ppe_net = pd.to_numeric(bs_wide.loc['ppe_net', y_str], errors='coerce')
            ppe_gross = pd.to_numeric(bs_wide.loc['ppe_gross', y_str], errors='coerce')
            ppe_accum = pd.to_numeric(bs_wide.loc['ppe_accum_dep', y_str], errors='coerce')
            
            if all(pd.notna(v) for v in [ppe_net, ppe_gross, ppe_accum]):
                calculated_net = ppe_gross - ppe_accum
                diff = abs(ppe_net - calculated_net)
                if diff > 1.0:  # Допустимая погрешность $1M
                    ppe_errors.append(f"{y}: diff=${diff:,.0f}")
        
        corkscrew_tests.append({
            'Corkscrew': 'PP&E',
            'Формула': 'PP&E_Net = PP&E_Gross - PP&E_AccumDep',
            'Статус': '✅' if not ppe_errors else '❌',
            'Ошибки': ', '.join(ppe_errors) if ppe_errors else 'Нет'
        })
    
    # 2. PP&E Corkscrew: PP&E_Gross(t) = PP&E_Gross(t-1) + CapEx - Disposals
    if 'ppe_gross' in bs_wide.index:
        ppe_gross_errors = []
        capex_df = load_model_table('CAPEX_BREAKDOWN', csv_filename='outputs/model/capex_breakdown.csv')
        
        if not capex_df.empty:
            if 'year' in capex_df.columns and 'metric' in capex_df.columns:
                capex_wide = capex_df.pivot(index='metric', columns='year', values='value')
            else:
                capex_wide = capex_df.set_index('metric') if 'metric' in capex_df.columns else capex_df
            
            # Загружаем историю для первого года
            hist_bs_temp = load_history_from_db('BS', canonical=True)
            if not hist_bs_temp.empty:
                if 'year' in hist_bs_temp.columns and 'metric' in hist_bs_temp.columns:
                    hist_bs_wide_temp = hist_bs_temp.pivot(index='metric', columns='year', values='value')
                else:
                    hist_bs_wide_temp = hist_bs_temp.set_index('metric') if 'metric' in hist_bs_temp.columns else hist_bs_temp
                
                history_end_year = max([int(c) for c in hist_bs_wide_temp.columns if str(c).isdigit() and int(c) <= 2024])
                prev_ppe_gross = pd.to_numeric(hist_bs_wide_temp.loc['ppe_gross', str(history_end_year)], errors='coerce') if 'ppe_gross' in hist_bs_wide_temp.index else None
                
                for i, y in enumerate(forecast_year_cols[:5]):
                    y_str = str(y)
                    curr_ppe_gross = pd.to_numeric(bs_wide.loc['ppe_gross', y_str], errors='coerce')
                    total_capex = pd.to_numeric(capex_wide.loc['total_capex', y_str], errors='coerce') if 'total_capex' in capex_wide.index else None
                    disposals = pd.to_numeric(capex_wide.loc['disposals', y_str], errors='coerce') if 'disposals' in capex_wide.index else 0.0
                    
                    if pd.notna(curr_ppe_gross) and pd.notna(prev_ppe_gross) and pd.notna(total_capex):
                        calculated_gross = prev_ppe_gross + total_capex - (disposals or 0.0)
                        diff = abs(curr_ppe_gross - calculated_gross)
                        if diff > 10.0:  # Допустимая погрешность $10M
                            ppe_gross_errors.append(f"{y}: diff=${diff:,.0f}")
                    
                    prev_ppe_gross = curr_ppe_gross if pd.notna(curr_ppe_gross) else prev_ppe_gross
        
        corkscrew_tests.append({
            'Corkscrew': 'PP&E Gross Roll',
            'Формула': 'PP&E_Gross(t) = PP&E_Gross(t-1) + CapEx - Disposals',
            'Статус': '✅' if not ppe_gross_errors else '⚠️',
            'Ошибки': ', '.join(ppe_gross_errors) if ppe_gross_errors else 'Нет'
        })
    
    # 3. Retained Earnings: RE(t) = RE(t-1) + Net Income - Dividends
    if 'retained_earnings' in bs_wide.index:
        # Загружаем forecast_is и forecast_cf если еще не загружены
        if 'forecast_is' not in locals() or forecast_is.empty:
            forecast_is = load_model_forecast_from_db('IS', canonical=True)
        if 'forecast_cf' not in locals() or forecast_cf.empty:
            forecast_cf = load_model_forecast_from_db('CF', canonical=True)
        
        if not forecast_is.empty:
            if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
                is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
            else:
                is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
            
            if not forecast_cf.empty:
                if 'year' in forecast_cf.columns and 'metric' in forecast_cf.columns:
                    cf_wide = forecast_cf.pivot(index='metric', columns='year', values='value')
                else:
                    cf_wide = forecast_cf.set_index('metric') if 'metric' in forecast_cf.columns else forecast_cf
            else:
                cf_wide = pd.DataFrame()
        
        if 'is_wide' in locals() and 'net_income' in is_wide.index:
            re_errors = []
            # Загружаем историю для первого года
            hist_bs_temp = load_history_from_db('BS', canonical=True)
            if not hist_bs_temp.empty:
                if 'year' in hist_bs_temp.columns and 'metric' in hist_bs_temp.columns:
                    hist_bs_wide_temp = hist_bs_temp.pivot(index='metric', columns='year', values='value')
                else:
                    hist_bs_wide_temp = hist_bs_temp.set_index('metric') if 'metric' in hist_bs_temp.columns else hist_bs_temp
                
                history_end_year = max([int(c) for c in hist_bs_wide_temp.columns if str(c).isdigit() and int(c) <= 2024])
                prev_re = pd.to_numeric(hist_bs_wide_temp.loc['retained_earnings', str(history_end_year)], errors='coerce') if 'retained_earnings' in hist_bs_wide_temp.index else None
                
                for i, y in enumerate(forecast_year_cols[:5]):
                    y_str = str(y)
                    curr_re = pd.to_numeric(bs_wide.loc['retained_earnings', y_str], errors='coerce')
                    ni = pd.to_numeric(is_wide.loc['net_income', y_str], errors='coerce')
                    dividends = pd.to_numeric(forecast_cf_wide.loc['cff_dividends', y_str], errors='coerce') if 'cff_dividends' in forecast_cf_wide.index else 0.0
                    
                    if pd.notna(curr_re) and pd.notna(prev_re) and pd.notna(ni):
                        calculated_re = prev_re + ni - (dividends or 0.0)
                        diff = abs(curr_re - calculated_re)
                        if diff > 1.0:
                            re_errors.append(f"{y}: diff=${diff:,.0f}")
                    
                    prev_re = curr_re if pd.notna(curr_re) else prev_re
        
        corkscrew_tests.append({
            'Corkscrew': 'Retained Earnings',
            'Формула': 'RE(t) = RE(t-1) + Net Income - Dividends',
            'Статус': '✅' if not re_errors else '❌',
            'Ошибки': ', '.join(re_errors) if re_errors else 'Нет'
        })

if corkscrew_tests:
    corkscrew_df = pd.DataFrame(corkscrew_tests)
    display(corkscrew_df)
else:
    print("⚠️ Не удалось проверить corkscrews - данные отсутствуют")


In [ ]:
display(Markdown("### ✅ Тест 3: Финансовые связи - BS Identity, Cash Bridge, Retained Earnings"))

financial_checks = []

# 1. BS Identity: Total Assets = Total Liabilities + Equity
if not forecast_bs.empty:
    if 'year' in forecast_bs.columns and 'metric' in forecast_bs.columns:
        bs_wide = forecast_bs.pivot(index='metric', columns='year', values='value')
    else:
        bs_wide = forecast_bs.set_index('metric') if 'metric' in forecast_bs.columns else forecast_bs
    
    forecast_year_cols = sorted([int(c) for c in bs_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    bs_identity_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        if 'total_assets' in bs_wide.index and 'total_liab_equity' in bs_wide.index:
            assets = pd.to_numeric(bs_wide.loc['total_assets', y_str], errors='coerce')
            liab_equity = pd.to_numeric(bs_wide.loc['total_liab_equity', y_str], errors='coerce')
            
            if pd.notna(assets) and pd.notna(liab_equity):
                diff = abs(assets - liab_equity)
                diff_pct = (diff / assets * 100) if assets != 0 else 0
                if diff > 1.0:  # Допустимая погрешность $1M
                    bs_identity_errors.append(f"{y}: diff=${diff:,.0f} ({diff_pct:.4f}%)")
    
    financial_checks.append({
        'Проверка': 'BS Identity',
        'Формула': 'Total Assets = Total Liabilities + Equity',
        'Статус': '✅' if not bs_identity_errors else '❌',
        'Ошибки': ', '.join(bs_identity_errors) if bs_identity_errors else 'Нет',
        'Годы проверки': len(forecast_year_cols[:5])
    })

# 2. Cash Bridge: ΔCash(BS) = NetChange(CF)
if not forecast_bs.empty and not forecast_cf.empty:
    if 'year' in forecast_cf.columns and 'metric' in forecast_cf.columns:
        cf_wide = forecast_cf.pivot(index='metric', columns='year', values='value')
    else:
        cf_wide = forecast_cf.set_index('metric') if 'metric' in forecast_cf.columns else forecast_cf
    
    cash_bridge_errors = []
    
    # Загружаем историю для первого года
    hist_bs_temp = load_history_from_db('BS', canonical=True)
    if not hist_bs_temp.empty:
        if 'year' in hist_bs_temp.columns and 'metric' in hist_bs_temp.columns:
            hist_bs_wide_temp = hist_bs_temp.pivot(index='metric', columns='year', values='value')
        else:
            hist_bs_wide_temp = hist_bs_temp.set_index('metric') if 'metric' in hist_bs_temp.columns else hist_bs_temp
        
        history_end_year = max([int(c) for c in hist_bs_wide_temp.columns if str(c).isdigit() and int(c) <= 2024])
        prev_cash = pd.to_numeric(hist_bs_wide_temp.loc['cash', str(history_end_year)], errors='coerce') if 'cash' in hist_bs_wide_temp.index else None
        
        for y in forecast_year_cols[:5]:
            y_str = str(y)
            curr_cash = pd.to_numeric(bs_wide.loc['cash', y_str], errors='coerce')
            net_change = pd.to_numeric(cf_wide.loc['net_change', y_str], errors='coerce') if 'net_change' in cf_wide.index else None
            
            if pd.notna(curr_cash) and pd.notna(prev_cash) and pd.notna(net_change):
                cash_delta = curr_cash - prev_cash
                diff = abs(cash_delta - net_change)
                if diff > 1.0:
                    cash_bridge_errors.append(f"{y}: diff=${diff:,.0f} (ΔCash=${cash_delta:,.0f}, NetChange=${net_change:,.0f})")
            
            prev_cash = curr_cash if pd.notna(curr_cash) else prev_cash
    
    financial_checks.append({
        'Проверка': 'Cash Bridge',
        'Формула': 'ΔCash(BS) = NetChange(CF)',
        'Статус': '✅' if not cash_bridge_errors else '❌',
        'Ошибки': ', '.join(cash_bridge_errors) if cash_bridge_errors else 'Нет',
        'Годы проверки': len(forecast_year_cols[:5])
    })

# 3. CFO Formula: CFO = Net Income + D&A - WC Delta - Tax Paid (упрощенная проверка)
# Убеждаемся что is_wide и cf_wide определены
if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)
if 'forecast_cf' not in locals() or forecast_cf.empty:
    forecast_cf = load_model_forecast_from_db('CF', canonical=True)

if not forecast_is.empty and not forecast_cf.empty:
    if 'is_wide' not in locals():
        if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
            is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
        else:
            is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
    
    if 'cf_wide' not in locals():
        if 'year' in forecast_cf.columns and 'metric' in forecast_cf.columns:
            cf_wide = forecast_cf.pivot(index='metric', columns='year', values='value')
        else:
            cf_wide = forecast_cf.set_index('metric') if 'metric' in forecast_cf.columns else forecast_cf
    cfo_formula_errors = []
    
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        ni = pd.to_numeric(is_wide.loc['net_income', y_str], errors='coerce') if 'net_income' in is_wide.index else None
        dep = pd.to_numeric(is_wide.loc['depreciation_owned', y_str], errors='coerce') if 'depreciation_owned' in is_wide.index else 0.0
        amort = pd.to_numeric(is_wide.loc['amortization', y_str], errors='coerce') if 'amortization' in is_wide.index else 0.0
        wc_delta = pd.to_numeric(cf_wide.loc['wc_delta', y_str], errors='coerce') if 'wc_delta' in cf_wide.index else 0.0
        tax_paid = pd.to_numeric(cf_wide.loc['tax_paid', y_str], errors='coerce') if 'tax_paid' in cf_wide.index else 0.0
        cfo = pd.to_numeric(cf_wide.loc['cfo', y_str], errors='coerce') if 'cfo' in cf_wide.index else None
        
        if all(pd.notna(v) for v in [ni, cfo] if v is not None):
            da = (dep or 0.0) + (amort or 0.0)
            calculated_cfo = ni + da - (wc_delta or 0.0) - (tax_paid or 0.0)
            diff = abs(cfo - calculated_cfo)
            if diff > 10.0:  # Допустимая погрешность $10M (учитывая другие компоненты CFO)
                cfo_formula_errors.append(f"{y}: diff=${diff:,.0f}")
    
    financial_checks.append({
        'Проверка': 'CFO Formula',
        'Формула': 'CFO ≈ Net Income + D&A - WC Delta - Tax Paid',
        'Статус': '✅' if not cfo_formula_errors else '⚠️',
        'Ошибки': ', '.join(cfo_formula_errors) if cfo_formula_errors else 'Нет',
        'Годы проверки': len(forecast_year_cols[:5])
    })

if financial_checks:
    financial_df = pd.DataFrame(financial_checks)
    display(financial_df)
else:
    print("⚠️ Не удалось проверить финансовые связи - данные отсутствуют")


In [ ]:
display(Markdown("### ✅ Тест 4: Соответствие историческим трендам"))

# Проверяем, что прогноз не слишком сильно отклоняется от исторических трендов
trend_tests = []

# Загружаем историю если еще не загружена
if 'hist_is' not in locals() or hist_is.empty:
    hist_is = load_history_from_db('IS', canonical=True)
if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)

if not hist_is.empty and not forecast_is.empty:
    # Преобразуем в wide format
    if 'year' in hist_is.columns and 'metric' in hist_is.columns:
        hist_is_wide = hist_is.pivot(index='metric', columns='year', values='value')
    else:
        hist_is_wide = hist_is.set_index('metric') if 'metric' in hist_is.columns else hist_is
    
    if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
        forecast_is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
    else:
        forecast_is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
    
    hist_year_cols = sorted([int(c) for c in hist_is_wide.columns if str(c).isdigit() and int(c) <= 2024])
    forecast_year_cols = sorted([int(c) for c in forecast_is_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    # Ключевые метрики для проверки трендов
    trend_metrics = ['revenue', 'cogs', 'sga', 'ebitda', 'net_income']
    
    for metric in trend_metrics:
        if metric not in hist_is_wide.index or metric not in forecast_is_wide.index:
            continue
        
        # Берем последние 3 года истории
        last_3_years = hist_year_cols[-3:] if len(hist_year_cols) >= 3 else hist_year_cols
        first_forecast_year = forecast_year_cols[0] if forecast_year_cols else None
        
        if not last_3_years or not first_forecast_year:
            continue
        
        # Рассчитываем средний темп роста за последние 3 года
        hist_values = [pd.to_numeric(hist_is_wide.loc[metric, str(y)], errors='coerce') for y in last_3_years]
        hist_values = [v for v in hist_values if pd.notna(v) and v > 0]
        
        if len(hist_values) < 2:
            continue
        
        # CAGR за последние 3 года
        growth_rates = []
        for i in range(1, len(hist_values)):
            if hist_values[i-1] > 0:
                growth = (hist_values[i] / hist_values[i-1]) - 1.0
                growth_rates.append(growth)
        
        avg_growth = np.mean(growth_rates) if growth_rates else 0.0
        
        # Прогнозное значение для первого года
        last_hist_value = hist_values[-1]
        forecast_value = pd.to_numeric(forecast_is_wide.loc[metric, str(first_forecast_year)], errors='coerce')
        
        if pd.notna(forecast_value) and last_hist_value > 0:
            forecast_growth = (forecast_value / last_hist_value) - 1.0
            
            # Проверяем, что прогнозный рост не слишком сильно отклоняется от исторического
            # Допускаем отклонение до 50% от исторического тренда
            max_deviation = abs(avg_growth) * 1.5 if avg_growth != 0 else 1.0
            deviation = abs(forecast_growth - avg_growth)
            
            status = '✅' if deviation <= max_deviation else '⚠️'
            
            trend_tests.append({
                'Метрика': metric.replace('_', ' ').title(),
                'Исторический CAGR (3 года)': f"{avg_growth:.1%}",
                'Прогнозный рост (1-й год)': f"{forecast_growth:.1%}",
                'Отклонение': f"{deviation:.1%}",
                'Статус': status,
                'Примечание': 'Соответствует тренду' if deviation <= max_deviation else 'Отклонение от тренда'
            })

if trend_tests:
    trend_df = pd.DataFrame(trend_tests)
    display(trend_df)
else:
    print("⚠️ Не удалось проверить соответствие трендам - недостаточно данных")


In [ ]:
display(Markdown("### ✅ Тест 5: Наличие драйверов и логики моделирования"))

# Проверяем, что для всех статей есть драйверы или логика моделирования
driver_tests = []

# Загружаем конфигурацию проекта
proj_yaml_path = croot / 'configs' / 'project.yaml'
if proj_yaml_path.exists():
    with open(proj_yaml_path, 'r', encoding='utf-8') as f:
        proj_yaml = yaml.safe_load(f)
    
    model_config = proj_yaml.get('model', {}).get('standard', {})
    
    # Проверяем наличие драйверов для ключевых статей
    driver_checks = {
        'Revenue': {
            'driver': 'macro_forecast' if proj_yaml.get('macro_forecast', {}).get('factors') else None,
            'method': 'Регрессия на макрофакторы' if proj_yaml.get('macro_forecast', {}).get('factors') else 'EWA темп роста',
            'required': True
        },
        'COGS': {
            'driver': model_config.get('cogs', {}).get('use_ppi_uplift', False),
            'method': f"% от Revenue × PPI uplift" if model_config.get('cogs', {}).get('use_ppi_uplift') else '% от Revenue',
            'required': True
        },
        'SG&A': {
            'driver': model_config.get('sgna', {}).get('index_by_cpi', False),
            'method': f"EWA темп + CPI индексация" if model_config.get('sgna', {}).get('index_by_cpi') else 'EWA темп роста',
            'required': True
        },
        'CapEx': {
            'driver': model_config.get('capex', {}).get('method', 'ratio_to_revenue'),
            'method': f"Метод: {model_config.get('capex', {}).get('method', 'ratio_to_revenue')}",
            'required': True
        },
        'Depreciation': {
            'driver': proj_yaml.get('features', {}).get('use_ppe_corkscrew', False),
            'method': 'PP&E corkscrew' if proj_yaml.get('features', {}).get('use_ppe_corkscrew') else '% метод',
            'required': True
        },
        'Working Capital': {
            'driver': proj_yaml.get('features', {}).get('use_wc_days', False),
            'method': 'Days method (DSO/DIO/DPO)' if proj_yaml.get('features', {}).get('use_wc_days') else '% метод',
            'required': True
        },
        'Debt': {
            'driver': proj_yaml.get('features', {}).get('use_debt_rc', False),
            'method': 'Debt/RC schedule с итерациями' if proj_yaml.get('features', {}).get('use_debt_rc') else 'Упрощенная логика',
            'required': True
        },
        'Tax': {
            'driver': proj_yaml.get('features', {}).get('use_tax_schedule', False),
            'method': 'Tax schedule (DTA/DTL)' if proj_yaml.get('features', {}).get('use_tax_schedule') else 'Effective rate',
            'required': True
        }
    }
    
    for article, check_info in driver_checks.items():
        has_driver = check_info['driver'] is not None and check_info['driver'] != False
        status = '✅' if has_driver or check_info['method'] else '⚠️'
        
        driver_tests.append({
            'Статья': article,
            'Драйвер/Логика': 'Есть' if has_driver else 'Базовая логика',
            'Метод': check_info['method'],
            'Статус': status
        })
    
    driver_df = pd.DataFrame(driver_tests)
    display(driver_df)
    
    # Проверяем наличие макрофакторов для Revenue
    display(Markdown("#### 📊 Проверка макрофакторов для Revenue"))
    
    macro_factors = proj_yaml.get('macro_forecast', {}).get('factors', [])
    if macro_factors:
        macro_check = []
        macro_forecast_dir = croot / 'outputs' / 'macro_forecast'
        
        for factor in macro_factors[:10]:  # Первые 10 факторов
            forecast_file = macro_forecast_dir / f"{factor}_forecast.csv"
            fitted_file = macro_forecast_dir / f"{factor}_ecm_fitted_*.csv"
            has_forecast = forecast_file.exists() or len(list(macro_forecast_dir.glob(f"{factor}_ecm_fitted_*.csv"))) > 0
            
            macro_check.append({
                'Фактор': factor,
                'Прогноз': '✅' if has_forecast else '❌',
                'Файл': forecast_file.name if forecast_file.exists() else 'Не найден'
            })
        
        macro_df = pd.DataFrame(macro_check)
        display(macro_df)
    else:
        print("⚠️ Макрофакторы не настроены в project.yaml")
else:
    print("⚠️ Не удалось загрузить конфигурацию проекта")


In [ ]:
display(Markdown("### ✅ Тест 6: Контрольные точки (Acceptance Checks)"))

# Запускаем acceptance checks если они еще не выполнены
from engine.acceptance.checks import run_acceptance

print("Запуск acceptance checks...")
try:
    run_acceptance(croot)
    print("✅ Acceptance checks выполнены")
except Exception as e:
    print(f"⚠️ Ошибка выполнения acceptance checks: {e}")

# Загружаем результаты acceptance checks
acceptance_df = pd.DataFrame()
mart = _open_data_mart()
if mart is not None:
    try:
        acceptance_df = mart.get_output('acceptance_checks')
        if not acceptance_df.empty:
            print("✅ Acceptance checks загружены из витрины")
    except Exception as e:
        print(f"⚠️ Не удалось загрузить acceptance checks из витрины: {e}")
    finally:
        mart.close()

if acceptance_df.empty:
    acceptance_path = croot / 'outputs' / 'checks' / 'acceptance_checks.csv'
    if acceptance_path.exists():
        acceptance_df = pd.read_csv(acceptance_path)
        print("✅ Acceptance checks загружены из CSV")

if not acceptance_df.empty:
    # Группируем по типам проверок
    if 'check' in acceptance_df.columns and 'ok' in acceptance_df.columns:
        check_summary = acceptance_df.groupby('check').agg({
            'ok': ['sum', 'count']
        }).reset_index()
        check_summary.columns = ['Проверка', 'Успешно', 'Всего']
        check_summary['Процент успеха'] = (check_summary['Успешно'] / check_summary['Всего'] * 100).round(1)
        check_summary['Статус'] = check_summary.apply(
            lambda row: '✅' if row['Процент успеха'] == 100.0 else ('⚠️' if row['Процент успеха'] >= 80.0 else '❌'),
            axis=1
        )
        
        display(check_summary)
        
        # Детальная таблица по годам
        display(Markdown("#### 📊 Детальные результаты по годам"))
        if 'year' in acceptance_df.columns:
            pivot_checks = acceptance_df.pivot_table(
                index='check', 
                columns='year', 
                values='ok', 
                aggfunc='first'
            )
            # Заменяем True/False на ✅/❌
            pivot_checks = pivot_checks.replace({True: '✅', False: '❌'})
            display(pivot_checks)
    else:
        display(acceptance_df.head(20))
else:
    print("⚠️ Acceptance checks не найдены. Запустите build_model() для генерации.")


In [ ]:
display(Markdown("### ✅ Тест 7: Проверка логики моделирования - формулы и связи"))

logic_tests = []

# Убеждаемся что данные загружены
if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)

if not forecast_is.empty:
    if 'is_wide' not in locals():
        if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
            is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
        else:
            is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
    
    forecast_year_cols = sorted([int(c) for c in is_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    # Загружаем конфигурацию если еще не загружена
    if 'proj_yaml' not in locals():
        proj_yaml_path = croot / 'configs' / 'project.yaml'
        if proj_yaml_path.exists():
            with open(proj_yaml_path, 'r', encoding='utf-8') as f:
                proj_yaml = yaml.safe_load(f)
        else:
            proj_yaml = {}
    
    # 1. EBITDA = Revenue - COGS - SG&A
    ebitda_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        revenue = pd.to_numeric(is_wide.loc['revenue', y_str], errors='coerce')
        cogs = pd.to_numeric(is_wide.loc['cogs', y_str], errors='coerce')
        sga = pd.to_numeric(is_wide.loc['sga', y_str], errors='coerce')
        ebitda = pd.to_numeric(is_wide.loc['ebitda', y_str], errors='coerce')
        
        if all(pd.notna(v) for v in [revenue, cogs, sga, ebitda]):
            calculated_ebitda = revenue - cogs - sga
            diff = abs(ebitda - calculated_ebitda)
            if diff > 1.0:
                ebitda_errors.append(f"{y}: diff=${diff:,.0f}")
    
    logic_tests.append({
        'Формула': 'EBITDA = Revenue - COGS - SG&A',
        'Статус': '✅' if not ebitda_errors else '❌',
        'Ошибки': ', '.join(ebitda_errors) if ebitda_errors else 'Нет'
    })

# 2. EBIT = EBITDA - Depreciation - Amortization
if not forecast_is.empty:
    ebit_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        ebitda = pd.to_numeric(is_wide.loc['ebitda', y_str], errors='coerce')
        dep = pd.to_numeric(is_wide.loc['depreciation_owned', y_str], errors='coerce') or 0.0
        amort = pd.to_numeric(is_wide.loc['amortization', y_str], errors='coerce') or 0.0
        ebit = pd.to_numeric(is_wide.loc['ebit', y_str], errors='coerce')
        
        if all(pd.notna(v) for v in [ebitda, ebit]):
            calculated_ebit = ebitda - dep - amort
            diff = abs(ebit - calculated_ebit)
            if diff > 1.0:
                ebit_errors.append(f"{y}: diff=${diff:,.0f}")
    
    logic_tests.append({
        'Формула': 'EBIT = EBITDA - Depreciation - Amortization',
        'Статус': '✅' if not ebit_errors else '❌',
        'Ошибки': ', '.join(ebit_errors) if ebit_errors else 'Нет'
    })

# 3. EBT = EBIT + Interest Income - Interest Expense - Lease Interest
if not forecast_is.empty:
    ebt_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        ebit = pd.to_numeric(is_wide.loc['ebit', y_str], errors='coerce')
        int_income = pd.to_numeric(is_wide.loc['interest_income', y_str], errors='coerce') or 0.0
        int_expense = pd.to_numeric(is_wide.loc['interest_expense', y_str], errors='coerce') or 0.0
        lease_int = pd.to_numeric(is_wide.loc['lease_interest', y_str], errors='coerce') or 0.0
        ebt = pd.to_numeric(is_wide.loc['ebt', y_str], errors='coerce')
        
        if pd.notna(ebit) and pd.notna(ebt):
            calculated_ebt = ebit + int_income - int_expense - lease_int
            diff = abs(ebt - calculated_ebt)
            if diff > 1.0:
                ebt_errors.append(f"{y}: diff=${diff:,.0f}")
    
    logic_tests.append({
        'Формула': 'EBT = EBIT + Interest Income - Interest Expense - Lease Interest',
        'Статус': '✅' if not ebt_errors else '❌',
        'Ошибки': ', '.join(ebt_errors) if ebt_errors else 'Нет'
    })

# 4. Net Income = EBT - Tax Expense
if not forecast_is.empty:
    ni_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        ebt = pd.to_numeric(is_wide.loc['ebt', y_str], errors='coerce')
        tax = pd.to_numeric(is_wide.loc['tax_expense', y_str], errors='coerce') or 0.0
        ni = pd.to_numeric(is_wide.loc['net_income', y_str], errors='coerce')
        
        if pd.notna(ebt) and pd.notna(ni):
            calculated_ni = ebt - tax
            diff = abs(ni - calculated_ni)
            if diff > 1.0:
                ni_errors.append(f"{y}: diff=${diff:,.0f}")
    
    logic_tests.append({
        'Формула': 'Net Income = EBT - Tax Expense',
        'Статус': '✅' if not ni_errors else '❌',
        'Ошибки': ', '.join(ni_errors) if ni_errors else 'Нет'
    })

# 5. COGS Ratio в допустимых пределах (0.55 - 0.98)
if not forecast_is.empty:
    cogs_clamp_errors = []
    cogs_config = proj_yaml.get('model', {}).get('standard', {}).get('cogs', {})
    cogs_clamp_min = cogs_config.get('clamp', {}).get('min', 0.55)
    cogs_clamp_max = cogs_config.get('clamp', {}).get('max', 0.98)
    
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        revenue = pd.to_numeric(is_wide.loc['revenue', y_str], errors='coerce')
        cogs = pd.to_numeric(is_wide.loc['cogs', y_str], errors='coerce')
        
        if pd.notna(revenue) and pd.notna(cogs) and revenue > 0:
            cogs_ratio = cogs / revenue
            if not (cogs_clamp_min <= cogs_ratio <= cogs_clamp_max):
                cogs_clamp_errors.append(f"{y}: ratio={cogs_ratio:.2%} (допустимо {cogs_clamp_min:.0%}-{cogs_clamp_max:.0%})")
    
    logic_tests.append({
        'Формула': f'COGS Ratio в пределах {cogs_clamp_min:.0%}-{cogs_clamp_max:.0%}',
        'Статус': '✅' if not cogs_clamp_errors else '❌',
        'Ошибки': ', '.join(cogs_clamp_errors) if cogs_clamp_errors else 'Нет'
    })

if logic_tests:
    logic_df = pd.DataFrame(logic_tests)
    display(logic_df)
else:
    print("⚠️ Не удалось проверить логику моделирования - данные отсутствуют")


In [ ]:
display(Markdown("### 📊 Итоговая сводка всех тестов"))

# Собираем результаты всех тестов
final_summary = {
    'Категория теста': [],
    'Пройдено': [],
    'Всего': [],
    'Статус': []
}

# 1. Полнота данных
if 'completeness_df' in locals() and not completeness_df.empty:
    passed = len(completeness_df[completeness_df['Статус'] == '✅'])
    total = len(completeness_df)
    final_summary['Категория теста'].append('Полнота данных (IS/BS/CF)')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 2. Корректность Corkscrews
if 'corkscrew_df' in locals() and not corkscrew_df.empty:
    passed = len(corkscrew_df[corkscrew_df['Статус'] == '✅'])
    total = len(corkscrew_df)
    final_summary['Категория теста'].append('Корректность Corkscrews')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 3. Финансовые связи
if 'financial_df' in locals() and not financial_df.empty:
    passed = len(financial_df[financial_df['Статус'] == '✅'])
    total = len(financial_df)
    final_summary['Категория теста'].append('Финансовые связи')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 4. Соответствие трендам
if 'trend_df' in locals() and not trend_df.empty:
    passed = len(trend_df[trend_df['Статус'] == '✅'])
    total = len(trend_df)
    final_summary['Категория теста'].append('Соответствие историческим трендам')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 5. Наличие драйверов
if 'driver_df' in locals() and not driver_df.empty:
    passed = len(driver_df[driver_df['Статус'] == '✅'])
    total = len(driver_df)
    final_summary['Категория теста'].append('Наличие драйверов и логики')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 6. Acceptance Checks
if 'check_summary' in locals() and not check_summary.empty:
    passed = len(check_summary[check_summary['Статус'] == '✅'])
    total = len(check_summary)
    final_summary['Категория теста'].append('Acceptance Checks')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 7. Логика моделирования
if 'logic_df' in locals() and not logic_df.empty:
    passed = len(logic_df[logic_df['Статус'] == '✅'])
    total = len(logic_df)
    final_summary['Категория теста'].append('Логика моделирования (формулы)')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

if final_summary['Категория теста']:
    summary_df = pd.DataFrame(final_summary)
    display(summary_df)
    
    # Общая статистика
    total_passed = sum(final_summary['Пройдено'])
    total_tests = sum(final_summary['Всего'])
    success_rate = (total_passed / total_tests * 100) if total_tests > 0 else 0
    
    print(f"\n📊 Итоговая статистика тестирования:")
    print(f"  Всего тестов: {total_tests}")
    print(f"  Пройдено: {total_passed}")
    print(f"  Процент успеха: {success_rate:.1f}%")
    print(f"  Статус: {'✅ Модель готова' if success_rate >= 90 else ('⚠️ Требуется доработка' if success_rate >= 70 else '❌ Критические ошибки')}")
else:
    print("⚠️ Тесты не выполнены. Запустите build_model() для генерации данных.")


## 6️⃣ Визуализация результатов <a id="section-6"></a>

In [ ]:
display(Markdown("### Графики ключевых метрик"))

try:
    if plt is None:
        import matplotlib.pyplot as plt
    is_df = load_model_forecast_from_db('IS', canonical=True)
    bs_df = load_model_forecast_from_db('BS', canonical=True)
    cf_df = load_model_forecast_from_db('CF', canonical=True)
    if is_df.empty or bs_df.empty or cf_df.empty:
        print("⚠️ Прогнозы моделей не найдены ни в витрине, ни в CSV fallback")
    else:
        is_idx = is_df.set_index('metric') if 'metric' in is_df.columns else is_df
        bs_idx = bs_df.set_index('metric') if 'metric' in bs_df.columns else bs_df
        year_cols = [c for c in is_df.columns if str(c).isdigit()]
        years = sorted(int(c) for c in year_cols)
        metrics_to_plot = {
            'Revenue': ('revenue', is_idx),
            'EBITDA': ('ebitda', is_idx),
            'Net Income': ('net_income', is_idx),
            'Total Assets': ('total_assets', bs_idx),
            'Net Debt': ('net_debt', bs_idx),
            'Cash': ('cash', bs_idx)
        }
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        for ax, (metric_name, (metric_key, df)) in zip(axes, metrics_to_plot.items()):
            if metric_key in df.index:
                values = [pd.to_numeric(df.loc[metric_key, str(y)], errors='coerce') for y in years]
                history_years = [y for y in years if y <= 2024]
                forecast_years = [y for y in years if y > 2024]
                if history_years:
                    ax.plot(history_years, [values[years.index(y)] for y in history_years], 'o-', linewidth=2, markersize=4, label='История')
                if forecast_years:
                    ax.plot(forecast_years, [values[years.index(y)] for y in forecast_years], 's--', linewidth=2, markersize=4, label='Прогноз', color='green')
                ax.set_title(metric_name)
                ax.set_xlabel('Год')
                ax.set_ylabel('Значение')
                ax.legend()
                ax.grid(True, alpha=0.3)
            else:
                ax.text(0.5, 0.5, f'{metric_name}
не найдено', ha='center', va='center')
                ax.set_title(metric_name)
                ax.axis('off')
        plt.tight_layout()
        plt.show()
except Exception as exc:
    print(f"⚠️ Ошибка построения графиков: {exc}")
    import traceback
    traceback.print_exc()


## 7️⃣ Итоговая сводка <a id="section-7"></a>

In [ ]:
display(Markdown("### 📋 Итоговая сводка тестирования модели"))

summary = {
    'Компонент': [],
    'Статус': [],
    'Детали': []
}

# Проверка входных данных
all_history_ok = all([p.exists() for p in history_paths.values()])
summary['Компонент'].append('Входные данные (IS/BS/CF)')
summary['Статус'].append('✅' if all_history_ok else '❌')
summary['Детали'].append(f"Найдено файлов: {sum([1 for p in history_paths.values() if p.exists()])}/3")

# Проверка макро-прогнозов
macro_ok = macro_forecast_dir.exists() and len(list(macro_forecast_dir.glob("*_forecast.csv"))) > 0
summary['Компонент'].append('Макро-прогнозы')
summary['Статус'].append('✅' if macro_ok else '⚠️')
summary['Детали'].append(f"Файлов: {len(list(macro_forecast_dir.glob('*_forecast.csv'))) if macro_forecast_dir.exists() else 0}")

# Проверка выходных файлов
output_files_ok = all([(output_dir / f).exists() for f in ['3statement_IS.csv', '3statement_BS.csv', '3statement_CF.csv']])
summary['Компонент'].append('Выходные файлы модели')
summary['Статус'].append('✅' if output_files_ok else '❌')
summary['Детали'].append(f"Создано: {sum([1 for f in output_files.values() if (output_dir / f).exists()])}/{len(output_files)}")

# Проверка acceptance checks
checks_path = croot / "outputs" / "checks" / "acceptance_checks.csv"
checks_ok = checks_path.exists()
summary['Компонент'].append('Acceptance Checks')
summary['Статус'].append('✅' if checks_ok else '⚠️')
summary['Детали'].append('Выполнены' if checks_ok else 'Не выполнены')

# Проверка ковенант
covenants_path = croot / "outputs" / "checks" / "covenants.csv"
covenants_ok = covenants_path.exists()
summary['Компонент'].append('Ковенанты (Covenants)')
summary['Статус'].append('✅' if covenants_ok else '⚠️')
summary['Детали'].append('Рассчитаны' if covenants_ok else 'Не рассчитаны')

summary_df = pd.DataFrame(summary)
display(summary_df)

print(f"\n📊 Статистика:")
print(f"  Всего проверок: {len(summary_df)}")
print(f"  Успешно: {sum([1 for s in summary_df['Статус'] if s == '✅'])}")
print(f"  Предупреждений: {sum([1 for s in summary_df['Статус'] if s == '⚠️'])}")
print(f"  Ошибок: {sum([1 for s in summary_df['Статус'] if s == '❌'])}")
